# Programación de microcontroladores · PIC16F877 / PIC16F877A
## Laboratorio en Colab con 10 ejemplos prácticos

Este cuaderno acompaña las diapositivas. Todo corre en Python:

1. Un **ensamblador** compatible con la sintaxis de MPASM y un **emulador** del PIC16F877A (núcleo, interrupciones, TMR0/1/2, PWM, ADC, USART y SPI) con modelos de **LCD 16x2, OLED SSD1306, teclado 4x4, servo, motores y displays**.
2. Los **10 programas** en ensamblador (y en C para XC8). Aquí se ensamblan y producen los `.hex` para **Proteus** o para grabar el PIC.
3. Cada ejemplo se **simula** y se grafica: la pantalla de la LCD, la OLED con los juegos, las curvas de los sensores, el pulso del servo, el PWM del motor, el cronómetro, el motor paso a paso y el semáforo.
4. **YOLO de verdad** (Ultralytics) detecta carros, motos y personas en imágenes y le envía la letra al PIC emulado.

> *Entorno de ejecución → Ejecutar todas*. Las celdas `%%writefile` solo guardan archivos.

| # | Ejemplo | Lo que se aprende |
|---|---|---|
| 1 | Contador 0000-9999 con LCD 16x2 | LCD en modo de 4 bits, antirrebote por software, conversión binario → BCD |
| 2 | Tres juegos en OLED 128x64 con pulsadores | SPI con el módulo MSSP, controlador gráfico SSD1306, lógica de juegos y números aleatorios (LFSR) |
| 3 | Sensores: temperatura (LM35) y luz (LDR) | ADC de 10 bits con dos canales, aritmética de 16/24 bits, control por umbral |
| 4 | Integración con YOLO por la USART | USART a 9600 baudios, protocolo de comandos de un carácter, integración PC–microcontrolador con visión artificial |
| 5 | Cerradura con teclado matricial 4x4 | barrido de teclado matricial, direccionamiento indirecto FSR/INDF, máquina de estados con intentos |
| 6 | Servomotor controlado con potenciómetro | generación de pulsos por software, temporización por ciclos de instrucción, escalado de lecturas del ADC |
| 7 | Motor DC con PWM y puente H L293D | PWM con CCP1 y TMR2, puente H y sentido de giro, lectura de pulsadores |
| 8 | Cronómetro con display multiplexado e interrupciones | interrupciones (TMR0 e INT), guardar el contexto (W, STATUS, PCLATH), multiplexado de displays |
| 9 | Motor paso a paso unipolar con ULN2003A | secuencias de excitación de bobinas, tablas con addwf PCL, control de velocidad |
| 10 | Semáforo con botón peatonal (INT + TMR1) | TMR1 con precarga, interrupciones de periféricos (PEIE), máquina de estados temporizada |

## 1. Herramientas: ensamblador, emulador y dispositivos

In [ ]:
%%writefile picasm.py
"""picasm: ensamblador didáctico para PIC16F877/877A (subconjunto de MPASM).
Directivas: LIST (p=, r=), #include (se ignora: los símbolos del PIC16F877A ya están cargados), #define, __CONFIG, ORG, EQU, SET,
CBLOCK/ENDC, DT, DA, BANKSEL, PAGESEL, ERRORLEVEL, END. Expresiones con + - * / % << >> & | ^ ~ ( ), HIGH, LOW, $.
Números: 0x1F, h'1F', b'0101', d'25', .25, 'A' y la base por defecto de LIST r= (hex si no se indica)."""
import re

SFR = dict(INDF=0x00, TMR0=0x01, PCL=0x02, STATUS=0x03, FSR=0x04, PORTA=0x05, PORTB=0x06, PORTC=0x07, PORTD=0x08, PORTE=0x09,
           PCLATH=0x0A, INTCON=0x0B, PIR1=0x0C, PIR2=0x0D, TMR1L=0x0E, TMR1H=0x0F, T1CON=0x10, TMR2=0x11, T2CON=0x12,
           SSPBUF=0x13, SSPCON=0x14, CCPR1L=0x15, CCPR1H=0x16, CCP1CON=0x17, RCSTA=0x18, TXREG=0x19, RCREG=0x1A,
           CCPR2L=0x1B, CCPR2H=0x1C, CCP2CON=0x1D, ADRESH=0x1E, ADCON0=0x1F,
           OPTION_REG=0x81, TRISA=0x85, TRISB=0x86, TRISC=0x87, TRISD=0x88, TRISE=0x89, PIE1=0x8C, PIE2=0x8D, PCON=0x8E,
           SSPCON2=0x91, PR2=0x92, SSPADD=0x93, SSPSTAT=0x94, TXSTA=0x98, SPBRG=0x99, CMCON=0x9C, CVRCON=0x9D, ADRESL=0x9E, ADCON1=0x9F,
           EEDATA=0x10C, EEADR=0x10D, EEDATH=0x10E, EEADRH=0x10F, EECON1=0x18C, EECON2=0x18D)
BITS = dict(C=0, DC=1, Z=2, NOT_PD=3, NOT_TO=4, RP0=5, RP1=6, IRP=7,
            RBIF=0, INTF=1, T0IF=2, RBIE=3, INTE=4, T0IE=5, PEIE=6, GIE=7, TMR0IF=2, TMR0IE=5,
            TMR1IF=0, TMR2IF=1, CCP1IF=2, SSPIF=3, TXIF=4, RCIF=5, ADIF=6, PSPIF=7,
            TMR1IE=0, TMR2IE=1, CCP1IE=2, SSPIE=3, TXIE=4, RCIE=5, ADIE=6, PSPIE=7,
            TMR1ON=0, TMR1CS=1, NOT_T1SYNC=2, T1OSCEN=3, T1CKPS0=4, T1CKPS1=5, TMR2ON=2, T2CKPS0=0, T2CKPS1=1,
            PS0=0, PS1=1, PS2=2, PSA=3, T0SE=4, T0CS=5, INTEDG=6, NOT_RBPU=7,
            ADON=0, GO=2, GO_DONE=2, NOT_DONE=2, CHS0=3, CHS1=4, CHS2=5, ADCS0=6, ADCS1=7, ADFM=7, PCFG0=0, PCFG1=1, PCFG2=2, PCFG3=3,
            RX9D=0, OERR=1, FERR=2, ADDEN=3, CREN=4, SREN=5, RX9=6, SPEN=7, TX9D=0, TRMT=1, BRGH=2, SYNC=4, TXEN=5, TX9=6, CSRC=7,
            BF=0, CKE=6, SMP=7, SSPM0=0, SSPM1=1, SSPM2=2, SSPM3=3, CKP=4, SSPEN=5, SSPOV=6, WCOL=7,
            RD=0, WR=1, WREN=2, WRERR=3, EEPGD=7, CCP1M0=0, CCP1M1=1, CCP1M2=2, CCP1M3=3, CCP1Y=4, CCP1X=5,
            F=1, W=0)
CONFIG = dict(_CP_OFF=0x3FFF, _CP_ALL=0x1FFF, _DEBUG_OFF=0x3FFF, _DEBUG_ON=0x37FF, _WRT_OFF=0x3FFF, _CPD_OFF=0x3FFF, _CPD_ON=0x3EFF,
              _LVP_OFF=0x3F7F, _LVP_ON=0x3FFF, _BOREN_OFF=0x3FBF, _BOREN_ON=0x3FFF, _BODEN_OFF=0x3FBF, _BODEN_ON=0x3FFF,
              _PWRTE_ON=0x3FF7, _PWRTE_OFF=0x3FFF, _WDT_OFF=0x3FFB, _WDTE_OFF=0x3FFB, _WDT_ON=0x3FFF, _WDTE_ON=0x3FFF,
              _LP_OSC=0x3FFC, _XT_OSC=0x3FFD, _HS_OSC=0x3FFE, _RC_OSC=0x3FFF)
BYTE = dict(ADDWF=0x0700, ANDWF=0x0500, COMF=0x0900, DECF=0x0300, DECFSZ=0x0B00, INCF=0x0A00, INCFSZ=0x0F00, IORWF=0x0400,
            MOVF=0x0800, RLF=0x0D00, RRF=0x0C00, SUBWF=0x0200, SWAPF=0x0E00, XORWF=0x0600)
BIT = dict(BCF=0x1000, BSF=0x1400, BTFSC=0x1800, BTFSS=0x1C00)
LIT = dict(ADDLW=0x3E00, ANDLW=0x3900, IORLW=0x3800, MOVLW=0x3000, RETLW=0x3400, SUBLW=0x3C00, XORLW=0x3A00)
SOLO = dict(NOP=0x0000, CLRW=0x0100, RETURN=0x0008, RETFIE=0x0009, SLEEP=0x0063, CLRWDT=0x0064)


class _Op:
    def __init__(self, f): self.f = f
    def __matmul__(self, v): return self.f(v)


_UNA = {'_hi': _Op(lambda v: (v >> 8) & 0xFF), '_lo': _Op(lambda v: v & 0xFF), '_up': _Op(lambda v: (v >> 16) & 0xFF)}


class ErrorASM(Exception):
    pass


class Ensamblador:
    def __init__(self, fuente):
        self.fuente = fuente
        self.radix = 16
        self.mem = {}
        self.conf = None
        self.lista = []

    # ---------- números y expresiones ----------
    def numero(self, t):
        t = t.strip(); u = t.upper()
        m = re.fullmatch(r"([HDBO])'([0-9A-F]+)'", u)
        if m: return int(m.group(2), {'H': 16, 'D': 10, 'B': 2, 'O': 8}[m.group(1)])
        if u.startswith('0X'): return int(u[2:], 16)
        if u.startswith('.') and u[1:].isdigit(): return int(u[1:])
        if re.fullmatch(r'[01]+B', u) and self.radix != 16: return int(u[:-1], 2)
        if re.fullmatch(r'[0-9A-F]+H', u) and u[0].isdigit(): return int(u[:-1], 16)
        if re.fullmatch(r'[0-9A-F]+', u) and u[0].isdigit(): return int(u, self.radix)
        return None

    def valor(self, expr, pasada, pc=0):
        expr = expr.strip()
        if not expr: raise ErrorASM('operando vacío')
        expr = re.sub(r"'(.)'", lambda m: str(ord(m.group(1))), expr)
        toks = re.findall(r"[HDBOhdbo]'[0-9A-Fa-f]+'|0[xX][0-9A-Fa-f]+|\.\d+|[A-Za-z_][A-Za-z0-9_]*|\d[0-9A-Fa-f]*[hH]?|<<|>>|[-+*/%&|^~()$]", expr)
        out = []
        for tk in toks:
            u = tk.upper()
            if tk in ('<<', '>>') or tk in '+-*/%&|^~()': out.append('//' if tk == '/' else tk); continue
            if tk == '$': out.append(str(pc)); continue
            if u == 'HIGH': out.append('_hi @'); continue
            if u == 'LOW': out.append('_lo @'); continue
            if u == 'UPPER': out.append('_up @'); continue
            n = self.numero(tk)
            if n is not None and not (u in self.simbolos or u in SFR or u in BITS): out.append(str(n)); continue
            if u in self.simbolos: out.append(str(self.simbolos[u])); continue
            if u in SFR: out.append(str(SFR[u])); continue
            if u in BITS: out.append(str(BITS[u])); continue
            if u in CONFIG: out.append(str(CONFIG[u])); continue
            if n is not None: out.append(str(n)); continue
            if pasada == 1: out.append('0'); continue
            raise ErrorASM(f'símbolo no definido: {tk}')
        try:
            return int(eval(' '.join(out), {'__builtins__': {}}, _UNA))
        except ErrorASM:
            raise
        except Exception:
            raise ErrorASM(f'expresión inválida: {expr}')

    @staticmethod
    def partir(ops):
        res, cur, q = [], '', None
        for ch in ops:
            if ch in '"\'' and (q is None or q == ch): q = None if q else ch
            if ch == ',' and not q: res.append(cur.strip()); cur = ''
            else: cur += ch
        if cur.strip(): res.append(cur.strip())
        return res

    # ---------- ensamblado en dos pasadas ----------
    def ensamblar(self):
        lineas = self.fuente.splitlines()
        for pasada in (1, 2):
            self.simbolos = {} if pasada == 1 else self.simbolos
            self.defines = {}
            self.mem = {}; self.lista = []; self.pc = 0; self.cblock = None; self.radix = 16
            for nlin, linea in enumerate(lineas, 1):
                try:
                    if self.linea(linea, pasada, nlin) == 'END': break
                except ErrorASM as e:
                    raise ErrorASM(f'línea {nlin}: {e}\n    {linea.strip()}')
        return self

    def linea(self, linea, pasada, nlin):
        # quitar comentario (respetando comillas)
        s, q = '', False
        for ch in linea:
            if ch in '"\'': q = not q
            if ch == ';' and not q: break
            s += ch
        if not s.strip(): return
        st = s.strip()
        if st.upper().startswith('#DEFINE'):
            p = st.split(None, 2); self.defines[p[1].upper()] = p[2] if len(p) > 2 else '1'; return
        if st.upper().startswith('#INCLUDE') or st.upper().startswith('ERRORLEVEL') or st.upper().startswith('PROCESSOR'): return
        for k, v in self.defines.items():
            s = re.sub(r'(?i)\b' + re.escape(k) + r'\b', v, s)
        etiqueta = None
        if not s[0].isspace():
            m = re.match(r'([A-Za-z_][A-Za-z0-9_]*):?\s*(.*)', s)
            etiqueta, s = m.group(1), m.group(2)
        partes = s.strip().split(None, 1)
        mn = partes[0].upper() if partes else ''
        ops = partes[1] if len(partes) > 1 else ''
        # el modo CBLOCK: cada línea es una lista de nombres
        if self.cblock is not None and mn != 'ENDC':
            nombres = ([etiqueta] if etiqueta else []) + self.partir(s)
            for n in nombres:
                n = n.strip()
                if not n: continue
                m = re.match(r'([A-Za-z_]\w*)\s*(?::\s*(.+))?', n)
                self.simbolos[m.group(1).upper()] = self.cblock
                self.cblock += self.valor(m.group(2), pasada) if m.group(2) else 1
            return
        if mn in ('EQU', 'SET'):
            self.simbolos[etiqueta.upper()] = self.valor(ops, pasada, self.pc); return
        if etiqueta:
            if pasada == 1 and etiqueta.upper() in self.simbolos and etiqueta.upper() not in ('_',):
                raise ErrorASM(f'etiqueta repetida: {etiqueta}')
            self.simbolos[etiqueta.upper()] = self.pc
        if not mn: return
        if mn == 'LIST':
            m = re.search(r'(?i)r\s*=\s*(\w+)', ops)
            if m: self.radix = {'DEC': 10, 'HEX': 16, 'OCT': 8}[m.group(1).upper()]
            return
        if mn == 'END': return 'END'
        if mn == 'ORG': self.pc = self.valor(ops, pasada); return
        if mn == 'CBLOCK': self.cblock = self.valor(ops, pasada) if ops.strip() else 0; return
        if mn == 'ENDC': self.cblock = None; return
        if mn == '__CONFIG':
            args = self.partir(ops); self.conf = self.valor(args[-1], pasada) & 0x3FFF; return
        if mn in ('DT', 'DA'):
            for a in self.partir(ops):
                if a.startswith('"'):
                    for ch in a.strip('"'): self.emitir(0x3400 | ord(ch), f'retlw {ch!r}', nlin, pasada, s)
                else:
                    self.emitir(0x3400 | (self.valor(a, pasada, self.pc) & 0xFF), 'retlw', nlin, pasada, s)
            return
        if mn == 'BANKSEL':
            a = self.valor(ops, pasada)
            self.emitir((0x1683 if a & 0x80 else 0x1283), '', nlin, pasada, s)
            self.emitir((0x1703 if a & 0x100 else 0x1303), '', nlin, pasada, s); return
        if mn == 'PAGESEL':
            a = self.valor(ops, pasada)
            self.emitir((0x158A if a & 0x800 else 0x118A), '', nlin, pasada, s)
            self.emitir((0x160A if a & 0x1000 else 0x120A), '', nlin, pasada, s); return
        args = self.partir(ops)
        V = lambda e: self.valor(e, pasada, self.pc)
        if mn in SOLO: w = SOLO[mn]
        elif mn in BYTE:
            d = 1 if len(args) < 2 else (V(args[1]) & 1)
            w = BYTE[mn] | (d << 7) | (V(args[0]) & 0x7F)
        elif mn == 'CLRF': w = 0x0180 | (V(args[0]) & 0x7F)
        elif mn == 'MOVWF': w = 0x0080 | (V(args[0]) & 0x7F)
        elif mn in BIT:
            if len(args) != 2: raise ErrorASM(f'{mn} necesita registro,bit')
            b = V(args[1])
            if not 0 <= b <= 7: raise ErrorASM('bit fuera de rango')
            w = BIT[mn] | (b << 7) | (V(args[0]) & 0x7F)
        elif mn in LIT: w = LIT[mn] | (V(args[0]) & 0xFF)
        elif mn in ('CALL', 'GOTO'):
            k = V(args[0])
            w = (0x2000 if mn == 'CALL' else 0x2800) | (k & 0x7FF)
        else:
            raise ErrorASM(f'instrucción desconocida: {mn}')
        self.emitir(w, '', nlin, pasada, s)

    def emitir(self, w, txt, nlin, pasada, fuente):
        if self.pc in self.mem and pasada == 2: raise ErrorASM(f'dirección 0x{self.pc:04X} usada dos veces')
        self.mem[self.pc] = w & 0x3FFF
        if pasada == 2: self.lista.append((self.pc, w & 0x3FFF, fuente.strip()))
        self.pc += 1

    def hex(self):
        datos = {}
        for a, w in self.mem.items(): datos[2 * a] = w & 0xFF; datos[2 * a + 1] = w >> 8
        if self.conf is not None: datos[0x400E] = self.conf & 0xFF; datos[0x400F] = self.conf >> 8
        out = [':020000040000FA']
        dirs = sorted(datos); i = 0
        while i < len(dirs):
            ini = dirs[i]; bloque = [datos[ini]]; i += 1
            while i < len(dirs) and dirs[i] == ini + len(bloque) and len(bloque) < 16 and (ini + len(bloque)) % 16:
                bloque.append(datos[dirs[i]]); i += 1
            reg = [len(bloque), ini >> 8, ini & 0xFF, 0] + bloque
            out.append(':' + ''.join(f'{b:02X}' for b in reg) + f'{(-sum(reg)) & 0xFF:02X}')
        out.append(':00000001FF')
        return '\n'.join(out) + '\n'

    def palabras(self):
        return len([a for a in self.mem if a < 0x2000])


def leer_hex(texto):
    datos, base = {}, 0
    for l in texto.split():
        if not l.startswith(':'): continue
        b = bytes.fromhex(l[1:])
        if sum(b) & 0xFF: raise ValueError('suma de verificación incorrecta: ' + l)
        n, a, t = b[0], (b[1] << 8) | b[2], b[3]
        if t == 0:
            for k in range(n): datos[base + a + k] = b[4 + k]
        elif t == 4: base = ((b[4] << 8) | b[5]) << 16
        elif t == 1: break
    prog, conf = {}, None
    for a in sorted(datos):
        if a % 2 == 0 and a + 1 in datos:
            w = datos[a] | (datos[a + 1] << 8)
            if a // 2 == 0x2007: conf = w
            elif a // 2 < 0x2000: prog[a // 2] = w & 0x3FFF
    return prog, conf


_INV = {v: k for k, v in SFR.items() if v < 0x80}
_INV1 = {v & 0x7F: k for k, v in SFR.items() if 0x80 <= v < 0x100}


def desensamblar(w):
    if w == 0x0008: return 'return'
    if w == 0x0009: return 'retfie'
    if w == 0x0063: return 'sleep'
    if w == 0x0064: return 'clrwdt'
    if w == 0x0100: return 'clrw'
    top = w & 0x3000; f = w & 0x7F; fn = _INV.get(f, f'0x{f:02X}')
    if top == 0:
        o = w & 0x0F00; d = (w >> 7) & 1
        if o == 0: return f'movwf  {fn}' if w & 0x80 else 'nop'
        if o == 0x0100: return f'clrf   {fn}'
        n = {v: k for k, v in BYTE.items()}.get(o, '??').lower()
        return f'{n:<7s}{fn},{"F" if d else "W"}'
    if top == 0x1000:
        n = ['bcf', 'bsf', 'btfsc', 'btfss'][(w >> 10) & 3]
        return f'{n:<7s}{fn},{(w >> 7) & 7}'
    if top == 0x2000: return f'{"goto" if w & 0x800 else "call":<7s}0x{w & 0x7FF:03X}'
    k = w & 0xFF
    for n, c in (('movlw', 0x3000), ('retlw', 0x3400)):
        if w & 0x3C00 == c: return f'{n:<7s}0x{k:02X}'
    for n, c in (('iorlw', 0x3800), ('andlw', 0x3900), ('xorlw', 0x3A00)):
        if w & 0x3F00 == c: return f'{n:<7s}0x{k:02X}'
    if w & 0x3E00 == 0x3C00: return f'sublw  0x{k:02X}'
    return f'addlw  0x{k:02X}'

In [ ]:
%%writefile pic16sim.py
"""pic16sim: emulador didáctico del PIC16F877/877A.
Núcleo de 35 instrucciones con temporización real, interrupciones (INT/RB0, TMR0, periféricos), TMR0, TMR1, TMR2,
PWM del CCP1/CCP2, ADC de 10 bits, USART y MSSP en modo SPI maestro. Los dispositivos externos (LCD, OLED, teclado...)
están en dispositivos.py y se conectan con escuchas de pines."""

PUERTOS = 'ABCDE'


class PIC16F877:
    def __init__(self, prog, fosc=4e6):
        self.prog = [0x3FFF] * 8192
        for a, w in prog.items(): self.prog[a] = w
        self.fosc = fosc
        self.ent = {'A': 0xFF, 'B': 0xFF, 'C': 0xFF, 'D': 0xFF, 'E': 0xFF}   # nivel externo de las entradas
        self.fn_ent = {}            # puerto -> función(pic) que calcula las entradas (teclado, etc.)
        self.escuchas = []          # función(pic, puerto, anterior, nuevo) al cambiar una salida
        self.spi = []               # función(pic, byte) al transmitir por SPI
        self.analog = [0.0] * 8
        self.reset()

    def reset(self):
        self.ram = [0] * 512
        self.w = 0; self.pc = 0; self.pila = []; self.cy = 0; self.dormido = False
        r = self.ram
        r[0x03] = 0x18; r[0x81] = 0xFF
        for t in (0x85, 0x86, 0x87, 0x88): r[t] = 0xFF
        r[0x89] = 0x07; r[0x92] = 0xFF; r[0x98] = 0x02; r[0x0C] = 0x10
        self.pre0 = 0; self.pre1 = 0; self.pre2 = 0; self.post2 = 0
        self.adc_fin = None; self.ssp_fin = None
        self.tsr = None; self.tsr_fin = 0; self.txreg_lleno = False; self.tx = []; self.rxq = []; self.rx_listo = 0
        self.salidas = {p: self.salida(p) for p in PUERTOS}
        self.rb0 = self.entrada_pin('B', 0)

    # ---------- memoria ----------
    def dirf(self, f):
        if f == 0: f = self.ram[0x04] | ((self.ram[0x03] & 0x80) << 1)
        else: f |= (self.ram[0x03] & 0x60) << 2
        b = f & 0x7F
        if b in (0x00, 0x02, 0x03, 0x04, 0x0A, 0x0B) or b >= 0x70: return b
        if f in (0x101, 0x106, 0x181, 0x186): return f & 0xFF
        return f

    def entradas(self, p):
        return self.fn_ent[p](self) if p in self.fn_ent else self.ent[p]

    def leer(self, a):
        if 5 <= a <= 9:
            p = PUERTOS[a - 5]; tris = self.ram[a + 0x80]
            v = (self.ram[a] & ~tris) | (self.entradas(p) & tris)
            if p == 'B' and not (self.ram[0x81] & 0x80): pass   # pull-ups: el nivel por defecto ya es 1
            return v & 0xFF
        if a == 0x02: return self.pc & 0xFF
        if a == 0x1A: self.ram[0x0C] &= ~0x20
        if a == 0x13: self.ram[0x94] &= ~0x01
        return self.ram[a]

    def salida(self, p):
        a = 5 + PUERTOS.index(p); tris = self.ram[a + 0x80]
        return self.ram[a] & ~tris & 0xFF   # pines en alta impedancia se ven en 0

    def entrada_pin(self, p, b):
        a = 5 + PUERTOS.index(p)
        return (self.leer(a) >> b) & 1

    def poner_entrada(self, p, v):
        """Cambia el nivel externo de un puerto (pulsadores) y detecta el flanco de RB0/INT."""
        self.ent[p] = v & 0xFF
        self.revisar_int()

    def revisar_int(self):
        n = self.entrada_pin('B', 0)
        if n != self.rb0 and (self.ram[0x86] & 1):
            sube = (self.ram[0x81] >> 6) & 1
            if (n == 1 and sube) or (n == 0 and not sube): self.ram[0x0B] |= 0x02
        self.rb0 = n

    def escribir(self, a, v):
        v &= 0xFF; r = self.ram
        if a == 0x02:
            r[2] = v; self.pc = ((r[0x0A] & 0x1F) << 8) | v; return 1
        if a == 0x03: v = (r[3] & 0x18) | (v & 0xE7)
        if a == 0x01: self.pre0 = 0
        r[a] = v
        if 5 <= a <= 9 or 0x85 <= a <= 0x89:
            p = PUERTOS[(a & 0x7F) - 5]; n = self.salida(p); ant = self.salidas[p]
            if n != ant:
                self.salidas[p] = n
                for e in self.escuchas: e(self, p, ant, n)
            if p == 'B': self.revisar_int()
        elif a == 0x1F:
            if (v & 0x05) == 0x05 and self.adc_fin is None:
                adcs = ((r[0x9F] >> 4) & 4) | ((v >> 6) & 3)
                div = {0: 2, 1: 8, 2: 32, 3: 0, 4: 4, 5: 16, 6: 64, 7: 0}[adcs]
                tad = 4e-6 if div == 0 else div / self.fosc
                self.adc_fin = self.cy + int(12 * tad * self.fosc / 4) + 1
        elif a == 0x19:
            if self.tsr is None: self.tsr = v; self.tsr_fin = self.cy + self.ciclos_char()
            else: self.txreg_lleno = True; r[0x0C] &= ~0x10
        elif a == 0x13 and (r[0x14] & 0x20):
            for e in self.spi: e(self, v)
            div = {0: 4, 1: 16, 2: 64}.get(r[0x14] & 0x0F, 4)
            self.ssp_fin = self.cy + 2 * div
        return 0

    def ciclos_char(self):
        brgh = (self.ram[0x98] >> 2) & 1
        baud = self.fosc / ((16 if brgh else 64) * (self.ram[0x99] + 1))
        return int(10 * (self.fosc / 4) / baud)

    def recibir(self, ch):
        self.rxq.extend(ch if isinstance(ch, (bytes, list)) else [ch])

    # ---------- periféricos ----------
    def tick(self, n):
        r = self.ram
        self.cy += n
        # TMR0
        op = r[0x81]
        if not (op & 0x20):
            pres = 1 if op & 0x08 else 2 << (op & 7)
            self.pre0 += n
            while self.pre0 >= pres:
                self.pre0 -= pres; r[1] = (r[1] + 1) & 0xFF
                if r[1] == 0: r[0x0B] |= 0x04
        # TMR1
        t1 = r[0x10]
        if t1 & 1:
            pres = 1 << ((t1 >> 4) & 3); self.pre1 += n
            while self.pre1 >= pres:
                self.pre1 -= pres; v = ((r[0x0F] << 8) | r[0x0E]) + 1
                if v > 0xFFFF: v = 0; r[0x0C] |= 0x01
                r[0x0E] = v & 0xFF; r[0x0F] = v >> 8
        # TMR2
        t2 = r[0x12]
        if t2 & 4:
            pres = (1, 4, 16, 16)[t2 & 3]; self.pre2 += n
            while self.pre2 >= pres:
                self.pre2 -= pres
                if r[0x11] == r[0x92]:
                    r[0x11] = 0; self.post2 += 1
                    if self.post2 >= ((t2 >> 3) & 15) + 1: self.post2 = 0; r[0x0C] |= 0x02
                else: r[0x11] = (r[0x11] + 1) & 0xFF
        # ADC
        if self.adc_fin is not None and self.cy >= self.adc_fin:
            ch = (r[0x1F] >> 3) & 7
            res = max(0, min(1023, round(self.analog[ch] / 5.0 * 1023)))
            if r[0x9F] & 0x80: r[0x1E] = res >> 8; r[0x9E] = res & 0xFF
            else: r[0x1E] = res >> 2; r[0x9E] = (res & 3) << 6
            r[0x1F] &= ~0x04; r[0x0C] |= 0x40; self.adc_fin = None
        # MSSP
        if self.ssp_fin is not None and self.cy >= self.ssp_fin:
            r[0x94] |= 0x01; r[0x0C] |= 0x08; self.ssp_fin = None
        # USART
        if self.tsr is not None:
            r[0x98] &= ~0x02
            if self.cy >= self.tsr_fin:
                self.tx.append(self.tsr); self.tsr = None
                if self.txreg_lleno:
                    self.txreg_lleno = False; self.tsr = r[0x19]; self.tsr_fin = self.cy + self.ciclos_char()
                r[0x0C] |= 0x10
                if self.tsr is None: r[0x98] |= 0x02
        if self.rxq and (r[0x18] & 0x90) == 0x90 and not (r[0x0C] & 0x20) and self.cy >= self.rx_listo:
            r[0x1A] = self.rxq.pop(0); r[0x0C] |= 0x20; self.rx_listo = self.cy + self.ciclos_char()

    def interrupcion(self):
        r = self.ram; ic = r[0x0B]
        pend = (ic >> 3) & ic & 0x07
        if (ic & 0x40) and (r[0x8C] & r[0x0C]): pend = 1
        if not pend: return
        self.dormido = False
        if ic & 0x80:
            if len(self.pila) == 8: self.pila.pop(0)
            self.pila.append(self.pc); self.pc = 4; r[0x0B] &= ~0x80
            self.tick(2)

    # ---------- núcleo ----------
    def flags(self, z=None, c=None, dc=None):
        s = self.ram[3]
        if z is not None: s = (s | 4) if z else (s & ~4)
        if c is not None: s = (s | 1) if c else (s & ~1)
        if dc is not None: s = (s | 2) if dc else (s & ~2)
        self.ram[3] = s & 0xFF

    def paso(self):
        if self.dormido:
            self.tick(1); self.interrupcion(); return
        n = self.ejecutar()
        self.tick(n)
        self.interrupcion()

    def ejecutar(self):
        op = self.prog[self.pc]
        self.pc = (self.pc + 1) & 0x1FFF
        top = op & 0x3000
        if top == 0:
            f = op & 0x7F; d = (op >> 7) & 1; o = op & 0x0F00
            if o == 0:
                if op & 0x80: return 1 + self.escribir(self.dirf(f), self.w)
                if op in (0x08, 0x09):
                    self.pc = self.pila.pop() if self.pila else 0
                    if op == 0x09: self.ram[0x0B] |= 0x80
                    return 2
                if op == 0x63: self.dormido = True
                return 1
            if o == 0x0100:
                if op & 0x80: self.escribir(self.dirf(f), 0)
                else: self.w = 0
                self.flags(z=True); return 1
            a = self.dirf(f); v = self.leer(a); w = self.w; skip = False
            if o == 0x0700: r = v + w; self.flags(z=(r & 0xFF) == 0, c=r > 255, dc=((v & 15) + (w & 15)) > 15)
            elif o == 0x0500: r = v & w; self.flags(z=r == 0)
            elif o == 0x0900: r = ~v & 0xFF; self.flags(z=r == 0)
            elif o == 0x0300: r = (v - 1) & 0xFF; self.flags(z=r == 0)
            elif o == 0x0B00: r = (v - 1) & 0xFF; skip = r == 0
            elif o == 0x0A00: r = (v + 1) & 0xFF; self.flags(z=r == 0)
            elif o == 0x0F00: r = (v + 1) & 0xFF; skip = r == 0
            elif o == 0x0400: r = v | w; self.flags(z=r == 0)
            elif o == 0x0800: r = v; self.flags(z=r == 0)
            elif o == 0x0D00: r = ((v << 1) | (self.ram[3] & 1)) & 0xFF; self.flags(c=bool(v & 0x80))
            elif o == 0x0C00: r = (v >> 1) | ((self.ram[3] & 1) << 7); self.flags(c=bool(v & 1))
            elif o == 0x0200: r = (v - w) & 0xFF; self.flags(z=r == 0, c=v >= w, dc=(v & 15) >= (w & 15))
            elif o == 0x0E00: r = ((v << 4) | (v >> 4)) & 0xFF
            elif o == 0x0600: r = v ^ w; self.flags(z=r == 0)
            else: raise ValueError(f'código {op:04X} inválido en {self.pc - 1:04X}')
            r &= 0xFF; extra = 0
            if d: extra = self.escribir(a, r)
            else: self.w = r
            if skip: self.pc = (self.pc + 1) & 0x1FFF; return 2
            return 1 + extra
        if top == 0x1000:
            f = op & 0x7F; b = (op >> 7) & 7; o = op & 0x0C00; a = self.dirf(f)
            if o == 0: self.escribir(a, self.ram[a] & ~(1 << b) if not 5 <= a <= 9 else self.leer(a) & ~(1 << b)); return 1
            if o == 0x400: self.escribir(a, self.ram[a] | (1 << b) if not 5 <= a <= 9 else self.leer(a) | (1 << b)); return 1
            bit = (self.leer(a) >> b) & 1
            if (o == 0x800 and not bit) or (o == 0xC00 and bit): self.pc = (self.pc + 1) & 0x1FFF; return 2
            return 1
        if top == 0x2000:
            k = op & 0x7FF
            if not op & 0x800:
                if len(self.pila) == 8: self.pila.pop(0)
                self.pila.append(self.pc)
            self.pc = ((self.ram[0x0A] & 0x18) << 8) | k
            return 2
        k = op & 0xFF; s = op & 0x3F00
        if s & 0x3C00 == 0x3000: self.w = k
        elif s & 0x3C00 == 0x3400: self.w = k; self.pc = self.pila.pop() if self.pila else 0; return 2
        elif s == 0x3800: self.w |= k; self.flags(z=self.w == 0)
        elif s == 0x3900: self.w &= k; self.flags(z=self.w == 0)
        elif s == 0x3A00: self.w ^= k; self.flags(z=self.w == 0)
        elif s & 0x3E00 == 0x3C00:
            r = k - self.w; self.flags(c=r >= 0, dc=(k & 15) >= (self.w & 15)); self.w = r & 0xFF; self.flags(z=self.w == 0)
        else:
            r = k + self.w; self.flags(c=r > 255, dc=((k & 15) + (self.w & 15)) > 15); self.w = r & 0xFF; self.flags(z=self.w == 0)
        return 1

    def correr(self, ciclos=None, segundos=None):
        if segundos is not None: ciclos = int(segundos * self.fosc / 4)
        fin = self.cy + ciclos
        while self.cy < fin: self.paso()

    def tiempo(self):
        return self.cy * 4 / self.fosc

    def puerto(self, p):
        return self.leer(5 + PUERTOS.index(p))

    def pwm(self, n=1):
        ccpcon, ccprl, pin = (0x17, 0x15, 2) if n == 1 else (0x1D, 0x1B, 1)
        t2 = self.ram[0x12]
        if (self.ram[ccpcon] & 0x0C) != 0x0C or not (t2 & 0x04): return None
        pres = (1, 4, 16, 16)[t2 & 3]
        per = (self.ram[0x92] + 1) * 4 * pres / self.fosc
        dc = ((self.ram[ccprl] << 2) | ((self.ram[ccpcon] >> 4) & 3)) * pres / self.fosc
        return {'f': 1 / per, 'D': min(1.0, dc / per), 'salida': not (self.ram[0x87] >> pin) & 1}

In [ ]:
%%writefile dispositivos.py
"""Modelos de los dispositivos externos que usan los ejemplos (equivalentes a los de Proteus)."""


def bit(v, b): return (v >> b) & 1


class LCD16x2:
    """HD44780 (LM016L en Proteus) en modo de 4 bits: D4-D7 = bits 4-7 del puerto, RS y E en el mismo puerto. RW a tierra."""
    def __init__(self, pic, puerto='D', rs=0, e=1):
        self.p, self.rs, self.e = puerto, rs, e
        self.ddram = [0x20] * 128; self.cgram = [0] * 64
        self.addr = 0; self.cg = False; self.modo8 = True; self.mitad = None; self.inc = 1
        self.encendido = False; self.ocupado = 0.0; self.errores = []; self.comandos = 0
        pic.escuchas.append(self.cambio)

    def cambio(self, pic, p, ant, n):
        if p != self.p or not (bit(ant, self.e) and not bit(n, self.e)): return
        t = pic.tiempo(); rs = bit(n, self.rs); nib = n >> 4
        if t < 0.015 and not self.errores: self.errores.append(f'{t*1e3:.2f} ms: escritura antes de 15 ms del encendido')
        if self.modo8:
            self.ejecutar(rs, nib << 4, t); return
        if self.mitad is None:
            if t < self.ocupado: self.errores.append(f'{t*1e3:.3f} ms: LCD ocupado (faltan {(self.ocupado - t)*1e6:.0f} µs)')
            self.mitad = nib; return
        b = (self.mitad << 4) | nib; self.mitad = None
        self.ejecutar(rs, b, t)

    def ejecutar(self, rs, b, t):
        self.comandos += 1
        dur = 43e-6
        if rs:
            if self.cg: self.cgram[self.addr & 63] = b; self.addr = (self.addr + self.inc) & 63
            else: self.ddram[self.addr & 127] = b; self.addr = (self.addr + self.inc) & 127
        elif b & 0x80: self.addr = b & 0x7F; self.cg = False
        elif b & 0x40: self.addr = b & 0x3F; self.cg = True
        elif b & 0x20:
            if not (b & 0x10): self.modo8 = False
            else: self.modo8 = True
        elif b & 0x10: pass
        elif b & 0x08: self.encendido = bool(b & 0x04)
        elif b & 0x04: self.inc = 1 if b & 2 else -1
        elif b & 0x02: self.addr = 0; self.cg = False; dur = 1.52e-3
        elif b & 0x01: self.ddram = [0x20] * 128; self.addr = 0; self.cg = False; dur = 1.52e-3
        self.ocupado = t + dur

    def linea(self, i):
        base = 0 if i == 0 else 0x40
        return ''.join(chr(c) if 32 <= c < 127 else ('°' if c == 0xDF else '█' if c < 8 else '?') for c in self.ddram[base:base + 16])

    def texto(self):
        return [self.linea(0), self.linea(1)] if self.encendido else ['', '']


class OLED:
    """SSD1306 128x64 por SPI (UG-2864HSWEG01 en Proteus): CS, D/C y RES en el puerto indicado."""
    def __init__(self, pic, puerto='C', cs=0, dc=1, res=2):
        self.p, self.cs, self.dc, self.res = puerto, cs, dc, res
        self.mem = [[0] * 128 for _ in range(8)]
        self.col = 0; self.pag = 0; self.modo = 2; self.c0, self.c1, self.p0, self.p1 = 0, 127, 0, 7
        self.esperando = []; self.cmd = None; self.encendido = False; self.bytes = 0
        pic.spi.append(self.byte)

    def byte(self, pic, v):
        o = pic.salidas[self.p]
        if bit(o, self.cs) or not bit(o, self.res): return
        self.bytes += 1
        if bit(o, self.dc): self.dato(v)
        else: self.comando(v)

    def dato(self, v):
        self.mem[self.pag & 7][self.col & 127] = v
        if self.modo == 0:
            self.col += 1
            if self.col > self.c1:
                self.col = self.c0; self.pag += 1
                if self.pag > self.p1: self.pag = self.p0
        else:
            self.col = min(self.col + 1, 127)

    def comando(self, v):
        if self.cmd is not None:
            self.esperando.append(v)
            c, n = self.cmd
            if len(self.esperando) == n:
                a = self.esperando
                if c == 0x20: self.modo = a[0] & 3
                elif c == 0x21: self.c0, self.c1 = a[0] & 127, a[1] & 127; self.col = self.c0
                elif c == 0x22: self.p0, self.p1 = a[0] & 7, a[1] & 7; self.pag = self.p0
                self.cmd = None; self.esperando = []
            return
        if v in (0x20, 0x81, 0x8D, 0xA8, 0xD3, 0xD5, 0xD9, 0xDA, 0xDB): self.cmd = (v, 1)
        elif v in (0x21, 0x22): self.cmd = (v, 2)
        elif v == 0xAF: self.encendido = True
        elif v == 0xAE: self.encendido = False
        elif 0xB0 <= v <= 0xB7: self.pag = v & 7
        elif v <= 0x0F: self.col = (self.col & 0xF0) | v
        elif v <= 0x1F: self.col = ((v & 0x0F) << 4) | (self.col & 0x0F)

    def pixel(self, x, y): return bit(self.mem[y >> 3][x], y & 7)

    def ascii(self, paso=2):
        filas = []
        for y in range(0, 64, paso):
            filas.append(''.join('#' if any(self.pixel(x + i, y + j) for i in range(paso) for j in range(paso) if x + i < 128 and y + j < 64) else '.' for x in range(0, 128, paso)))
        return '\n'.join(filas)


class Teclado4x4:
    """KEYPAD-SMALLCALC: filas en RB0-RB3 (salidas), columnas en RB4-RB7 (entradas con pull-up)."""
    TECLAS = ['789/', '456*', '123-', 'C0=+']

    def __init__(self, pic):
        self.tecla = None
        pic.fn_ent['B'] = self.entradas

    def entradas(self, pic):
        v = 0xFF
        if self.tecla:
            for f, fila in enumerate(self.TECLAS):
                c = fila.find(self.tecla)
                if c >= 0 and not bit(pic.salidas['B'], f): v &= ~(1 << (4 + c))
        return v & 0xFF


class Pulso:
    """Mide el ancho y el periodo de los pulsos en un pin (servomotor)."""
    def __init__(self, pic, puerto, b):
        self.p, self.b = puerto, b; self.subida = None; self.ancho = None; self.periodo = None
        pic.escuchas.append(self.cambio)

    def cambio(self, pic, p, ant, n):
        if p != self.p or bit(ant, self.b) == bit(n, self.b): return
        t = pic.tiempo()
        if bit(n, self.b):
            if self.subida is not None: self.periodo = t - self.subida
            self.subida = t
        elif self.subida is not None: self.ancho = t - self.subida


class Paso:
    """Motor paso a paso unipolar por ULN2003: bobinas en los bits 0-3 del puerto (secuencia de medio paso)."""
    SEQ = [0b0001, 0b0011, 0b0010, 0b0110, 0b0100, 0b1100, 0b1000, 0b1001]

    def __init__(self, pic, puerto='D', grados=3.75):
        self.p = puerto; self.idx = None; self.pasos = 0; self.grados = grados; self.errores = 0
        pic.escuchas.append(self.cambio)

    def cambio(self, pic, p, ant, n):
        if p != self.p or (ant & 15) == (n & 15): return
        v = n & 15
        if v not in self.SEQ: return
        i = self.SEQ.index(v)
        if self.idx is not None:
            d = (i - self.idx) % 8
            if d == 1: self.pasos += 1
            elif d == 7: self.pasos -= 1
            elif d: self.errores += 1
        self.idx = i

    def angulo(self): return self.pasos * self.grados


class Multiplex7:
    """Display de 4 dígitos (7SEG-MPX4-CC): segmentos en PORTD (a = bit 0, dp = bit 7), comunes en RA0-RA3 activos en 0."""
    def __init__(self, pic, seg='D', dig='A'):
        self.seg, self.dig = seg, dig; self.vista = [0] * 4; self.cambios = 0
        pic.escuchas.append(self.cambio)

    def cambio(self, pic, p, ant, n):
        if p not in (self.seg, self.dig): return
        d = pic.salidas[self.dig]; s = pic.salidas[self.seg]
        for k in range(4):
            if not bit(d, k): self.vista[k] = s; self.cambios += 1

    SEGS = {0x3F: '0', 0x06: '1', 0x5B: '2', 0x4F: '3', 0x66: '4', 0x6D: '5', 0x7D: '6', 0x07: '7', 0x7F: '8', 0x6F: '9', 0: ' '}

    def texto(self):
        return ''.join(self.SEGS.get(v & 0x7F, '?') + ('.' if v & 0x80 else '') for v in self.vista)

## 2. Los programas en ensamblador
Cada archivo es autónomo (incluye sus rutinas de LCD, retardos y aritmética), así que se puede abrir directamente en MPLAB X o ensamblar con gpasm.

In [ ]:
%%writefile ej01_contador_lcd.asm
;===========================================================================
; Ejemplo 1 · Contador 0000-9999 en LCD 16x2 con tres pulsadores
; PIC16F877A · cristal 4 MHz
;   LCD: RS = RD0, E = RD1, D4..D7 = RD4..RD7 (RW a tierra)
;   Pulsadores a tierra (pull-ups internos): RB0 = subir, RB1 = bajar, RB2 = reiniciar
;===========================================================================
        LIST    P=16F877A, R=DEC
        #include <p16f877a.inc>
        __CONFIG _XT_OSC & _WDT_OFF & _PWRTE_ON & _BOREN_OFF & _LVP_OFF & _CP_OFF

        CBLOCK  0x20
        CUENTA_H, CUENTA_L, TECLA
        ENDC

        ORG     0x000
        goto    INICIO
        ORG     0x005
INICIO
        BANKSEL OPTION_REG
        bcf     OPTION_REG,NOT_RBPU     ; activa las resistencias de pull-up de PORTB
        movlw   0xFF
        movwf   TRISB                   ; PORTB como entradas
        BANKSEL PORTB
        clrf    CUENTA_H
        clrf    CUENTA_L
        call    LCD_INIT
        movlw   0x80                    ; línea 1, columna 0
        call    LCD_CMD
        movlw   HIGH TXT_TITULO
        movwf   PTR_H
        movlw   LOW TXT_TITULO
        movwf   PTR_L
        call    LCD_CADENA
        movlw   0xC0                    ; línea 2, columna 0
        call    LCD_CMD
        movlw   HIGH TXT_CUENTA
        movwf   PTR_H
        movlw   LOW TXT_CUENTA
        movwf   PTR_L
        call    LCD_CADENA
        call    MOSTRAR

LAZO    call    LEER_TECLAS             ; W = teclas presionadas (bit en 1)
        btfsc   STATUS,Z
        goto    LAZO
        movlw   20                      ; antirrebote de 20 ms
        call    RETARDO_MS
        call    LEER_TECLAS
        btfsc   STATUS,Z
        goto    LAZO                    ; fue ruido
        movwf   TECLA
        btfsc   TECLA,0
        call    SUBIR
        btfsc   TECLA,1
        call    BAJAR
        btfsc   TECLA,2
        call    REINICIAR
        call    MOSTRAR
SOLTAR  call    LEER_TECLAS             ; espera a que se suelten
        btfss   STATUS,Z
        goto    SOLTAR
        movlw   20
        call    RETARDO_MS
        goto    LAZO

LEER_TECLAS                             ; W = ~PORTB & 0x07, Z = 1 si no hay tecla
        comf    PORTB,W
        andlw   0x07
        return

SUBIR   incf    CUENTA_L,F
        btfsc   STATUS,Z
        incf    CUENTA_H,F
        movf    CUENTA_H,W              ; ¿llegó a 10000 (0x2710)?
        xorlw   0x27
        btfss   STATUS,Z
        return
        movf    CUENTA_L,W
        xorlw   0x10
        btfss   STATUS,Z
        return
REINICIAR
        clrf    CUENTA_H
        clrf    CUENTA_L
        return

BAJAR   movf    CUENTA_H,W
        iorwf   CUENTA_L,W
        btfsc   STATUS,Z
        goto    MAXIMO                  ; de 0000 pasa a 9999
        movf    CUENTA_L,W
        btfsc   STATUS,Z
        decf    CUENTA_H,F
        decf    CUENTA_L,F
        return
MAXIMO  movlw   0x27                    ; 9999 = 0x270F
        movwf   CUENTA_H
        movlw   0x0F
        movwf   CUENTA_L
        return

MOSTRAR movf    CUENTA_H,W              ; convierte a decimal y escribe 4 dígitos
        movwf   NUM_H
        movf    CUENTA_L,W
        movwf   NUM_L
        call    BCD16
        movlw   0xC8                    ; línea 2, columna 8
        call    LCD_CMD
        movf    DIG3,W
        call    LCD_NUM
        movf    DIG2,W
        call    LCD_NUM
        movf    DIG1,W
        call    LCD_NUM
        movf    DIG0,W
        goto    LCD_NUM

TXT_TITULO DT   "Contador 0-9999",0
TXT_CUENTA DT   "Cuenta:",0

;---------------------------------------------------------------------------
; Retardos para cristal de 4 MHz (1 ciclo de instrucción = 1 µs)
;---------------------------------------------------------------------------
        CBLOCK  0x68
        R_MS, R_US
        ENDC
RETARDO_MS                      ; espera W milisegundos (W = 1..255)
        movwf   R_MS
RMS1    movlw   249             ; 2 + 248*4 + 3 + 3 = 1000 ciclos por milisegundo
        movwf   R_US
RMS2    nop
        decfsz  R_US,F
        goto    RMS2
        decfsz  R_MS,F
        goto    RMS1
        return
RETARDO_50US                    ; 50 µs contando el call y el return
        movlw   15
        movwf   R_US
R50     decfsz  R_US,F
        goto    R50
        return

;---------------------------------------------------------------------------
; LCD 16x2 (HD44780 / LM016L) en modo de 4 bits sobre PORTD
;   RS = RD0, E = RD1, D4..D7 = RD4..RD7, RW a tierra
;---------------------------------------------------------------------------
        CBLOCK  0x6A
        LCD_TMP, LCD_DAT, PTR_H, PTR_L
        ENDC
LCD_INIT
        BANKSEL TRISD
        clrf    TRISD
        BANKSEL PORTD
        clrf    PORTD
        movlw   20
        call    RETARDO_MS      ; más de 15 ms después del encendido
        movlw   0x30
        call    LCD_NIB
        movlw   5
        call    RETARDO_MS
        movlw   0x30
        call    LCD_NIB
        call    RETARDO_50US
        call    RETARDO_50US
        movlw   0x30
        call    LCD_NIB
        call    RETARDO_50US
        movlw   0x20            ; pasa a modo de 4 bits
        call    LCD_NIB
        call    RETARDO_50US
        movlw   0x28            ; 4 bits, 2 líneas, caracteres de 5x8
        call    LCD_CMD
        movlw   0x0C            ; display encendido, sin cursor
        call    LCD_CMD
        movlw   0x06            ; la dirección aumenta después de cada carácter
        call    LCD_CMD
        movlw   0x01            ; borrar pantalla
        goto    LCD_CMD

LCD_CMD                         ; envía el comando que está en W
        bcf     PORTD,0         ; RS = 0
        goto    LCD_BYTE
LCD_NUM                         ; escribe el dígito 0..9 que está en W
        addlw   '0'
LCD_DATO                        ; escribe el carácter que está en W
        bsf     PORTD,0         ; RS = 1
LCD_BYTE
        movwf   LCD_DAT
        call    LCD_NIB         ; nibble alto
        swapf   LCD_DAT,W
        call    LCD_NIB         ; nibble bajo
        call    RETARDO_50US
        btfsc   PORTD,0         ; datos: listo
        return
        movlw   3               ; borrar (1) e inicio (2) tardan 1,52 ms
        subwf   LCD_DAT,W
        btfsc   STATUS,C
        return
        movlw   2
        goto    RETARDO_MS

LCD_NIB                         ; pone W<7:4> en D7..D4 y da un pulso en E
        andlw   0xF0
        movwf   LCD_TMP
        movf    PORTD,W
        andlw   0x0F
        iorwf   LCD_TMP,W
        movwf   PORTD
        bsf     PORTD,1         ; E = 1
        nop
        bcf     PORTD,1         ; E = 0: la LCD lee el dato en el flanco de bajada
        return

LCD_CADENA                      ; escribe la cadena (DT "...",0) que empieza en PTR_H:PTR_L
        call    LEER_TABLA
        iorlw   0
        btfsc   STATUS,Z
        return
        call    LCD_DATO
        incf    PTR_L,F
        btfsc   STATUS,Z
        incf    PTR_H,F
        goto    LCD_CADENA

LEER_TABLA                      ; W = byte de la tabla en PTR_H:PTR_L
        movf    PTR_H,W
        movwf   PCLATH
        movf    PTR_L,W
        movwf   PCL             ; salta a la RETLW, que vuelve a quien llamó a LEER_TABLA

;---------------------------------------------------------------------------
; Aritmética de 16 y 24 bits
;---------------------------------------------------------------------------
        CBLOCK  0x50
        NUM_H, NUM_L, K_H, K_L, TMP_L, CNT
        DIG4, DIG3, DIG2, DIG1, DIG0
        P2, P1, P0, X2, X1, X0
        ENDC
BCD16                           ; NUM_H:NUM_L (0..65535) -> DIG4..DIG0 (decenas de mil .. unidades)
        movlw   HIGH 10000
        movwf   K_H
        movlw   LOW 10000
        movwf   K_L
        call    CONTAR
        movwf   DIG4
        movlw   HIGH 1000
        movwf   K_H
        movlw   LOW 1000
        movwf   K_L
        call    CONTAR
        movwf   DIG3
        clrf    K_H
        movlw   100
        movwf   K_L
        call    CONTAR
        movwf   DIG2
        movlw   10
        movwf   K_L
        call    CONTAR
        movwf   DIG1
        movf    NUM_L,W
        movwf   DIG0
        return
CONTAR                          ; W = cuántas veces cabe K en NUM (y se lo resta)
        clrf    CNT
CT1     movf    K_L,W
        subwf   NUM_L,W         ; W = NUM_L - K_L   (C = 0 si hubo préstamo)
        movwf   TMP_L
        movf    K_H,W
        btfss   STATUS,C
        addlw   1               ; el préstamo se suma al byte alto que se resta
        subwf   NUM_H,W
        btfss   STATUS,C
        goto    CT2             ; ya no cabe
        movwf   NUM_H
        movf    TMP_L,W
        movwf   NUM_L
        incf    CNT,F
        goto    CT1
CT2     movf    CNT,W
        return

MUL16                           ; P2:P1:P0 = NUM_H:NUM_L x K_H:K_L  (resultado menor que 2^24)
        clrf    P0
        clrf    P1
        clrf    P2
        clrf    X2
        movf    NUM_H,W
        movwf   X1
        movf    NUM_L,W
        movwf   X0
        movlw   16
        movwf   CNT
ML1     bcf     STATUS,C
        rrf     K_H,F
        rrf     K_L,F           ; bit menos significativo de K en C
        btfss   STATUS,C
        goto    ML2
        movf    X0,W            ; P = P + X (24 bits)
        addwf   P0,F
        movf    X1,W
        btfsc   STATUS,C
        incfsz  X1,W
        addwf   P1,F
        movf    X2,W
        btfsc   STATUS,C
        incfsz  X2,W
        addwf   P2,F
ML2     bcf     STATUS,C        ; X = X * 2
        rlf     X0,F
        rlf     X1,F
        rlf     X2,F
        decfsz  CNT,F
        goto    ML1
        return

        END

In [ ]:
%%writefile ej02_oled_juegos.asm
;===========================================================================
; Ejemplo 2 · Tres juegos en una pantalla OLED 128x64 (SSD1306 por SPI) con pulsadores
;             1) PONG   2) CARRERA (esquivar obstáculos)   3) SALTO (dinosaurio)
; PIC16F877A · cristal de 20 MHz (1 ciclo = 0,2 µs)
;   OLED (UG-2864HSWEG01 en Proteus): D0/SCK = RC3, D1/SDA = RC5, CS = RC0, D/C = RC1, RES = RC2
;   Pulsadores a tierra (pull-ups internos): RB0 = ARRIBA, RB1 = ABAJO, RB2 = OK / SALTAR
;===========================================================================
        LIST    P=16F877A, R=DEC
        #include <p16f877a.inc>
        __CONFIG _HS_OSC & _WDT_OFF & _PWRTE_ON & _BOREN_OFF & _LVP_OFF & _CP_OFF

        CBLOCK  0x20
        BTN, PREV, NUEVO, SEL                   ; pulsadores y menú
        PX, PY, N, PAT, AUX, CH, I              ; dibujo
        PTR_H, PTR_L, FP_H, FP_L                ; punteros a cadenas y a la fuente
        R1, R2, R3, RND, SCORE, D100, D10
        BX, BY, DIRX, DIRY, PAD, VIDAS          ; pong
        CAR, NUEVA_P, SPAWN, OX, OP             ; carrera
        OBS:6                                   ; carrera: tres obstáculos (x, página)
        DINO, JUMP, CX, ESPERA                  ; salto
        ENDC

#define CS      PORTC,0
#define DC      PORTC,1
#define RES     PORTC,2

        ORG     0x000
        goto    INICIO
        ORG     0x005

;===================== tablas (cerca del inicio) =====================
TABLA_SALTO                                     ; página del dinosaurio en cada cuadro del salto
        clrf    PCLATH                          ; las tablas están en las direcciones 0x000-0x0FF
        addwf   PCL,F
        DT      4,3,2,2,2,3,4,5
SPR_CARRO
        clrf    PCLATH
        addwf   PCL,F
        DT      0x24,0x5A,0xFF,0xFF,0xFF,0xFF,0x5A,0x24
SPR_ROCA
        clrf    PCLATH
        addwf   PCL,F
        DT      0x1E,0x3F,0x7F,0x7F,0x7F,0x7E,0x3C,0x18
SPR_DINO
        clrf    PCLATH
        addwf   PCL,F
        DT      0x18,0x30,0xE0,0x78,0xFF,0x3D,0x17,0x07
SPR_CACTUS
        clrf    PCLATH
        addwf   PCL,F
        DT      0x1C,0x10,0xFF,0x08,0x0E,0x00

TABLA_INIT                                      ; secuencia de inicio del SSD1306
        clrf    PCLATH
        addwf   PCL,F
        DT      0xAE, 0xD5,0x80, 0xA8,0x3F, 0xD3,0x00, 0x40, 0x8D,0x14, 0x20,0x00
        DT      0xA1, 0xC8, 0xDA,0x12, 0x81,0xCF, 0xD9,0xF1, 0xDB,0x40, 0xA4, 0xA6, 0xAF
FIN_INIT

;===================== inicio =====================
INICIO
        BANKSEL OPTION_REG
        bcf     OPTION_REG,NOT_RBPU             ; pull-ups de PORTB
        movlw   b'11010000'                     ; RC0, RC1, RC2, RC3, RC5 salidas; RC4 (SDI) entrada
        movwf   TRISC
        movlw   b'01000000'                     ; CKE = 1 (dato estable en el flanco de subida)
        movwf   SSPSTAT
        BANKSEL PORTC
        movlw   b'00100000'                     ; SSPEN = 1, SPI maestro, reloj Fosc/4 = 5 MHz
        movwf   SSPCON
        movlw   0xA5
        movwf   RND
        clrf    SEL
        clrf    PREV
        call    OLED_INIT

;===================== menú =====================
MENU    call    BORRAR
        movlw   28
        movwf   PX
        clrf    PY
        movlw   HIGH TXT_TITULO
        movwf   PTR_H
        movlw   LOW TXT_TITULO
        call    TEXTO
        movlw   22
        movwf   PX
        movlw   2
        movwf   PY
        movlw   HIGH TXT_PONG
        movwf   PTR_H
        movlw   LOW TXT_PONG
        call    TEXTO
        movlw   22
        movwf   PX
        movlw   3
        movwf   PY
        movlw   HIGH TXT_CARRERA
        movwf   PTR_H
        movlw   LOW TXT_CARRERA
        call    TEXTO
        movlw   22
        movwf   PX
        movlw   4
        movwf   PY
        movlw   HIGH TXT_SALTO
        movwf   PTR_H
        movlw   LOW TXT_SALTO
        call    TEXTO
        clrf    PX
        movlw   7
        movwf   PY
        movlw   HIGH TXT_AYUDA
        movwf   PTR_H
        movlw   LOW TXT_AYUDA
        call    TEXTO
MENU_LAZO
        call    CURSOR_DIBUJAR
        movlw   30
        call    RET_MS
        call    RANDOM                          ; el azar depende de cuánto tarda el jugador
        call    LEER_BOTONES
        btfsc   NUEVO,0
        goto    MENU_ARRIBA
        btfsc   NUEVO,1
        goto    MENU_ABAJO
        btfss   NUEVO,2
        goto    MENU_LAZO
        movf    SEL,W                           ; OK: inicia el juego elegido
        btfsc   STATUS,Z
        goto    PONG
        xorlw   1
        btfsc   STATUS,Z
        goto    CARRERA
        goto    SALTO
MENU_ARRIBA
        call    CURSOR_BORRAR
        decf    SEL,F
        btfss   SEL,7
        goto    MENU_LAZO
        movlw   2
        movwf   SEL
        goto    MENU_LAZO
MENU_ABAJO
        call    CURSOR_BORRAR
        incf    SEL,F
        movlw   3
        xorwf   SEL,W
        btfsc   STATUS,Z
        clrf    SEL
        goto    MENU_LAZO

CURSOR_DIBUJAR
        movlw   '>'
        goto    CURSOR
CURSOR_BORRAR
        movlw   ' '
CURSOR  movwf   CH
        movlw   12
        movwf   PX
        movf    SEL,W
        addlw   2
        movwf   PY
        call    VENTANA
        movf    CH,W
        goto    CARACTER

;===================== juego 1: PONG =====================
PONG    call    BORRAR
        clrf    SCORE
        movlw   3
        movwf   VIDAS
        movlw   3
        movwf   PAD
        movlw   127                             ; pared derecha
        movwf   PX
        clrf    PY
        call    VENTANA
        movlw   8
        movwf   N
        movlw   0xFF
        call    RELLENAR_COL
PONG_BOLA
        movlw   64
        movwf   BX
        movlw   32
        movwf   BY
        movlw   1
        movwf   DIRX                            ; hacia la izquierda
        call    RANDOM
        andlw   1
        movwf   DIRY
PONG_LAZO
        call    LEER_BOTONES
        btfss   BTN,0                           ; raqueta arriba
        goto    PONG_B1
        movf    PAD,W
        btfss   STATUS,Z
        decf    PAD,F
PONG_B1 btfss   BTN,1                           ; raqueta abajo
        goto    PONG_B2
        movlw   6
        subwf   PAD,W
        btfss   STATUS,C
        incf    PAD,F
PONG_B2 call    BOLA_BORRAR
        ;--- movimiento vertical
        btfsc   DIRY,0
        goto    PONG_SUBE
        movlw   60
        xorwf   BY,W
        btfsc   STATUS,Z
        goto    PONG_REBOTE_Y
        movlw   4
        addwf   BY,F
        goto    PONG_X
PONG_SUBE
        movf    BY,W
        btfsc   STATUS,Z
        goto    PONG_REBOTE_Y
        movlw   4
        subwf   BY,F
        goto    PONG_X
PONG_REBOTE_Y
        movlw   1
        xorwf   DIRY,F
        ;--- movimiento horizontal
PONG_X  btfsc   DIRX,0
        goto    PONG_IZQ
        movlw   120
        xorwf   BX,W
        btfsc   STATUS,Z
        goto    PONG_REBOTE_X
        movlw   4
        addwf   BX,F
        goto    PONG_DIBUJO
PONG_IZQ
        movlw   8                               ; ¿la bola llegó a la raqueta?
        xorwf   BX,W
        btfss   STATUS,Z
        goto    PONG_IZQ2
        rrf     BY,W                            ; página de la bola = BY / 8
        movwf   AUX
        rrf     AUX,F
        rrf     AUX,W
        andlw   7
        movwf   AUX
        movf    PAD,W
        subwf   AUX,W                           ; página - PAD debe ser 0 o 1
        sublw   1
        btfss   STATUS,C
        goto    PONG_IZQ2
        incf    SCORE,F                         ; golpe: un punto
        goto    PONG_REBOTE_X
PONG_IZQ2
        movf    BX,W
        btfsc   STATUS,Z
        goto    PONG_FALLA
        movlw   4
        subwf   BX,F
        goto    PONG_DIBUJO
PONG_REBOTE_X
        movlw   1
        xorwf   DIRX,F
        goto    PONG_DIBUJO
PONG_FALLA
        decfsz  VIDAS,F
        goto    PONG_BOLA
        goto    FIN_JUEGO
PONG_DIBUJO
        call    RAQUETA
        call    BOLA_DIBUJAR
        movlw   25
        call    RET_MS
        goto    PONG_LAZO

RAQUETA                                         ; raqueta de 3x16 px en x = 2..4, páginas PAD y PAD+1
        movlw   2
        movwf   PX
        movf    PAD,W
        btfsc   STATUS,Z
        goto    RAQ1
        movwf   PY
        decf    PY,F
        movlw   0x00                            ; borra la página de arriba
        call    COLUMNA3
RAQ1    movf    PAD,W
        movwf   PY
        movlw   0xFF
        call    COLUMNA3
        incf    PY,F
        movlw   0xFF
        call    COLUMNA3
        incf    PY,F
        btfsc   PY,3                            ; PY = 8: no hay página de abajo
        return
        movlw   0x00
COLUMNA3                                        ; 3 bytes con el patrón W en (PX, PY)
        movwf   PAT
        call    VENTANA
        movlw   3
        movwf   N
        movf    PAT,W
        goto    RELLENAR

BOLA_BORRAR
        clrw
        goto    BOLA
BOLA_DIBUJAR
        movlw   0x0F                            ; mitad superior o inferior de la página
        btfsc   BY,2
        movlw   0xF0
BOLA    movwf   PAT
        movf    BX,W
        movwf   PX
        rrf     BY,W
        movwf   PY
        rrf     PY,F
        rrf     PY,W
        andlw   7
        movwf   PY
        call    VENTANA
        movlw   4
        movwf   N
        movf    PAT,W
        goto    RELLENAR

;===================== juego 2: CARRERA =====================
CARRERA call    BORRAR
        clrf    SCORE
        call    MARCADOR_TEXTO
        movlw   4
        movwf   CAR
        clrf    OBS+1                           ; página 0 = obstáculo inactivo
        clrf    OBS+3
        clrf    OBS+5
        movlw   1
        movwf   SPAWN
CAR_LAZO
        call    LEER_BOTONES
        movf    CAR,W
        movwf   NUEVA_P
        btfss   NUEVO,0
        goto    CAR_B1
        movlw   1
        xorwf   CAR,W
        btfss   STATUS,Z
        decf    NUEVA_P,F
CAR_B1  btfss   NUEVO,1
        goto    CAR_B2
        movlw   7
        xorwf   CAR,W
        btfss   STATUS,Z
        incf    NUEVA_P,F
CAR_B2  movf    CAR,W                           ; si cambió de carril, borra el carro
        xorwf   NUEVA_P,W
        btfsc   STATUS,Z
        goto    CAR_OBS
        movlw   4
        movwf   PX
        movf    CAR,W
        movwf   PY
        call    VENTANA
        movlw   8
        movwf   N
        clrw
        call    RELLENAR
        movf    NUEVA_P,W
        movwf   CAR
CAR_OBS movlw   OBS                             ; mueve los tres obstáculos
        call    OBSTACULO
        movlw   OBS+2
        call    OBSTACULO
        movlw   OBS+4
        call    OBSTACULO
        decfsz  SPAWN,F                         ; ¿aparece un obstáculo nuevo?
        goto    CAR_DIBUJO
        call    RANDOM
        andlw   7
        addlw   5
        movwf   SPAWN
        movlw   OBS+1
        movwf   FSR
        movf    INDF,W
        btfsc   STATUS,Z
        goto    CAR_ACTIVAR
        movlw   OBS+3
        movwf   FSR
        movf    INDF,W
        btfsc   STATUS,Z
        goto    CAR_ACTIVAR
        movlw   OBS+5
        movwf   FSR
        movf    INDF,W
        btfss   STATUS,Z
        goto    CAR_DIBUJO                      ; los tres están en pantalla
CAR_ACTIVAR
        call    RANDOM                          ; carril aleatorio 1..7
        andlw   7
        btfsc   STATUS,Z
        movlw   7
        movwf   INDF
        decf    FSR,F
        movlw   116
        movwf   INDF
CAR_DIBUJO
        movlw   4                               ; dibuja el carro del jugador
        movwf   PX
        movf    CAR,W
        movwf   PY
        call    VENTANA
        clrf    I
CAR_SPR movf    I,W
        call    SPR_CARRO
        call    SPI
        incf    I,F
        btfss   I,3
        goto    CAR_SPR
        call    MARCADOR
        movlw   30
        call    RET_MS
        goto    CAR_LAZO

OBSTACULO                                       ; W = dirección del obstáculo (x, página)
        movwf   FSR
        movf    INDF,W
        movwf   OX
        incf    FSR,F
        movf    INDF,W
        movwf   OP
        btfsc   STATUS,Z
        return                                  ; inactivo
        movf    OX,W
        movwf   PX
        movf    OP,W
        movwf   PY
        movf    OX,W
        btfss   STATUS,Z
        goto    OBS_MOVER
        clrf    INDF                            ; llegó al borde: desaparece y suma un punto
        incf    SCORE,F
        call    VENTANA
        movlw   12
        movwf   N
        clrw
        goto    RELLENAR
OBS_MOVER
        movlw   4
        subwf   OX,F
        decf    FSR,F
        movf    OX,W
        movwf   INDF
        movwf   PX
        call    VENTANA                         ; roca de 8 px y 4 columnas en blanco detrás
        clrf    I
OBS_SPR movf    I,W
        call    SPR_ROCA
        call    SPI
        incf    I,F
        btfss   I,3
        goto    OBS_SPR
        movlw   4
        movwf   N
        clrw
        call    RELLENAR
        movf    OP,W                            ; choque: mismo carril y x < 12
        xorwf   CAR,W
        btfss   STATUS,Z
        return
        movlw   12
        subwf   OX,W
        btfsc   STATUS,C
        return
        goto    FIN_JUEGO_POP

;===================== juego 3: SALTO =====================
SALTO   call    BORRAR
        clrf    SCORE
        call    MARCADOR_TEXTO
        clrf    PX                              ; suelo
        movlw   6
        movwf   PY
        call    VENTANA
        movlw   128
        movwf   N
        movlw   0x01
        call    RELLENAR
        movlw   5
        movwf   DINO
        clrf    JUMP
        movlw   116
        movwf   CX
        movlw   10
        movwf   ESPERA
SAL_LAZO
        call    LEER_BOTONES
        movf    NUEVO,W                         ; OK o ARRIBA: saltar
        andlw   b'00000101'
        btfsc   STATUS,Z
        goto    SAL_AIRE
        movf    JUMP,W
        btfsc   STATUS,Z
        incf    JUMP,F
SAL_AIRE
        movf    JUMP,W
        btfsc   STATUS,Z
        goto    SAL_CACTUS
        addlw   255                             ; W = JUMP - 1
        call    TABLA_SALTO
        movwf   NUEVA_P
        incf    JUMP,F
        movlw   9
        xorwf   JUMP,W
        btfsc   STATUS,Z
        clrf    JUMP
        movf    DINO,W                          ; borra el dinosaurio en su página anterior
        xorwf   NUEVA_P,W
        btfsc   STATUS,Z
        goto    SAL_CACTUS
        movlw   8
        movwf   PX
        movf    DINO,W
        movwf   PY
        call    VENTANA
        movlw   8
        movwf   N
        clrw
        call    RELLENAR
        movf    NUEVA_P,W
        movwf   DINO
SAL_CACTUS
        movf    ESPERA,W
        btfsc   STATUS,Z
        goto    SAL_MOVER
        decf    ESPERA,F
        goto    SAL_DINO
SAL_MOVER
        movf    CX,W
        movwf   PX
        movlw   5
        movwf   PY
        movf    CX,W
        btfss   STATUS,Z
        goto    SAL_AVANZA
        call    VENTANA                         ; salió de la pantalla: punto y nuevo cactus
        movlw   10
        movwf   N
        clrw
        call    RELLENAR
        incf    SCORE,F
        movlw   116
        movwf   CX
        call    RANDOM
        andlw   15
        movwf   ESPERA
        goto    SAL_DINO
SAL_AVANZA
        movlw   4
        subwf   CX,F
        movf    CX,W
        movwf   PX
        call    VENTANA
        clrf    I
SAL_SPR movf    I,W
        call    SPR_CACTUS
        call    SPI
        incf    I,F
        movlw   6
        xorwf   I,W
        btfss   STATUS,Z
        goto    SAL_SPR
        movlw   4
        movwf   N
        clrw
        call    RELLENAR
        movlw   5                               ; choque: dinosaurio en el suelo y 4 <= CX < 16
        xorwf   DINO,W
        btfss   STATUS,Z
        goto    SAL_DINO
        movlw   16
        subwf   CX,W
        btfsc   STATUS,C
        goto    SAL_DINO
        movlw   4
        subwf   CX,W
        btfsc   STATUS,C
        goto    FIN_JUEGO
SAL_DINO
        movlw   8
        movwf   PX
        movf    DINO,W
        movwf   PY
        call    VENTANA
        clrf    I
SAL_DSPR
        movf    I,W
        call    SPR_DINO
        call    SPI
        incf    I,F
        btfss   I,3
        goto    SAL_DSPR
        call    MARCADOR
        movlw   35
        call    RET_MS
        goto    SAL_LAZO

;===================== fin del juego =====================
FIN_JUEGO_POP                                   ; se llega desde una subrutina: descarta su retorno
FIN_JUEGO
        movlw   250
        call    RET_MS
        call    BORRAR
        movlw   22
        movwf   PX
        movlw   2
        movwf   PY
        movlw   HIGH TXT_FIN
        movwf   PTR_H
        movlw   LOW TXT_FIN
        call    TEXTO
        movlw   28
        movwf   PX
        movlw   4
        movwf   PY
        movlw   HIGH TXT_PUNTOS
        movwf   PTR_H
        movlw   LOW TXT_PUNTOS
        call    TEXTO
        call    NUMERO
        movlw   34
        movwf   PX
        movlw   6
        movwf   PY
        movlw   HIGH TXT_OK
        movwf   PTR_H
        movlw   LOW TXT_OK
        call    TEXTO
FIN_ESPERA
        movlw   30
        call    RET_MS
        call    LEER_BOTONES
        btfss   NUEVO,2
        goto    FIN_ESPERA
        goto    MENU

MARCADOR_TEXTO                                  ; "PUNTOS" en la página 0
        clrf    PX
        clrf    PY
        movlw   HIGH TXT_PUNTOS
        movwf   PTR_H
        movlw   LOW TXT_PUNTOS
        goto    TEXTO
MARCADOR                                        ; número en la página 0
        movlw   48
        movwf   PX
        clrf    PY
        call    VENTANA
NUMERO                                          ; escribe SCORE con 3 dígitos en la posición actual
        movf    SCORE,W
        movwf   AUX
        clrf    D100
        clrf    D10
NUM1    movlw   100
        subwf   AUX,W
        btfss   STATUS,C
        goto    NUM2
        movwf   AUX
        incf    D100,F
        goto    NUM1
NUM2    movlw   10
        subwf   AUX,W
        btfss   STATUS,C
        goto    NUM3
        movwf   AUX
        incf    D10,F
        goto    NUM2
NUM3    movf    D100,W
        addlw   '0'
        call    CARACTER
        movf    D10,W
        addlw   '0'
        call    CARACTER
        movf    AUX,W
        addlw   '0'
        goto    CARACTER

;===================== utilidades =====================
LEER_BOTONES                                    ; BTN = presionados, NUEVO = recién presionados
        comf    PORTB,W
        andlw   0x07
        movwf   BTN
        comf    PREV,W
        andwf   BTN,W
        movwf   NUEVO
        movf    BTN,W
        movwf   PREV
        return

RANDOM                                          ; registro de desplazamiento con realimentación (LFSR)
        bcf     STATUS,C
        rrf     RND,F
        movlw   0xB8
        btfsc   STATUS,C
        xorwf   RND,F
        movf    RND,W
        return

RET_MS                                          ; W milisegundos a 20 MHz (5000 ciclos por ms)
        movwf   R1
RM1     movlw   5
        movwf   R2
RM2     movlw   249
        movwf   R3
RM3     nop
        decfsz  R3,F
        goto    RM3
        decfsz  R2,F
        goto    RM2
        decfsz  R1,F
        goto    RM1
        return

;===================== controlador SSD1306 =====================
SPI                                             ; envía W por SPI y espera a que termine
        movwf   SSPBUF
SPI1    btfss   PIR1,SSPIF
        goto    SPI1
        bcf     PIR1,SSPIF
        movf    SSPBUF,W                        ; lee el dato recibido para limpiar BF
        return

COMANDO bcf     DC
        goto    SPI

OLED_INIT
        bcf     CS                              ; la pantalla queda seleccionada
        bcf     RES                             ; reinicio por hardware
        movlw   10
        call    RET_MS
        bsf     RES
        movlw   10
        call    RET_MS
        clrf    I
OI1     movf    I,W
        call    TABLA_INIT
        call    COMANDO
        incf    I,F
        movlw   FIN_INIT - TABLA_INIT - 2
        xorwf   I,W
        btfss   STATUS,Z
        goto    OI1
        goto    BORRAR

VENTANA                                         ; escribe a partir de la columna PX y la página PY
        bcf     DC
        movlw   0x21
        call    SPI
        movf    PX,W
        call    SPI
        movlw   127
        call    SPI
        movlw   0x22
        call    SPI
        movf    PY,W
        call    SPI
        movlw   7
        call    SPI
        bsf     DC                              ; lo siguiente son datos
        return

RELLENAR                                        ; envía N bytes iguales a W
        movwf   PAT
RL1     movf    PAT,W
        call    SPI
        decfsz  N,F
        goto    RL1
        return

RELLENAR_COL                                    ; N páginas de una sola columna (PX fijo)
        movwf   PAT
RC1     bcf     DC
        movlw   0x21
        call    SPI
        movf    PX,W
        call    SPI
        movf    PX,W
        call    SPI
        bsf     DC
        movf    PAT,W
        call    SPI
        decfsz  N,F
        goto    RC1
        return

BORRAR  clrf    PX
        clrf    PY
        call    VENTANA
        movlw   4
        movwf   AUX
BO1     clrf    N                               ; 256 bytes
        clrw
        call    RELLENAR
        decfsz  AUX,F
        goto    BO1
        return

TEXTO                                           ; cadena PTR_H:W en (PX, PY)
        movwf   PTR_L
        call    VENTANA
TX1     call    LEER_PTR
        iorlw   0
        btfsc   STATUS,Z
        return
        call    CARACTER
        incf    PTR_L,F
        btfsc   STATUS,Z
        incf    PTR_H,F
        goto    TX1

CARACTER                                        ; dibuja el carácter W (5x7 + 1 columna de espacio)
        addlw   -32
        movwf   FP_L
        clrf    FP_H
        bcf     STATUS,C                        ; FP = (W - 32) x 5 + FUENTE
        rlf     FP_L,F
        rlf     FP_H,F
        rlf     FP_L,F
        rlf     FP_H,F
        addwf   FP_L,F
        btfsc   STATUS,C
        incf    FP_H,F
        movlw   LOW FUENTE
        addwf   FP_L,F
        btfsc   STATUS,C
        incf    FP_H,F
        movlw   HIGH FUENTE
        addwf   FP_H,F
        movlw   5
        movwf   I
CA1     call    LEER_FUENTE
        call    SPI
        incf    FP_L,F
        btfsc   STATUS,Z
        incf    FP_H,F
        decfsz  I,F
        goto    CA1
        movlw   0
        goto    SPI

LEER_PTR
        movf    PTR_H,W
        movwf   PCLATH
        movf    PTR_L,W
        movwf   PCL
LEER_FUENTE
        movf    FP_H,W
        movwf   PCLATH
        movf    FP_L,W
        movwf   PCL

;===================== textos y fuente =====================
TXT_TITULO  DT  "PIC JUEGOS",0
TXT_PONG    DT  "1 PONG",0
TXT_CARRERA DT  "2 CARRERA",0
TXT_SALTO   DT  "3 SALTO",0
TXT_AYUDA   DT  "ARRIBA/ABAJO  OK",0
TXT_FIN     DT  "FIN DEL JUEGO",0
TXT_PUNTOS  DT  "PUNTOS ",0
TXT_OK      DT  "OK: MENU",0

FUENTE                                          ; ASCII 32 (espacio) a 90 (Z), 5 columnas por carácter
        DT 0x00,0x00,0x00,0x00,0x00, 0x00,0x00,0x5F,0x00,0x00, 0x00,0x07,0x00,0x07,0x00, 0x14,0x7F,0x14,0x7F,0x14
        DT 0x24,0x2A,0x7F,0x2A,0x12, 0x23,0x13,0x08,0x64,0x62, 0x36,0x49,0x55,0x22,0x50, 0x00,0x05,0x03,0x00,0x00
        DT 0x00,0x1C,0x22,0x41,0x00, 0x00,0x41,0x22,0x1C,0x00, 0x08,0x2A,0x1C,0x2A,0x08, 0x08,0x08,0x3E,0x08,0x08
        DT 0x00,0x50,0x30,0x00,0x00, 0x08,0x08,0x08,0x08,0x08, 0x00,0x60,0x60,0x00,0x00, 0x20,0x10,0x08,0x04,0x02
        DT 0x3E,0x51,0x49,0x45,0x3E, 0x00,0x42,0x7F,0x40,0x00, 0x42,0x61,0x51,0x49,0x46, 0x21,0x41,0x45,0x4B,0x31
        DT 0x18,0x14,0x12,0x7F,0x10, 0x27,0x45,0x45,0x45,0x39, 0x3C,0x4A,0x49,0x49,0x30, 0x01,0x71,0x09,0x05,0x03
        DT 0x36,0x49,0x49,0x49,0x36, 0x06,0x49,0x49,0x29,0x1E, 0x00,0x36,0x36,0x00,0x00, 0x00,0x56,0x36,0x00,0x00
        DT 0x08,0x14,0x22,0x41,0x00, 0x14,0x14,0x14,0x14,0x14, 0x00,0x41,0x22,0x14,0x08, 0x02,0x01,0x51,0x09,0x06
        DT 0x32,0x49,0x79,0x41,0x3E, 0x7E,0x11,0x11,0x11,0x7E, 0x7F,0x49,0x49,0x49,0x36, 0x3E,0x41,0x41,0x41,0x22
        DT 0x7F,0x41,0x41,0x22,0x1C, 0x7F,0x49,0x49,0x49,0x41, 0x7F,0x09,0x09,0x01,0x01, 0x3E,0x41,0x41,0x51,0x32
        DT 0x7F,0x08,0x08,0x08,0x7F, 0x00,0x41,0x7F,0x41,0x00, 0x20,0x40,0x41,0x3F,0x01, 0x7F,0x08,0x14,0x22,0x41
        DT 0x7F,0x40,0x40,0x40,0x40, 0x7F,0x02,0x04,0x02,0x7F, 0x7F,0x04,0x08,0x10,0x7F, 0x3E,0x41,0x41,0x41,0x3E
        DT 0x7F,0x09,0x09,0x09,0x06, 0x3E,0x41,0x51,0x21,0x5E, 0x7F,0x09,0x19,0x29,0x46, 0x46,0x49,0x49,0x49,0x31
        DT 0x01,0x01,0x7F,0x01,0x01, 0x3F,0x40,0x40,0x40,0x3F, 0x1F,0x20,0x40,0x20,0x1F, 0x7F,0x20,0x18,0x20,0x7F
        DT 0x63,0x14,0x08,0x14,0x63, 0x03,0x04,0x78,0x04,0x03, 0x61,0x51,0x49,0x45,0x43
        END

In [ ]:
%%writefile ej03_sensores.asm
;===========================================================================
; Ejemplo 3 · Sensores: temperatura (LM35 en AN0) y luz (LDR en AN1) en LCD 16x2
; PIC16F877A · cristal 4 MHz
;   LM35: 10 mV/°C -> AN0 (RA0).  LDR entre +5 V y AN1 (RA1), 10 kΩ de AN1 a tierra.
;   RB0 = ventilador (se enciende si T > 30,0 °C) · RB1 = lámpara (se enciende si luz < 30 %)
;   LCD: RS = RD0, E = RD1, D4..D7 = RD4..RD7
;===========================================================================
        LIST    P=16F877A, R=DEC
        #include <p16f877a.inc>
        __CONFIG _XT_OSC & _WDT_OFF & _PWRTE_ON & _BOREN_OFF & _LVP_OFF & _CP_OFF

        CBLOCK  0x20
        ADC_H, ADC_L, T_H, T_L, LUZ
        ENDC

        ORG     0x000
        goto    INICIO
        ORG     0x005
INICIO
        BANKSEL TRISB
        clrf    TRISB                   ; PORTB salidas (actuadores)
        movlw   0x84                    ; justificado a la derecha, AN0, AN1 y AN3 analógicos
        movwf   ADCON1
        BANKSEL PORTB
        clrf    PORTB
        call    LCD_INIT
        movlw   0x80
        call    LCD_CMD
        movlw   HIGH TXT_T
        movwf   PTR_H
        movlw   LOW TXT_T
        movwf   PTR_L
        call    LCD_CADENA
        movlw   0xC0
        call    LCD_CMD
        movlw   HIGH TXT_L
        movwf   PTR_H
        movlw   LOW TXT_L
        movwf   PTR_L
        call    LCD_CADENA

LAZO
        ;----- temperatura: décimas de °C = ADC x 625 / 128  (ADC x 4,883 mV / 10 mV/°C x 10)
        movlw   0x41                    ; Fosc/8, canal 0, ADC encendido
        call    LEER_ADC
        movf    ADC_H,W
        movwf   NUM_H
        movf    ADC_L,W
        movwf   NUM_L
        movlw   HIGH 625
        movwf   K_H
        movlw   LOW 625
        movwf   K_L
        call    MUL16                   ; P2:P1:P0 = ADC x 625
        rlf     P0,F                    ; (P << 1) >> 8 = P >> 7
        rlf     P1,F
        rlf     P2,F
        movf    P2,W
        movwf   T_H
        movwf   NUM_H
        movf    P1,W
        movwf   T_L
        movwf   NUM_L
        call    BCD16
        movlw   0x86                    ; línea 1, columna 6: " 25.3°C"
        call    LCD_CMD
        movf    DIG3,W                  ; centenas (espacio si es 0)
        btfsc   STATUS,Z
        movlw   ' ' - '0'
        call    LCD_NUM
        movf    DIG2,W
        call    LCD_NUM
        movf    DIG1,W
        call    LCD_NUM
        movlw   '.'
        call    LCD_DATO
        movf    DIG0,W
        call    LCD_NUM
        movlw   0xDF                    ; símbolo de grado en la LCD
        call    LCD_DATO
        movlw   'C'
        call    LCD_DATO
        ; ventilador: T > 300 décimas  <=>  T - 301 >= 0
        movlw   LOW 301
        subwf   T_L,W
        movlw   HIGH 301
        btfss   STATUS,C
        addlw   1
        subwf   T_H,W
        btfsc   STATUS,C
        bsf     PORTB,0
        btfss   STATUS,C
        bcf     PORTB,0

        ;----- luz: porcentaje = ADC x 25 / 256  (0..99 %)
        movlw   0x49                    ; Fosc/8, canal 1, ADC encendido
        call    LEER_ADC
        movf    ADC_H,W
        movwf   NUM_H
        movf    ADC_L,W
        movwf   NUM_L
        clrf    K_H
        movlw   25
        movwf   K_L
        call    MUL16
        movf    P1,W                    ; P >> 8 (cabe en un byte)
        movwf   LUZ
        clrf    NUM_H
        movwf   NUM_L
        call    BCD16
        movlw   0xC7
        call    LCD_CMD
        movf    DIG2,W                  ; centenas (espacio si es 0)
        btfsc   STATUS,Z
        movlw   ' ' - '0'
        call    LCD_NUM
        movf    DIG2,W                  ; decenas (espacio si centenas y decenas son 0)
        iorwf   DIG1,W
        btfsc   STATUS,Z
        goto    SIN_DEC
        movf    DIG1,W
        goto    CON_DEC
SIN_DEC movlw   ' ' - '0'
CON_DEC call    LCD_NUM
        movf    DIG0,W
        call    LCD_NUM
        movlw   '%'
        call    LCD_DATO
        movlw   30                      ; lámpara: luz < 30 %
        subwf   LUZ,W
        btfss   STATUS,C
        bsf     PORTB,1
        btfsc   STATUS,C
        bcf     PORTB,1

        movlw   250
        call    RETARDO_MS
        goto    LAZO

LEER_ADC                                ; W = ADCON0; deja el resultado de 10 bits en ADC_H:ADC_L
        movwf   ADCON0
        call    RETARDO_50US            ; tiempo de adquisición (> 20 µs)
        bsf     ADCON0,GO
ESPERA  btfsc   ADCON0,GO
        goto    ESPERA
        movf    ADRESH,W
        movwf   ADC_H
        BANKSEL ADRESL
        movf    ADRESL,W
        BANKSEL PORTB
        movwf   ADC_L
        return

TXT_T   DT      "Temp:",0
TXT_L   DT      "Luz:",0

;---------------------------------------------------------------------------
; Retardos para cristal de 4 MHz (1 ciclo de instrucción = 1 µs)
;---------------------------------------------------------------------------
        CBLOCK  0x68
        R_MS, R_US
        ENDC
RETARDO_MS                      ; espera W milisegundos (W = 1..255)
        movwf   R_MS
RMS1    movlw   249             ; 2 + 248*4 + 3 + 3 = 1000 ciclos por milisegundo
        movwf   R_US
RMS2    nop
        decfsz  R_US,F
        goto    RMS2
        decfsz  R_MS,F
        goto    RMS1
        return
RETARDO_50US                    ; 50 µs contando el call y el return
        movlw   15
        movwf   R_US
R50     decfsz  R_US,F
        goto    R50
        return

;---------------------------------------------------------------------------
; LCD 16x2 (HD44780 / LM016L) en modo de 4 bits sobre PORTD
;   RS = RD0, E = RD1, D4..D7 = RD4..RD7, RW a tierra
;---------------------------------------------------------------------------
        CBLOCK  0x6A
        LCD_TMP, LCD_DAT, PTR_H, PTR_L
        ENDC
LCD_INIT
        BANKSEL TRISD
        clrf    TRISD
        BANKSEL PORTD
        clrf    PORTD
        movlw   20
        call    RETARDO_MS      ; más de 15 ms después del encendido
        movlw   0x30
        call    LCD_NIB
        movlw   5
        call    RETARDO_MS
        movlw   0x30
        call    LCD_NIB
        call    RETARDO_50US
        call    RETARDO_50US
        movlw   0x30
        call    LCD_NIB
        call    RETARDO_50US
        movlw   0x20            ; pasa a modo de 4 bits
        call    LCD_NIB
        call    RETARDO_50US
        movlw   0x28            ; 4 bits, 2 líneas, caracteres de 5x8
        call    LCD_CMD
        movlw   0x0C            ; display encendido, sin cursor
        call    LCD_CMD
        movlw   0x06            ; la dirección aumenta después de cada carácter
        call    LCD_CMD
        movlw   0x01            ; borrar pantalla
        goto    LCD_CMD

LCD_CMD                         ; envía el comando que está en W
        bcf     PORTD,0         ; RS = 0
        goto    LCD_BYTE
LCD_NUM                         ; escribe el dígito 0..9 que está en W
        addlw   '0'
LCD_DATO                        ; escribe el carácter que está en W
        bsf     PORTD,0         ; RS = 1
LCD_BYTE
        movwf   LCD_DAT
        call    LCD_NIB         ; nibble alto
        swapf   LCD_DAT,W
        call    LCD_NIB         ; nibble bajo
        call    RETARDO_50US
        btfsc   PORTD,0         ; datos: listo
        return
        movlw   3               ; borrar (1) e inicio (2) tardan 1,52 ms
        subwf   LCD_DAT,W
        btfsc   STATUS,C
        return
        movlw   2
        goto    RETARDO_MS

LCD_NIB                         ; pone W<7:4> en D7..D4 y da un pulso en E
        andlw   0xF0
        movwf   LCD_TMP
        movf    PORTD,W
        andlw   0x0F
        iorwf   LCD_TMP,W
        movwf   PORTD
        bsf     PORTD,1         ; E = 1
        nop
        bcf     PORTD,1         ; E = 0: la LCD lee el dato en el flanco de bajada
        return

LCD_CADENA                      ; escribe la cadena (DT "...",0) que empieza en PTR_H:PTR_L
        call    LEER_TABLA
        iorlw   0
        btfsc   STATUS,Z
        return
        call    LCD_DATO
        incf    PTR_L,F
        btfsc   STATUS,Z
        incf    PTR_H,F
        goto    LCD_CADENA

LEER_TABLA                      ; W = byte de la tabla en PTR_H:PTR_L
        movf    PTR_H,W
        movwf   PCLATH
        movf    PTR_L,W
        movwf   PCL             ; salta a la RETLW, que vuelve a quien llamó a LEER_TABLA

;---------------------------------------------------------------------------
; Aritmética de 16 y 24 bits
;---------------------------------------------------------------------------
        CBLOCK  0x50
        NUM_H, NUM_L, K_H, K_L, TMP_L, CNT
        DIG4, DIG3, DIG2, DIG1, DIG0
        P2, P1, P0, X2, X1, X0
        ENDC
BCD16                           ; NUM_H:NUM_L (0..65535) -> DIG4..DIG0 (decenas de mil .. unidades)
        movlw   HIGH 10000
        movwf   K_H
        movlw   LOW 10000
        movwf   K_L
        call    CONTAR
        movwf   DIG4
        movlw   HIGH 1000
        movwf   K_H
        movlw   LOW 1000
        movwf   K_L
        call    CONTAR
        movwf   DIG3
        clrf    K_H
        movlw   100
        movwf   K_L
        call    CONTAR
        movwf   DIG2
        movlw   10
        movwf   K_L
        call    CONTAR
        movwf   DIG1
        movf    NUM_L,W
        movwf   DIG0
        return
CONTAR                          ; W = cuántas veces cabe K en NUM (y se lo resta)
        clrf    CNT
CT1     movf    K_L,W
        subwf   NUM_L,W         ; W = NUM_L - K_L   (C = 0 si hubo préstamo)
        movwf   TMP_L
        movf    K_H,W
        btfss   STATUS,C
        addlw   1               ; el préstamo se suma al byte alto que se resta
        subwf   NUM_H,W
        btfss   STATUS,C
        goto    CT2             ; ya no cabe
        movwf   NUM_H
        movf    TMP_L,W
        movwf   NUM_L
        incf    CNT,F
        goto    CT1
CT2     movf    CNT,W
        return

MUL16                           ; P2:P1:P0 = NUM_H:NUM_L x K_H:K_L  (resultado menor que 2^24)
        clrf    P0
        clrf    P1
        clrf    P2
        clrf    X2
        movf    NUM_H,W
        movwf   X1
        movf    NUM_L,W
        movwf   X0
        movlw   16
        movwf   CNT
ML1     bcf     STATUS,C
        rrf     K_H,F
        rrf     K_L,F           ; bit menos significativo de K en C
        btfss   STATUS,C
        goto    ML2
        movf    X0,W            ; P = P + X (24 bits)
        addwf   P0,F
        movf    X1,W
        btfsc   STATUS,C
        incfsz  X1,W
        addwf   P1,F
        movf    X2,W
        btfsc   STATUS,C
        incfsz  X2,W
        addwf   P2,F
ML2     bcf     STATUS,C        ; X = X * 2
        rlf     X0,F
        rlf     X1,F
        rlf     X2,F
        decfsz  CNT,F
        goto    ML1
        return

        END

In [ ]:
%%writefile ej04_yolo_serial.asm
;===========================================================================
; Ejemplo 4 · Integración con YOLO por puerto serie (USART 9600 baudios, 8N1)
; PIC16F877A · cristal 4 MHz
;   Un programa en Python con YOLO detecta el objeto y envía una letra:
;     'C' = carro -> LED verde (RB0)   'M' = moto -> LED rojo (RB1)
;     'P' = persona -> LED amarillo (RB2)   'N' = nada -> apaga los LED
;   El PIC muestra el objeto en la LCD y responde "OK:x" (x = letra recibida).
;   RC6/TX y RC7/RX al conversor USB-serie (o COMPIM / VIRTUAL TERMINAL en Proteus)
;   LCD: RS = RD0, E = RD1, D4..D7 = RD4..RD7
;===========================================================================
        LIST    P=16F877A, R=DEC
        #include <p16f877a.inc>
        __CONFIG _XT_OSC & _WDT_OFF & _PWRTE_ON & _BOREN_OFF & _LVP_OFF & _CP_OFF

        CBLOCK  0x20
        DATO, TOTAL
        ENDC

        ORG     0x000
        goto    INICIO
        ORG     0x005
INICIO
        BANKSEL TRISB
        clrf    TRISB
        movlw   25                      ; 9600 baudios con BRGH = 1: 4 MHz / (16 x 26) = 9615
        movwf   SPBRG
        movlw   0x24                    ; TXEN = 1, BRGH = 1, asíncrono
        movwf   TXSTA
        BANKSEL RCSTA
        movlw   0x90                    ; SPEN = 1, CREN = 1
        movwf   RCSTA
        clrf    PORTB
        clrf    TOTAL
        call    LCD_INIT
        movlw   0x80
        call    LCD_CMD
        movlw   HIGH TXT_TIT
        movwf   PTR_H
        movlw   LOW TXT_TIT
        movwf   PTR_L
        call    LCD_CADENA
        movlw   HIGH TXT_ESP
        call    LINEA2
        movlw   HIGH TXT_LISTO
        movwf   PTR_H
        movlw   LOW TXT_LISTO
        movwf   PTR_L
        call    TX_CADENA

LAZO    btfsc   RCSTA,OERR              ; si hubo desbordamiento, reinicia el receptor
        call    LIMPIAR_OERR
        btfss   PIR1,RCIF               ; ¿llegó un carácter?
        goto    LAZO
        movf    RCREG,W
        andlw   0xDF                    ; minúscula -> mayúscula
        movwf   DATO
        xorlw   'C'
        btfsc   STATUS,Z
        goto    CARRO
        movf    DATO,W
        xorlw   'M'
        btfsc   STATUS,Z
        goto    MOTO
        movf    DATO,W
        xorlw   'P'
        btfsc   STATUS,Z
        goto    PERSONA
        movf    DATO,W
        xorlw   'N'
        btfsc   STATUS,Z
        goto    NADA
        goto    LAZO                    ; cualquier otro carácter (\r, \n...) se ignora

CARRO   movlw   b'00000001'             ; verde
        movwf   PORTB
        movlw   HIGH TXT_CARRO
        movwf   PTR_H
        movlw   LOW TXT_CARRO
        goto    RESPONDER
MOTO    movlw   b'00000010'             ; rojo
        movwf   PORTB
        movlw   HIGH TXT_MOTO
        movwf   PTR_H
        movlw   LOW TXT_MOTO
        goto    RESPONDER
PERSONA movlw   b'00000100'             ; amarillo
        movwf   PORTB
        movlw   HIGH TXT_PERS
        movwf   PTR_H
        movlw   LOW TXT_PERS
        goto    RESPONDER
NADA    clrf    PORTB
        movlw   HIGH TXT_NADA
        movwf   PTR_H
        movlw   LOW TXT_NADA

RESPONDER
        movwf   PTR_L
        movlw   0xC0
        call    LCD_CMD
        call    LCD_CADENA
        incf    TOTAL,F
        movlw   'O'
        call    TX_BYTE
        movlw   'K'
        call    TX_BYTE
        movlw   ':'
        call    TX_BYTE
        movf    DATO,W
        call    TX_BYTE
        movlw   13
        call    TX_BYTE
        movlw   10
        call    TX_BYTE
        goto    LAZO

LINEA2  movwf   PTR_H                   ; escribe la cadena TXT_ESP en la línea 2
        movlw   LOW TXT_ESP
        movwf   PTR_L
        movlw   0xC0
        call    LCD_CMD
        goto    LCD_CADENA

LIMPIAR_OERR
        bcf     RCSTA,CREN
        bsf     RCSTA,CREN
        return

TX_BYTE                                 ; transmite W cuando el buffer está libre
        btfss   PIR1,TXIF
        goto    TX_BYTE
        movwf   TXREG
        return

TX_CADENA                               ; transmite la cadena de PTR_H:PTR_L
        call    LEER_TABLA
        iorlw   0
        btfsc   STATUS,Z
        return
        call    TX_BYTE
        incf    PTR_L,F
        btfsc   STATUS,Z
        incf    PTR_H,F
        goto    TX_CADENA

TXT_TIT   DT    "YOLO + PIC16F877",0
TXT_ESP   DT    "Esperando...    ",0
TXT_CARRO DT    "Objeto: CARRO   ",0
TXT_MOTO  DT    "Objeto: MOTO    ",0
TXT_PERS  DT    "Objeto: PERSONA ",0
TXT_NADA  DT    "Objeto: ninguno ",0
TXT_LISTO DT    "PIC listo: envie C, M, P o N",13,10,0

;---------------------------------------------------------------------------
; Retardos para cristal de 4 MHz (1 ciclo de instrucción = 1 µs)
;---------------------------------------------------------------------------
        CBLOCK  0x68
        R_MS, R_US
        ENDC
RETARDO_MS                      ; espera W milisegundos (W = 1..255)
        movwf   R_MS
RMS1    movlw   249             ; 2 + 248*4 + 3 + 3 = 1000 ciclos por milisegundo
        movwf   R_US
RMS2    nop
        decfsz  R_US,F
        goto    RMS2
        decfsz  R_MS,F
        goto    RMS1
        return
RETARDO_50US                    ; 50 µs contando el call y el return
        movlw   15
        movwf   R_US
R50     decfsz  R_US,F
        goto    R50
        return

;---------------------------------------------------------------------------
; LCD 16x2 (HD44780 / LM016L) en modo de 4 bits sobre PORTD
;   RS = RD0, E = RD1, D4..D7 = RD4..RD7, RW a tierra
;---------------------------------------------------------------------------
        CBLOCK  0x6A
        LCD_TMP, LCD_DAT, PTR_H, PTR_L
        ENDC
LCD_INIT
        BANKSEL TRISD
        clrf    TRISD
        BANKSEL PORTD
        clrf    PORTD
        movlw   20
        call    RETARDO_MS      ; más de 15 ms después del encendido
        movlw   0x30
        call    LCD_NIB
        movlw   5
        call    RETARDO_MS
        movlw   0x30
        call    LCD_NIB
        call    RETARDO_50US
        call    RETARDO_50US
        movlw   0x30
        call    LCD_NIB
        call    RETARDO_50US
        movlw   0x20            ; pasa a modo de 4 bits
        call    LCD_NIB
        call    RETARDO_50US
        movlw   0x28            ; 4 bits, 2 líneas, caracteres de 5x8
        call    LCD_CMD
        movlw   0x0C            ; display encendido, sin cursor
        call    LCD_CMD
        movlw   0x06            ; la dirección aumenta después de cada carácter
        call    LCD_CMD
        movlw   0x01            ; borrar pantalla
        goto    LCD_CMD

LCD_CMD                         ; envía el comando que está en W
        bcf     PORTD,0         ; RS = 0
        goto    LCD_BYTE
LCD_NUM                         ; escribe el dígito 0..9 que está en W
        addlw   '0'
LCD_DATO                        ; escribe el carácter que está en W
        bsf     PORTD,0         ; RS = 1
LCD_BYTE
        movwf   LCD_DAT
        call    LCD_NIB         ; nibble alto
        swapf   LCD_DAT,W
        call    LCD_NIB         ; nibble bajo
        call    RETARDO_50US
        btfsc   PORTD,0         ; datos: listo
        return
        movlw   3               ; borrar (1) e inicio (2) tardan 1,52 ms
        subwf   LCD_DAT,W
        btfsc   STATUS,C
        return
        movlw   2
        goto    RETARDO_MS

LCD_NIB                         ; pone W<7:4> en D7..D4 y da un pulso en E
        andlw   0xF0
        movwf   LCD_TMP
        movf    PORTD,W
        andlw   0x0F
        iorwf   LCD_TMP,W
        movwf   PORTD
        bsf     PORTD,1         ; E = 1
        nop
        bcf     PORTD,1         ; E = 0: la LCD lee el dato en el flanco de bajada
        return

LCD_CADENA                      ; escribe la cadena (DT "...",0) que empieza en PTR_H:PTR_L
        call    LEER_TABLA
        iorlw   0
        btfsc   STATUS,Z
        return
        call    LCD_DATO
        incf    PTR_L,F
        btfsc   STATUS,Z
        incf    PTR_H,F
        goto    LCD_CADENA

LEER_TABLA                      ; W = byte de la tabla en PTR_H:PTR_L
        movf    PTR_H,W
        movwf   PCLATH
        movf    PTR_L,W
        movwf   PCL             ; salta a la RETLW, que vuelve a quien llamó a LEER_TABLA

        END

In [ ]:
%%writefile ej05_teclado_clave.asm
;===========================================================================
; Ejemplo 5 · Cerradura electrónica con teclado matricial 4x4 y LCD 16x2
; PIC16F877A · cristal 4 MHz
;   Teclado (KEYPAD-SMALLCALC): filas A..D = RB0..RB3 (salidas), columnas 1..4 = RB4..RB7 (entradas, pull-up interno)
;       7 8 9 /
;       4 5 6 x
;       1 2 3 -
;       C 0 = +      C = borrar, = = confirmar.   Clave: 1234
;   RC0 = LED verde, RC1 = LED rojo, RC2 = relé de la cerradura
;   LCD: RS = RD0, E = RD1, D4..D7 = RD4..RD7
;===========================================================================
        LIST    P=16F877A, R=DEC
        #include <p16f877a.inc>
        __CONFIG _XT_OSC & _WDT_OFF & _PWRTE_ON & _BOREN_OFF & _LVP_OFF & _CP_OFF

        CBLOCK  0x20
        TECLA, FILA, COL, MASCARA, AUX, N, INTENTOS, VECES
        CLAVE:4                         ; dígitos escritos
        ENDC

        ORG     0x000
        goto    INICIO
        ORG     0x005
INICIO
        BANKSEL OPTION_REG
        bcf     OPTION_REG,NOT_RBPU     ; pull-ups en las columnas
        movlw   0xF0                    ; RB0-RB3 salidas (filas), RB4-RB7 entradas (columnas)
        movwf   TRISB
        movlw   0xF8                    ; RC0-RC2 salidas
        movwf   TRISC
        BANKSEL PORTC
        clrf    PORTC
        clrf    INTENTOS
        call    LCD_INIT

NUEVA   clrf    PORTC
        clrf    N
        movlw   0x01
        call    LCD_CMD
        movlw   HIGH TXT_CLAVE
        movwf   PTR_H
        movlw   LOW TXT_CLAVE
        movwf   PTR_L
        call    LCD_CADENA
        movlw   0xC0
        call    LCD_CMD
        movlw   '>'
        call    LCD_DATO
        movlw   ' '
        call    LCD_DATO

LAZO    call    ESCANEAR                ; W = carácter de la tecla o 0
        iorlw   0
        btfsc   STATUS,Z
        goto    LAZO
        movwf   TECLA
        movlw   20
        call    RETARDO_MS
SOLTAR  call    ESCANEAR                ; espera a que se suelte la tecla
        iorlw   0
        btfss   STATUS,Z
        goto    SOLTAR
        movlw   20
        call    RETARDO_MS

        movf    TECLA,W
        xorlw   'C'
        btfsc   STATUS,Z
        goto    NUEVA                   ; borrar
        movf    TECLA,W
        xorlw   '='
        btfsc   STATUS,Z
        goto    VERIFICAR
        movlw   '0'                     ; ¿es un dígito '0'..'9'?
        subwf   TECLA,W
        btfss   STATUS,C
        goto    LAZO
        sublw   9
        btfss   STATUS,C
        goto    LAZO
        movlw   4                       ; máximo 4 dígitos
        subwf   N,W
        btfsc   STATUS,C
        goto    LAZO
        movlw   CLAVE                   ; CLAVE[N] = tecla (direccionamiento indirecto)
        addwf   N,W
        movwf   FSR
        movf    TECLA,W
        movwf   INDF
        incf    N,F
        movlw   '*'
        call    LCD_DATO
        goto    LAZO

VERIFICAR
        movlw   4
        xorwf   N,W
        btfss   STATUS,Z
        goto    MAL
        movf    CLAVE+0,W
        xorlw   '1'
        btfss   STATUS,Z
        goto    MAL
        movf    CLAVE+1,W
        xorlw   '2'
        btfss   STATUS,Z
        goto    MAL
        movf    CLAVE+2,W
        xorlw   '3'
        btfss   STATUS,Z
        goto    MAL
        movf    CLAVE+3,W
        xorlw   '4'
        btfss   STATUS,Z
        goto    MAL
BIEN    clrf    INTENTOS
        movlw   b'00000101'             ; LED verde y relé
        movwf   PORTC
        movlw   HIGH TXT_OK
        movwf   PTR_H
        movlw   LOW TXT_OK
        call    MENSAJE
        movlw   12                      ; 3 s abierta
        call    ESPERAR
        goto    NUEVA
MAL     movlw   b'00000010'             ; LED rojo
        movwf   PORTC
        incf    INTENTOS,F
        movlw   3
        subwf   INTENTOS,W
        btfsc   STATUS,C
        goto    BLOQUEO
        movlw   HIGH TXT_MAL
        movwf   PTR_H
        movlw   LOW TXT_MAL
        call    MENSAJE
        movlw   8                       ; 2 s
        call    ESPERAR
        goto    NUEVA
BLOQUEO clrf    INTENTOS
        movlw   HIGH TXT_BLOQ
        movwf   PTR_H
        movlw   LOW TXT_BLOQ
        call    MENSAJE
        movlw   20                      ; 10 s con el LED rojo parpadeando
        movwf   VECES
PARPADEO
        movlw   b'00000010'
        xorwf   PORTC,F
        movlw   2
        call    ESPERAR
        decfsz  VECES,F
        goto    PARPADEO
        goto    NUEVA

MENSAJE                                 ; escribe en la línea 2 la cadena PTR_H:W
        movwf   PTR_L
        movlw   0xC0
        call    LCD_CMD
        goto    LCD_CADENA

ESPERAR                                 ; espera W x 250 ms
        movwf   AUX
ESP1    movlw   250
        call    RETARDO_MS
        decfsz  AUX,F
        goto    ESP1
        return

ESCANEAR                                ; devuelve en W el carácter de la tecla presionada (0 = ninguna)
        clrf    FILA
        movlw   b'11111110'
        movwf   MASCARA
ESC1    movf    MASCARA,W               ; pone en 0 solo la fila que se revisa
        movwf   PORTB
        nop
        nop
        swapf   PORTB,W                 ; columnas en el nibble bajo
        andlw   0x0F
        xorlw   0x0F                    ; columna presionada = bit en 1
        btfss   STATUS,Z
        goto    HAY
        incf    FILA,F
        bsf     STATUS,C
        rlf     MASCARA,F
        movlw   4
        xorwf   FILA,W
        btfss   STATUS,Z
        goto    ESC1
        retlw   0
HAY     movwf   AUX
        clrf    COL
HAY1    rrf     AUX,F                   ; busca el número de la columna
        btfsc   STATUS,C
        goto    HAY2
        incf    COL,F
        goto    HAY1
HAY2    bcf     STATUS,C                ; índice = FILA x 4 + COL
        rlf     FILA,F
        rlf     FILA,W
        addwf   COL,W
        addlw   LOW TECLAS
        movwf   PTR_L
        movlw   HIGH TECLAS
        btfsc   STATUS,C
        addlw   1
        movwf   PTR_H
        goto    LEER_TABLA

TECLAS    DT    "789/456*123-C0=+"
TXT_CLAVE DT    "Ingrese la clave",0
TXT_OK    DT    "ACCESO CONCEDIDO",0
TXT_MAL   DT    "CLAVE INCORRECTA",0
TXT_BLOQ  DT    "BLOQUEADO  10 s ",0

;---------------------------------------------------------------------------
; Retardos para cristal de 4 MHz (1 ciclo de instrucción = 1 µs)
;---------------------------------------------------------------------------
        CBLOCK  0x68
        R_MS, R_US
        ENDC
RETARDO_MS                      ; espera W milisegundos (W = 1..255)
        movwf   R_MS
RMS1    movlw   249             ; 2 + 248*4 + 3 + 3 = 1000 ciclos por milisegundo
        movwf   R_US
RMS2    nop
        decfsz  R_US,F
        goto    RMS2
        decfsz  R_MS,F
        goto    RMS1
        return
RETARDO_50US                    ; 50 µs contando el call y el return
        movlw   15
        movwf   R_US
R50     decfsz  R_US,F
        goto    R50
        return

;---------------------------------------------------------------------------
; LCD 16x2 (HD44780 / LM016L) en modo de 4 bits sobre PORTD
;   RS = RD0, E = RD1, D4..D7 = RD4..RD7, RW a tierra
;---------------------------------------------------------------------------
        CBLOCK  0x6A
        LCD_TMP, LCD_DAT, PTR_H, PTR_L
        ENDC
LCD_INIT
        BANKSEL TRISD
        clrf    TRISD
        BANKSEL PORTD
        clrf    PORTD
        movlw   20
        call    RETARDO_MS      ; más de 15 ms después del encendido
        movlw   0x30
        call    LCD_NIB
        movlw   5
        call    RETARDO_MS
        movlw   0x30
        call    LCD_NIB
        call    RETARDO_50US
        call    RETARDO_50US
        movlw   0x30
        call    LCD_NIB
        call    RETARDO_50US
        movlw   0x20            ; pasa a modo de 4 bits
        call    LCD_NIB
        call    RETARDO_50US
        movlw   0x28            ; 4 bits, 2 líneas, caracteres de 5x8
        call    LCD_CMD
        movlw   0x0C            ; display encendido, sin cursor
        call    LCD_CMD
        movlw   0x06            ; la dirección aumenta después de cada carácter
        call    LCD_CMD
        movlw   0x01            ; borrar pantalla
        goto    LCD_CMD

LCD_CMD                         ; envía el comando que está en W
        bcf     PORTD,0         ; RS = 0
        goto    LCD_BYTE
LCD_NUM                         ; escribe el dígito 0..9 que está en W
        addlw   '0'
LCD_DATO                        ; escribe el carácter que está en W
        bsf     PORTD,0         ; RS = 1
LCD_BYTE
        movwf   LCD_DAT
        call    LCD_NIB         ; nibble alto
        swapf   LCD_DAT,W
        call    LCD_NIB         ; nibble bajo
        call    RETARDO_50US
        btfsc   PORTD,0         ; datos: listo
        return
        movlw   3               ; borrar (1) e inicio (2) tardan 1,52 ms
        subwf   LCD_DAT,W
        btfsc   STATUS,C
        return
        movlw   2
        goto    RETARDO_MS

LCD_NIB                         ; pone W<7:4> en D7..D4 y da un pulso en E
        andlw   0xF0
        movwf   LCD_TMP
        movf    PORTD,W
        andlw   0x0F
        iorwf   LCD_TMP,W
        movwf   PORTD
        bsf     PORTD,1         ; E = 1
        nop
        bcf     PORTD,1         ; E = 0: la LCD lee el dato en el flanco de bajada
        return

LCD_CADENA                      ; escribe la cadena (DT "...",0) que empieza en PTR_H:PTR_L
        call    LEER_TABLA
        iorlw   0
        btfsc   STATUS,Z
        return
        call    LCD_DATO
        incf    PTR_L,F
        btfsc   STATUS,Z
        incf    PTR_H,F
        goto    LCD_CADENA

LEER_TABLA                      ; W = byte de la tabla en PTR_H:PTR_L
        movf    PTR_H,W
        movwf   PCLATH
        movf    PTR_L,W
        movwf   PCL             ; salta a la RETLW, que vuelve a quien llamó a LEER_TABLA

        END

In [ ]:
%%writefile ej06_servo.asm
;===========================================================================
; Ejemplo 6 · Servomotor controlado con un potenciómetro
; PIC16F877A · cristal 4 MHz
;   Potenciómetro de 10 kΩ en AN0 (RA0). Señal del servo en RC2: pulso de 1,0 a 2,0 ms cada 20 ms
;   (MOTOR-PWMSERVO en Proteus: 1 ms = -90°, 1,5 ms = 0°, 2 ms = +90°)
;   LCD: RS = RD0, E = RD1, D4..D7 = RD4..RD7
;===========================================================================
        LIST    P=16F877A, R=DEC
        #include <p16f877a.inc>
        __CONFIG _XT_OSC & _WDT_OFF & _PWRTE_ON & _BOREN_OFF & _LVP_OFF & _CP_OFF

        CBLOCK  0x20
        POS, CUENTA
        ENDC

        ORG     0x000
        goto    INICIO
        ORG     0x005
INICIO
        BANKSEL TRISC
        bcf     TRISC,2                 ; RC2 salida hacia el servo
        movlw   0x0E                    ; solo AN0 analógico, justificado a la izquierda (8 bits en ADRESH)
        movwf   ADCON1
        BANKSEL PORTC
        bcf     PORTC,2
        movlw   0x41                    ; Fosc/8, canal 0, ADC encendido
        movwf   ADCON0
        movlw   128
        movwf   POS
        call    LCD_INIT
        movlw   0x80
        call    LCD_CMD
        movlw   HIGH TXT_ANG
        movwf   PTR_H
        movlw   LOW TXT_ANG
        movwf   PTR_L
        call    LCD_CADENA
        movlw   0xC0
        call    LCD_CMD
        movlw   HIGH TXT_PUL
        movwf   PTR_H
        movlw   LOW TXT_PUL
        movwf   PTR_L
        call    LCD_CADENA

LAZO    ;----- pulso de 1000 + 4 x POS microsegundos
        bsf     PORTC,2
        movlw   1
        call    RETARDO_MS              ; 1000 µs
        movf    POS,W
        btfsc   STATUS,Z
        goto    FIN_PULSO
        movwf   CUENTA
PUL1    nop                             ; 4 µs por vuelta
        decfsz  CUENTA,F
        goto    PUL1
FIN_PULSO
        bcf     PORTC,2
        ;----- nueva lectura del potenciómetro (el canal ya está adquirido)
        bsf     ADCON0,GO
ESPERA  btfsc   ADCON0,GO
        goto    ESPERA
        movf    ADRESH,W
        movwf   POS
        ;----- ángulo = POS x 180 / 256 (0..179°)
        clrf    NUM_H
        movwf   NUM_L
        clrf    K_H
        movlw   180
        movwf   K_L
        call    MUL16
        clrf    NUM_H
        movf    P1,W
        movwf   NUM_L
        call    BCD16
        movlw   0x88
        call    LCD_CMD
        movf    DIG2,W
        call    LCD_NUM
        movf    DIG1,W
        call    LCD_NUM
        movf    DIG0,W
        call    LCD_NUM
        movlw   0xDF
        call    LCD_DATO
        ;----- ancho del pulso = 1000 + 4 x POS
        clrf    NUM_H
        movf    POS,W
        movwf   NUM_L
        bcf     STATUS,C
        rlf     NUM_L,F
        rlf     NUM_H,F
        rlf     NUM_L,F
        rlf     NUM_H,F
        movlw   LOW 1000
        addwf   NUM_L,F
        btfsc   STATUS,C
        incf    NUM_H,F
        movlw   HIGH 1000
        addwf   NUM_H,F
        call    BCD16
        movlw   0xC7
        call    LCD_CMD
        movf    DIG3,W
        call    LCD_NUM
        movf    DIG2,W
        call    LCD_NUM
        movf    DIG1,W
        call    LCD_NUM
        movf    DIG0,W
        call    LCD_NUM
        ;----- completa el periodo de unos 20 ms
        movlw   17
        call    RETARDO_MS
        goto    LAZO

TXT_ANG DT      "Angulo:",0
TXT_PUL DT      "Pulso:      us",0

;---------------------------------------------------------------------------
; Retardos para cristal de 4 MHz (1 ciclo de instrucción = 1 µs)
;---------------------------------------------------------------------------
        CBLOCK  0x68
        R_MS, R_US
        ENDC
RETARDO_MS                      ; espera W milisegundos (W = 1..255)
        movwf   R_MS
RMS1    movlw   249             ; 2 + 248*4 + 3 + 3 = 1000 ciclos por milisegundo
        movwf   R_US
RMS2    nop
        decfsz  R_US,F
        goto    RMS2
        decfsz  R_MS,F
        goto    RMS1
        return
RETARDO_50US                    ; 50 µs contando el call y el return
        movlw   15
        movwf   R_US
R50     decfsz  R_US,F
        goto    R50
        return

;---------------------------------------------------------------------------
; LCD 16x2 (HD44780 / LM016L) en modo de 4 bits sobre PORTD
;   RS = RD0, E = RD1, D4..D7 = RD4..RD7, RW a tierra
;---------------------------------------------------------------------------
        CBLOCK  0x6A
        LCD_TMP, LCD_DAT, PTR_H, PTR_L
        ENDC
LCD_INIT
        BANKSEL TRISD
        clrf    TRISD
        BANKSEL PORTD
        clrf    PORTD
        movlw   20
        call    RETARDO_MS      ; más de 15 ms después del encendido
        movlw   0x30
        call    LCD_NIB
        movlw   5
        call    RETARDO_MS
        movlw   0x30
        call    LCD_NIB
        call    RETARDO_50US
        call    RETARDO_50US
        movlw   0x30
        call    LCD_NIB
        call    RETARDO_50US
        movlw   0x20            ; pasa a modo de 4 bits
        call    LCD_NIB
        call    RETARDO_50US
        movlw   0x28            ; 4 bits, 2 líneas, caracteres de 5x8
        call    LCD_CMD
        movlw   0x0C            ; display encendido, sin cursor
        call    LCD_CMD
        movlw   0x06            ; la dirección aumenta después de cada carácter
        call    LCD_CMD
        movlw   0x01            ; borrar pantalla
        goto    LCD_CMD

LCD_CMD                         ; envía el comando que está en W
        bcf     PORTD,0         ; RS = 0
        goto    LCD_BYTE
LCD_NUM                         ; escribe el dígito 0..9 que está en W
        addlw   '0'
LCD_DATO                        ; escribe el carácter que está en W
        bsf     PORTD,0         ; RS = 1
LCD_BYTE
        movwf   LCD_DAT
        call    LCD_NIB         ; nibble alto
        swapf   LCD_DAT,W
        call    LCD_NIB         ; nibble bajo
        call    RETARDO_50US
        btfsc   PORTD,0         ; datos: listo
        return
        movlw   3               ; borrar (1) e inicio (2) tardan 1,52 ms
        subwf   LCD_DAT,W
        btfsc   STATUS,C
        return
        movlw   2
        goto    RETARDO_MS

LCD_NIB                         ; pone W<7:4> en D7..D4 y da un pulso en E
        andlw   0xF0
        movwf   LCD_TMP
        movf    PORTD,W
        andlw   0x0F
        iorwf   LCD_TMP,W
        movwf   PORTD
        bsf     PORTD,1         ; E = 1
        nop
        bcf     PORTD,1         ; E = 0: la LCD lee el dato en el flanco de bajada
        return

LCD_CADENA                      ; escribe la cadena (DT "...",0) que empieza en PTR_H:PTR_L
        call    LEER_TABLA
        iorlw   0
        btfsc   STATUS,Z
        return
        call    LCD_DATO
        incf    PTR_L,F
        btfsc   STATUS,Z
        incf    PTR_H,F
        goto    LCD_CADENA

LEER_TABLA                      ; W = byte de la tabla en PTR_H:PTR_L
        movf    PTR_H,W
        movwf   PCLATH
        movf    PTR_L,W
        movwf   PCL             ; salta a la RETLW, que vuelve a quien llamó a LEER_TABLA

;---------------------------------------------------------------------------
; Aritmética de 16 y 24 bits
;---------------------------------------------------------------------------
        CBLOCK  0x50
        NUM_H, NUM_L, K_H, K_L, TMP_L, CNT
        DIG4, DIG3, DIG2, DIG1, DIG0
        P2, P1, P0, X2, X1, X0
        ENDC
BCD16                           ; NUM_H:NUM_L (0..65535) -> DIG4..DIG0 (decenas de mil .. unidades)
        movlw   HIGH 10000
        movwf   K_H
        movlw   LOW 10000
        movwf   K_L
        call    CONTAR
        movwf   DIG4
        movlw   HIGH 1000
        movwf   K_H
        movlw   LOW 1000
        movwf   K_L
        call    CONTAR
        movwf   DIG3
        clrf    K_H
        movlw   100
        movwf   K_L
        call    CONTAR
        movwf   DIG2
        movlw   10
        movwf   K_L
        call    CONTAR
        movwf   DIG1
        movf    NUM_L,W
        movwf   DIG0
        return
CONTAR                          ; W = cuántas veces cabe K en NUM (y se lo resta)
        clrf    CNT
CT1     movf    K_L,W
        subwf   NUM_L,W         ; W = NUM_L - K_L   (C = 0 si hubo préstamo)
        movwf   TMP_L
        movf    K_H,W
        btfss   STATUS,C
        addlw   1               ; el préstamo se suma al byte alto que se resta
        subwf   NUM_H,W
        btfss   STATUS,C
        goto    CT2             ; ya no cabe
        movwf   NUM_H
        movf    TMP_L,W
        movwf   NUM_L
        incf    CNT,F
        goto    CT1
CT2     movf    CNT,W
        return

MUL16                           ; P2:P1:P0 = NUM_H:NUM_L x K_H:K_L  (resultado menor que 2^24)
        clrf    P0
        clrf    P1
        clrf    P2
        clrf    X2
        movf    NUM_H,W
        movwf   X1
        movf    NUM_L,W
        movwf   X0
        movlw   16
        movwf   CNT
ML1     bcf     STATUS,C
        rrf     K_H,F
        rrf     K_L,F           ; bit menos significativo de K en C
        btfss   STATUS,C
        goto    ML2
        movf    X0,W            ; P = P + X (24 bits)
        addwf   P0,F
        movf    X1,W
        btfsc   STATUS,C
        incfsz  X1,W
        addwf   P1,F
        movf    X2,W
        btfsc   STATUS,C
        incfsz  X2,W
        addwf   P2,F
ML2     bcf     STATUS,C        ; X = X * 2
        rlf     X0,F
        rlf     X1,F
        rlf     X2,F
        decfsz  CNT,F
        goto    ML1
        return

        END

In [ ]:
%%writefile ej07_motor_dc.asm
;===========================================================================
; Ejemplo 7 · Motor DC con puente H L293D: velocidad por PWM y sentido de giro
; PIC16F877A · cristal 4 MHz
;   Potenciómetro en AN0 -> ciclo útil del PWM en RC2/CCP1 (3,9 kHz) -> EN1 del L293D
;   RC0 -> IN1, RC1 -> IN2.   Pulsadores: RB0 = adelante, RB1 = atrás, RB2 = parar
;   LCD: RS = RD0, E = RD1, D4..D7 = RD4..RD7
;===========================================================================
        LIST    P=16F877A, R=DEC
        #include <p16f877a.inc>
        __CONFIG _XT_OSC & _WDT_OFF & _PWRTE_ON & _BOREN_OFF & _LVP_OFF & _CP_OFF

        CBLOCK  0x20
        VEL, TECLAS
        ENDC

        ORG     0x000
        goto    INICIO
        ORG     0x005
INICIO
        BANKSEL OPTION_REG
        bcf     OPTION_REG,NOT_RBPU
        movlw   0xF8                    ; RC0, RC1 y RC2 salidas
        movwf   TRISC
        movlw   0x0E                    ; AN0 analógico, justificado a la izquierda
        movwf   ADCON1
        movlw   255                     ; periodo del PWM: (255+1) x 4 x 0,25 µs = 256 µs -> 3,9 kHz
        movwf   PR2
        BANKSEL PORTC
        clrf    PORTC
        clrf    CCPR1L
        movlw   0x0C                    ; CCP1 en modo PWM
        movwf   CCP1CON
        movlw   0x04                    ; TMR2 encendido, preescalador 1
        movwf   T2CON
        movlw   0x41
        movwf   ADCON0
        call    LCD_INIT
        movlw   0x80
        call    LCD_CMD
        movlw   HIGH TXT_TIT
        movwf   PTR_H
        movlw   LOW TXT_TIT
        movwf   PTR_L
        call    LCD_CADENA
        call    PARAR

LAZO    call    RETARDO_50US
        bsf     ADCON0,GO
ESPERA  btfsc   ADCON0,GO
        goto    ESPERA
        movf    ADRESH,W
        movwf   VEL
        movwf   CCPR1L                  ; 8 bits altos del ciclo útil
        comf    PORTB,W                 ; pulsadores (1 = presionado)
        andlw   0x07
        movwf   TECLAS
        btfsc   TECLAS,0
        call    ADELANTE
        btfsc   TECLAS,1
        call    ATRAS
        btfsc   TECLAS,2
        call    PARAR
        ;----- velocidad en % = VEL x 101 / 256
        clrf    NUM_H
        movf    VEL,W
        movwf   NUM_L
        clrf    K_H
        movlw   101
        movwf   K_L
        call    MUL16
        clrf    NUM_H
        movf    P1,W
        movwf   NUM_L
        call    BCD16
        movlw   0xCB
        call    LCD_CMD
        movf    DIG2,W
        btfsc   STATUS,Z
        movlw   ' ' - '0'
        call    LCD_NUM
        movf    DIG1,W
        call    LCD_NUM
        movf    DIG0,W
        call    LCD_NUM
        movlw   '%'
        call    LCD_DATO
        movlw   100
        call    RETARDO_MS
        goto    LAZO

ADELANTE
        bsf     PORTC,0                 ; IN1 = 1, IN2 = 0
        bcf     PORTC,1
        movlw   HIGH TXT_ADE
        movwf   PTR_H
        movlw   LOW TXT_ADE
        goto    SENTIDO
ATRAS   bcf     PORTC,0                 ; IN1 = 0, IN2 = 1
        bsf     PORTC,1
        movlw   HIGH TXT_ATR
        movwf   PTR_H
        movlw   LOW TXT_ATR
        goto    SENTIDO
PARAR   bcf     PORTC,0                 ; IN1 = IN2 = 0: freno
        bcf     PORTC,1
        movlw   HIGH TXT_PAR
        movwf   PTR_H
        movlw   LOW TXT_PAR
SENTIDO movwf   PTR_L
        movlw   0xC0
        call    LCD_CMD
        goto    LCD_CADENA

TXT_TIT DT      "Motor DC  L293D",0
TXT_ADE DT      "ADELANTE",0
TXT_ATR DT      "ATRAS   ",0
TXT_PAR DT      "PARADO  ",0

;---------------------------------------------------------------------------
; Retardos para cristal de 4 MHz (1 ciclo de instrucción = 1 µs)
;---------------------------------------------------------------------------
        CBLOCK  0x68
        R_MS, R_US
        ENDC
RETARDO_MS                      ; espera W milisegundos (W = 1..255)
        movwf   R_MS
RMS1    movlw   249             ; 2 + 248*4 + 3 + 3 = 1000 ciclos por milisegundo
        movwf   R_US
RMS2    nop
        decfsz  R_US,F
        goto    RMS2
        decfsz  R_MS,F
        goto    RMS1
        return
RETARDO_50US                    ; 50 µs contando el call y el return
        movlw   15
        movwf   R_US
R50     decfsz  R_US,F
        goto    R50
        return

;---------------------------------------------------------------------------
; LCD 16x2 (HD44780 / LM016L) en modo de 4 bits sobre PORTD
;   RS = RD0, E = RD1, D4..D7 = RD4..RD7, RW a tierra
;---------------------------------------------------------------------------
        CBLOCK  0x6A
        LCD_TMP, LCD_DAT, PTR_H, PTR_L
        ENDC
LCD_INIT
        BANKSEL TRISD
        clrf    TRISD
        BANKSEL PORTD
        clrf    PORTD
        movlw   20
        call    RETARDO_MS      ; más de 15 ms después del encendido
        movlw   0x30
        call    LCD_NIB
        movlw   5
        call    RETARDO_MS
        movlw   0x30
        call    LCD_NIB
        call    RETARDO_50US
        call    RETARDO_50US
        movlw   0x30
        call    LCD_NIB
        call    RETARDO_50US
        movlw   0x20            ; pasa a modo de 4 bits
        call    LCD_NIB
        call    RETARDO_50US
        movlw   0x28            ; 4 bits, 2 líneas, caracteres de 5x8
        call    LCD_CMD
        movlw   0x0C            ; display encendido, sin cursor
        call    LCD_CMD
        movlw   0x06            ; la dirección aumenta después de cada carácter
        call    LCD_CMD
        movlw   0x01            ; borrar pantalla
        goto    LCD_CMD

LCD_CMD                         ; envía el comando que está en W
        bcf     PORTD,0         ; RS = 0
        goto    LCD_BYTE
LCD_NUM                         ; escribe el dígito 0..9 que está en W
        addlw   '0'
LCD_DATO                        ; escribe el carácter que está en W
        bsf     PORTD,0         ; RS = 1
LCD_BYTE
        movwf   LCD_DAT
        call    LCD_NIB         ; nibble alto
        swapf   LCD_DAT,W
        call    LCD_NIB         ; nibble bajo
        call    RETARDO_50US
        btfsc   PORTD,0         ; datos: listo
        return
        movlw   3               ; borrar (1) e inicio (2) tardan 1,52 ms
        subwf   LCD_DAT,W
        btfsc   STATUS,C
        return
        movlw   2
        goto    RETARDO_MS

LCD_NIB                         ; pone W<7:4> en D7..D4 y da un pulso en E
        andlw   0xF0
        movwf   LCD_TMP
        movf    PORTD,W
        andlw   0x0F
        iorwf   LCD_TMP,W
        movwf   PORTD
        bsf     PORTD,1         ; E = 1
        nop
        bcf     PORTD,1         ; E = 0: la LCD lee el dato en el flanco de bajada
        return

LCD_CADENA                      ; escribe la cadena (DT "...",0) que empieza en PTR_H:PTR_L
        call    LEER_TABLA
        iorlw   0
        btfsc   STATUS,Z
        return
        call    LCD_DATO
        incf    PTR_L,F
        btfsc   STATUS,Z
        incf    PTR_H,F
        goto    LCD_CADENA

LEER_TABLA                      ; W = byte de la tabla en PTR_H:PTR_L
        movf    PTR_H,W
        movwf   PCLATH
        movf    PTR_L,W
        movwf   PCL             ; salta a la RETLW, que vuelve a quien llamó a LEER_TABLA

;---------------------------------------------------------------------------
; Aritmética de 16 y 24 bits
;---------------------------------------------------------------------------
        CBLOCK  0x50
        NUM_H, NUM_L, K_H, K_L, TMP_L, CNT
        DIG4, DIG3, DIG2, DIG1, DIG0
        P2, P1, P0, X2, X1, X0
        ENDC
BCD16                           ; NUM_H:NUM_L (0..65535) -> DIG4..DIG0 (decenas de mil .. unidades)
        movlw   HIGH 10000
        movwf   K_H
        movlw   LOW 10000
        movwf   K_L
        call    CONTAR
        movwf   DIG4
        movlw   HIGH 1000
        movwf   K_H
        movlw   LOW 1000
        movwf   K_L
        call    CONTAR
        movwf   DIG3
        clrf    K_H
        movlw   100
        movwf   K_L
        call    CONTAR
        movwf   DIG2
        movlw   10
        movwf   K_L
        call    CONTAR
        movwf   DIG1
        movf    NUM_L,W
        movwf   DIG0
        return
CONTAR                          ; W = cuántas veces cabe K en NUM (y se lo resta)
        clrf    CNT
CT1     movf    K_L,W
        subwf   NUM_L,W         ; W = NUM_L - K_L   (C = 0 si hubo préstamo)
        movwf   TMP_L
        movf    K_H,W
        btfss   STATUS,C
        addlw   1               ; el préstamo se suma al byte alto que se resta
        subwf   NUM_H,W
        btfss   STATUS,C
        goto    CT2             ; ya no cabe
        movwf   NUM_H
        movf    TMP_L,W
        movwf   NUM_L
        incf    CNT,F
        goto    CT1
CT2     movf    CNT,W
        return

MUL16                           ; P2:P1:P0 = NUM_H:NUM_L x K_H:K_L  (resultado menor que 2^24)
        clrf    P0
        clrf    P1
        clrf    P2
        clrf    X2
        movf    NUM_H,W
        movwf   X1
        movf    NUM_L,W
        movwf   X0
        movlw   16
        movwf   CNT
ML1     bcf     STATUS,C
        rrf     K_H,F
        rrf     K_L,F           ; bit menos significativo de K en C
        btfss   STATUS,C
        goto    ML2
        movf    X0,W            ; P = P + X (24 bits)
        addwf   P0,F
        movf    X1,W
        btfsc   STATUS,C
        incfsz  X1,W
        addwf   P1,F
        movf    X2,W
        btfsc   STATUS,C
        incfsz  X2,W
        addwf   P2,F
ML2     bcf     STATUS,C        ; X = X * 2
        rlf     X0,F
        rlf     X1,F
        rlf     X2,F
        decfsz  CNT,F
        goto    ML1
        return

        END

In [ ]:
%%writefile ej08_cronometro_7seg.asm
;===========================================================================
; Ejemplo 8 · Cronómetro 00.00-99.99 s en display de 4 dígitos multiplexado
;             con interrupciones de TMR0 (cada 1 ms) y de RB0/INT
; PIC16F877A · cristal 4 MHz
;   Segmentos a..g, dp = RD0..RD7 (con 330 Ω)  ·  comunes de los dígitos 1..4 = RA0..RA3 (activos en 0)
;   RB0/INT = iniciar / detener (interrupción por flanco de bajada) · RB1 = poner en cero
;   RC0 = LED "corriendo"
;===========================================================================
        LIST    P=16F877A, R=DEC
        #include <p16f877a.inc>
        __CONFIG _XT_OSC & _WDT_OFF & _PWRTE_ON & _BOREN_OFF & _LVP_OFF & _CP_OFF

        CBLOCK  0x20
        VALS:4                          ; dígitos de izquierda a derecha: decenas, unidades, décimas, centésimas
        DIGITO, MS, CORRIENDO, REBOTE
        ENDC
        CBLOCK  0x70                    ; RAM común a los cuatro bancos
        W_TEMP, S_TEMP, P_TEMP
        ENDC

        ORG     0x000
        goto    INICIO

        ORG     0x004                   ;---------- vector de interrupción
        movwf   W_TEMP                  ; guarda el contexto
        swapf   STATUS,W
        clrf    STATUS                  ; banco 0
        movwf   S_TEMP
        movf    PCLATH,W
        movwf   P_TEMP
        clrf    PCLATH
        btfsc   INTCON,INTF
        call    ISR_INT
        btfss   INTCON,T0IF
        goto    FIN_ISR
        bcf     INTCON,T0IF
        movlw   6                       ; 256 - 6 = 250 cuentas x 4 µs = 1 ms
        addwf   TMR0,F
        call    MULTIPLEXAR
        call    CONTAR
FIN_ISR movf    P_TEMP,W                ; recupera el contexto
        movwf   PCLATH
        swapf   S_TEMP,W
        movwf   STATUS
        swapf   W_TEMP,F
        swapf   W_TEMP,W
        retfie

SEG7    addwf   PCL,F                   ; código de 7 segmentos (cátodo común) del dígito en W
        DT      0x3F,0x06,0x5B,0x4F,0x66,0x6D,0x7D,0x07,0x7F,0x6F
COMUN   addwf   PCL,F                   ; máscara para encender el dígito W (común en 0)
        DT      b'11111110',b'11111101',b'11111011',b'11110111'

ISR_INT bcf     INTCON,INTF
        movf    REBOTE,W                ; ignora los rebotes durante 200 ms
        btfss   STATUS,Z
        return
        movlw   200
        movwf   REBOTE
        movlw   1
        xorwf   CORRIENDO,F
        return

MULTIPLEXAR
        movlw   0x0F
        iorwf   PORTA,F                 ; apaga los cuatro dígitos
        incf    DIGITO,W
        andlw   3
        movwf   DIGITO
        addlw   VALS
        movwf   FSR
        movf    INDF,W
        call    SEG7
        movwf   PORTD
        movf    DIGITO,W                ; punto decimal después de los segundos
        xorlw   1
        btfsc   STATUS,Z
        bsf     PORTD,7
        movf    DIGITO,W
        call    COMUN
        andwf   PORTA,F                 ; enciende solo el dígito actual
        return

CONTAR  movf    REBOTE,W
        btfss   STATUS,Z
        decf    REBOTE,F
        btfss   CORRIENDO,0
        return
        incf    MS,F                    ; 10 ms = una centésima
        movlw   10
        subwf   MS,W
        btfss   STATUS,C
        return
        clrf    MS
        movlw   VALS+3                  ; suma 1 en BCD empezando por las centésimas
        movwf   FSR
CT1     incf    INDF,F
        movlw   10
        subwf   INDF,W
        btfss   STATUS,C
        return
        clrf    INDF
        decf    FSR,F
        movlw   VALS-1
        xorwf   FSR,W
        btfss   STATUS,Z
        goto    CT1
        return

INICIO
        BANKSEL ADCON1
        movlw   0x06                    ; PORTA digital
        movwf   ADCON1
        movlw   0xF0
        movwf   TRISA
        clrf    TRISD
        bcf     TRISC,0
        movlw   b'00000001'             ; pull-ups, INT por flanco de bajada, TMR0 interno, preescalador 1:4
        movwf   OPTION_REG
        BANKSEL PORTA
        movlw   0x0F
        movwf   PORTA
        clrf    PORTD
        clrf    PORTC
        clrf    VALS+0
        clrf    VALS+1
        clrf    VALS+2
        clrf    VALS+3
        clrf    DIGITO
        clrf    MS
        clrf    CORRIENDO
        clrf    REBOTE
        movlw   b'10110000'             ; GIE, T0IE, INTE
        movwf   INTCON

LAZO    btfsc   PORTB,1                 ; RB1: poner en cero
        goto    LED
        clrf    CORRIENDO
        clrf    MS
        clrf    VALS+0
        clrf    VALS+1
        clrf    VALS+2
        clrf    VALS+3
LED     btfss   CORRIENDO,0
        bcf     PORTC,0
        btfsc   CORRIENDO,0
        bsf     PORTC,0
        goto    LAZO
        END

In [ ]:
%%writefile ej09_paso_a_paso.asm
;===========================================================================
; Ejemplo 9 · Motor paso a paso unipolar con ULN2003A (secuencia de medio paso)
; PIC16F877A · cristal 4 MHz
;   RD0..RD3 -> IN1..IN4 del ULN2003A -> bobinas A, B, C, D del motor (MOTOR-STEPPER)
;   RB0 = girar en sentido horario, RB1 = antihorario (mientras se mantenga presionado)
;   Potenciómetro en AN0: tiempo entre pasos de 2 a 33 ms
;   RC0 = LED horario, RC1 = LED antihorario
;===========================================================================
        LIST    P=16F877A, R=DEC
        #include <p16f877a.inc>
        __CONFIG _XT_OSC & _WDT_OFF & _PWRTE_ON & _BOREN_OFF & _LVP_OFF & _CP_OFF

        CBLOCK  0x20
        IDX, DEMORA
        ENDC

        ORG     0x000
        goto    INICIO
        ORG     0x005
SECUENCIA                               ; medio paso: A, AB, B, BC, C, CD, D, DA
        addwf   PCL,F
        DT      b'0001',b'0011',b'0010',b'0110',b'0100',b'1100',b'1000',b'1001'

INICIO
        BANKSEL OPTION_REG
        bcf     OPTION_REG,NOT_RBPU
        movlw   0xF0
        movwf   TRISD
        movlw   0xFC
        movwf   TRISC
        movlw   0x0E                    ; AN0 analógico, justificado a la izquierda
        movwf   ADCON1
        BANKSEL PORTD
        clrf    PORTD
        clrf    PORTC
        clrf    IDX
        movlw   0x41
        movwf   ADCON0

LAZO    call    RETARDO_50US            ; lee la velocidad
        bsf     ADCON0,GO
ESPERA  btfsc   ADCON0,GO
        goto    ESPERA
        rrf     ADRESH,W                ; DEMORA = 2 + ADRESH / 8
        movwf   DEMORA
        rrf     DEMORA,F
        rrf     DEMORA,W
        andlw   0x1F
        addlw   2
        movwf   DEMORA

        btfss   PORTB,0
        goto    HORARIO
        btfss   PORTB,1
        goto    ANTIHORARIO
        clrf    PORTC                   ; ningún botón: LED apagados, bobinas sin corriente
        clrf    PORTD
        goto    LAZO

HORARIO incf    IDX,W
        andlw   7
        movwf   IDX
        movlw   b'00000001'
        goto    MOVER
ANTIHORARIO
        decf    IDX,W
        andlw   7
        movwf   IDX
        movlw   b'00000010'
MOVER   movwf   PORTC
        movf    IDX,W
        call    SECUENCIA
        movwf   PORTD
        movf    DEMORA,W
        call    RETARDO_MS
        goto    LAZO

;---------------------------------------------------------------------------
; Retardos para cristal de 4 MHz (1 ciclo de instrucción = 1 µs)
;---------------------------------------------------------------------------
        CBLOCK  0x68
        R_MS, R_US
        ENDC
RETARDO_MS                      ; espera W milisegundos (W = 1..255)
        movwf   R_MS
RMS1    movlw   249             ; 2 + 248*4 + 3 + 3 = 1000 ciclos por milisegundo
        movwf   R_US
RMS2    nop
        decfsz  R_US,F
        goto    RMS2
        decfsz  R_MS,F
        goto    RMS1
        return
RETARDO_50US                    ; 50 µs contando el call y el return
        movlw   15
        movwf   R_US
R50     decfsz  R_US,F
        goto    R50
        return

        END

In [ ]:
%%writefile ej10_semaforo.asm
;===========================================================================
; Ejemplo 10 · Semáforo con botón peatonal: interrupción externa (RB0/INT) y TMR1 (base de 100 ms)
; PIC16F877A · cristal 4 MHz
;   Vehicular: RD0 = rojo, RD1 = amarillo, RD2 = verde
;   Peatonal:  RD3 = rojo, RD4 = verde,  RD5 = LED "espere" (solicitud registrada)
;   RB0/INT = pulsador peatonal (a tierra, pull-up interno)
;   Ciclo: verde 10 s (3 s si un peatón lo pidió) -> amarillo 2 s -> rojo con peatón 5 s + 3 s parpadeando -> despeje 1 s
;===========================================================================
        LIST    P=16F877A, R=DEC
        #include <p16f877a.inc>
        __CONFIG _XT_OSC & _WDT_OFF & _PWRTE_ON & _BOREN_OFF & _LVP_OFF & _CP_OFF

        CBLOCK  0x20
        DECIMAS, SOLICITUD, LIMITE, VECES
        ENDC
        CBLOCK  0x70
        W_TEMP, S_TEMP
        ENDC

ROJO_V  EQU     0
AMAR_V  EQU     1
VERDE_V EQU     2
ROJO_P  EQU     3
VERDE_P EQU     4
ESPERE  EQU     5

        ORG     0x000
        goto    INICIO
        ORG     0x004                   ;---------- interrupciones
        movwf   W_TEMP
        swapf   STATUS,W
        clrf    STATUS
        movwf   S_TEMP
        btfss   PIR1,TMR1IF
        goto    NO_T1
        bcf     PIR1,TMR1IF             ; TMR1 = 65536 - 12500: 12500 x 8 µs = 100 ms
        bcf     T1CON,TMR1ON
        movlw   HIGH (65536 - 12500)
        movwf   TMR1H
        movlw   LOW (65536 - 12500)
        movwf   TMR1L
        bsf     T1CON,TMR1ON
        incf    DECIMAS,F
NO_T1   btfss   INTCON,INTF
        goto    FIN_ISR
        bcf     INTCON,INTF             ; un peatón presionó el botón
        movlw   1
        movwf   SOLICITUD
FIN_ISR swapf   S_TEMP,W
        movwf   STATUS
        swapf   W_TEMP,F
        swapf   W_TEMP,W
        retfie

INICIO
        BANKSEL TRISD
        clrf    TRISD
        movlw   b'00000000'             ; pull-ups activos, INT en flanco de bajada
        movwf   OPTION_REG
        bsf     PIE1,TMR1IE
        BANKSEL PORTD
        clrf    PORTD
        clrf    SOLICITUD
        movlw   HIGH (65536 - 12500)
        movwf   TMR1H
        movlw   LOW (65536 - 12500)
        movwf   TMR1L
        movlw   b'00110001'             ; TMR1 encendido, preescalador 1:8, reloj interno
        movwf   T1CON
        movlw   b'11010000'             ; GIE, PEIE, INTE
        movwf   INTCON

VERDE   movlw   (1 << VERDE_V) | (1 << ROJO_P)
        movwf   PORTD
        clrf    DECIMAS
V1      btfsc   SOLICITUD,0             ; muestra que la solicitud quedó registrada
        bsf     PORTD,ESPERE
        movlw   100                     ; 10 s como máximo
        subwf   DECIMAS,W
        btfsc   STATUS,C
        goto    AMARILLO
        btfss   SOLICITUD,0
        goto    V1
        movlw   30                      ; con solicitud: al menos 3 s de verde
        subwf   DECIMAS,W
        btfss   STATUS,C
        goto    V1

AMARILLO
        movlw   (1 << AMAR_V) | (1 << ROJO_P)
        btfsc   SOLICITUD,0
        iorlw   (1 << ESPERE)
        movwf   PORTD
        movlw   20
        call    ESPERAR

ROJO    clrf    SOLICITUD
        movlw   (1 << ROJO_V) | (1 << VERDE_P)
        movwf   PORTD
        movlw   50
        call    ESPERAR
        movlw   6                       ; el verde peatonal parpadea 3 s
        movwf   VECES
PARPADEO
        movlw   (1 << VERDE_P)
        xorwf   PORTD,F
        movlw   5
        call    ESPERAR
        decfsz  VECES,F
        goto    PARPADEO
        movlw   (1 << ROJO_V) | (1 << ROJO_P)   ; despeje: todo en rojo
        movwf   PORTD
        movlw   10
        call    ESPERAR
        clrf    SOLICITUD               ; las solicitudes durante el rojo ya se atendieron
        goto    VERDE

ESPERAR                                 ; espera W décimas de segundo usando el contador de la interrupción
        movwf   LIMITE
        clrf    DECIMAS
ES1     movf    LIMITE,W
        subwf   DECIMAS,W
        btfss   STATUS,C
        goto    ES1
        return
        END

### Las mismas aplicaciones en C (MPLAB X + XC8)
Se compilan en MPLAB X con XC8 (proyecto para PIC16F877A). Aquí solo se guardan para descargarlas.

In [ ]:
%%writefile ej01_contador_lcd.c
/* Ejemplo 1 · Contador 0000-9999 en LCD 16x2 con tres pulsadores (MPLAB X + XC8)
   RB0 = subir, RB1 = bajar, RB2 = reiniciar (pull-ups internos). LCD en PORTD. */
#include <xc.h>
#include <stdint.h>
#pragma config FOSC = XT, WDTE = OFF, PWRTE = ON, BOREN = OFF, LVP = OFF, CPD = OFF, WRT = OFF, CP = OFF
#define _XTAL_FREQ 4000000UL
/* ---------- LCD 16x2 (HD44780) en modo de 4 bits: RS = RD0, E = RD1, D4..D7 = RD4..RD7, RW a tierra ---------- */
#define LCD_RS PORTDbits.RD0
#define LCD_E  PORTDbits.RD1

void lcd_nibble(uint8_t n) {            /* pone n<7:4> en D7..D4 y da un pulso en E */
    PORTD = (uint8_t)((PORTD & 0x0F) | (n & 0xF0));
    LCD_E = 1; __delay_us(1); LCD_E = 0;
}
void lcd_cmd(uint8_t c) {
    LCD_RS = 0; lcd_nibble(c); lcd_nibble((uint8_t)(c << 4));
    if (c < 3) __delay_ms(2); else __delay_us(50);
}
void lcd_char(char d) {
    LCD_RS = 1; lcd_nibble((uint8_t)d); lcd_nibble((uint8_t)(d << 4)); __delay_us(50);
}
void lcd_init(void) {
    TRISD = 0x00; PORTD = 0x00;
    __delay_ms(20);
    lcd_nibble(0x30); __delay_ms(5);
    lcd_nibble(0x30); __delay_us(150);
    lcd_nibble(0x30); __delay_us(50);
    lcd_nibble(0x20); __delay_us(50);     /* modo de 4 bits */
    lcd_cmd(0x28);                        /* 2 líneas, 5x8 */
    lcd_cmd(0x0C);                        /* display encendido, sin cursor */
    lcd_cmd(0x06);                        /* incremento automático */
    lcd_cmd(0x01);                        /* borrar */
}
void lcd_goto(uint8_t fila, uint8_t col) { lcd_cmd((uint8_t)((fila ? 0xC0 : 0x80) + col)); }
void lcd_print(const char *s) { while (*s) lcd_char(*s++); }

void mostrar(uint16_t n) {
    lcd_goto(1, 8);
    lcd_char('0' + n / 1000); lcd_char('0' + (n / 100) % 10);
    lcd_char('0' + (n / 10) % 10); lcd_char('0' + n % 10);
}

void main(void) {
    uint16_t cuenta = 0;
    uint8_t teclas;
    OPTION_REGbits.nRBPU = 0;               /* pull-ups de PORTB */
    TRISB = 0xFF;
    lcd_init();
    lcd_goto(0, 0); lcd_print("Contador 0-9999");
    lcd_goto(1, 0); lcd_print("Cuenta:");
    mostrar(cuenta);
    while (1) {
        teclas = (uint8_t)(~PORTB & 0x07);  /* 1 = presionado */
        if (!teclas) continue;
        __delay_ms(20);                      /* antirrebote */
        teclas = (uint8_t)(~PORTB & 0x07);
        if (!teclas) continue;
        if (teclas & 0x01) cuenta = (cuenta == 9999) ? 0 : cuenta + 1;
        if (teclas & 0x02) cuenta = (cuenta == 0) ? 9999 : cuenta - 1;
        if (teclas & 0x04) cuenta = 0;
        mostrar(cuenta);
        while (~PORTB & 0x07);               /* espera a que se suelten */
        __delay_ms(20);
    }
}

In [ ]:
%%writefile ej02_oled_juegos.c
/* Ejemplo 2 · Tres juegos (PONG, CARRERA, SALTO) en OLED SSD1306 128x64 por SPI (MPLAB X + XC8)
   Cristal de 20 MHz. SCK = RC3, SDO = RC5, CS = RC0, D/C = RC1, RES = RC2.
   Pulsadores (pull-ups internos): RB0 = ARRIBA, RB1 = ABAJO, RB2 = OK / SALTAR.
   La RAM del PIC (368 bytes) no alcanza para una copia de la pantalla (1024 bytes):
   cada objeto se borra y se vuelve a dibujar directamente en la memoria del SSD1306. */
#include <xc.h>
#include <stdint.h>
#pragma config FOSC = HS, WDTE = OFF, PWRTE = ON, BOREN = OFF, LVP = OFF, CPD = OFF, WRT = OFF, CP = OFF
#define _XTAL_FREQ 20000000UL
#define OLED_CS  RC0
#define OLED_DC  RC1
#define OLED_RES RC2

const uint8_t FUENTE[59][5] = {               /* ASCII 32..90 */
 {0x00,0x00,0x00,0x00,0x00},{0x00,0x00,0x5F,0x00,0x00},{0x00,0x07,0x00,0x07,0x00},{0x14,0x7F,0x14,0x7F,0x14},
 {0x24,0x2A,0x7F,0x2A,0x12},{0x23,0x13,0x08,0x64,0x62},{0x36,0x49,0x55,0x22,0x50},{0x00,0x05,0x03,0x00,0x00},
 {0x00,0x1C,0x22,0x41,0x00},{0x00,0x41,0x22,0x1C,0x00},{0x08,0x2A,0x1C,0x2A,0x08},{0x08,0x08,0x3E,0x08,0x08},
 {0x00,0x50,0x30,0x00,0x00},{0x08,0x08,0x08,0x08,0x08},{0x00,0x60,0x60,0x00,0x00},{0x20,0x10,0x08,0x04,0x02},
 {0x3E,0x51,0x49,0x45,0x3E},{0x00,0x42,0x7F,0x40,0x00},{0x42,0x61,0x51,0x49,0x46},{0x21,0x41,0x45,0x4B,0x31},
 {0x18,0x14,0x12,0x7F,0x10},{0x27,0x45,0x45,0x45,0x39},{0x3C,0x4A,0x49,0x49,0x30},{0x01,0x71,0x09,0x05,0x03},
 {0x36,0x49,0x49,0x49,0x36},{0x06,0x49,0x49,0x29,0x1E},{0x00,0x36,0x36,0x00,0x00},{0x00,0x56,0x36,0x00,0x00},
 {0x08,0x14,0x22,0x41,0x00},{0x14,0x14,0x14,0x14,0x14},{0x00,0x41,0x22,0x14,0x08},{0x02,0x01,0x51,0x09,0x06},
 {0x32,0x49,0x79,0x41,0x3E},{0x7E,0x11,0x11,0x11,0x7E},{0x7F,0x49,0x49,0x49,0x36},{0x3E,0x41,0x41,0x41,0x22},
 {0x7F,0x41,0x41,0x22,0x1C},{0x7F,0x49,0x49,0x49,0x41},{0x7F,0x09,0x09,0x01,0x01},{0x3E,0x41,0x41,0x51,0x32},
 {0x7F,0x08,0x08,0x08,0x7F},{0x00,0x41,0x7F,0x41,0x00},{0x20,0x40,0x41,0x3F,0x01},{0x7F,0x08,0x14,0x22,0x41},
 {0x7F,0x40,0x40,0x40,0x40},{0x7F,0x02,0x04,0x02,0x7F},{0x7F,0x04,0x08,0x10,0x7F},{0x3E,0x41,0x41,0x41,0x3E},
 {0x7F,0x09,0x09,0x09,0x06},{0x3E,0x41,0x51,0x21,0x5E},{0x7F,0x09,0x19,0x29,0x46},{0x46,0x49,0x49,0x49,0x31},
 {0x01,0x01,0x7F,0x01,0x01},{0x3F,0x40,0x40,0x40,0x3F},{0x1F,0x20,0x40,0x20,0x1F},{0x7F,0x20,0x18,0x20,0x7F},
 {0x63,0x14,0x08,0x14,0x63},{0x03,0x04,0x78,0x04,0x03},{0x61,0x51,0x49,0x45,0x43}};
const uint8_t CARRO[8]  = {0x24,0x5A,0xFF,0xFF,0xFF,0xFF,0x5A,0x24};
const uint8_t ROCA[8]   = {0x1E,0x3F,0x7F,0x7F,0x7F,0x7E,0x3C,0x18};
const uint8_t DINO[8]   = {0x18,0x30,0xE0,0x78,0xFF,0x3D,0x17,0x07};
const uint8_t CACTUS[6] = {0x1C,0x10,0xFF,0x08,0x0E,0x00};
const uint8_t SALTO_P[8] = {4,3,2,2,2,3,4,5};
const uint8_t INIT[] = {0xAE,0xD5,0x80,0xA8,0x3F,0xD3,0x00,0x40,0x8D,0x14,0x20,0x00,
                        0xA1,0xC8,0xDA,0x12,0x81,0xCF,0xD9,0xF1,0xDB,0x40,0xA4,0xA6,0xAF};

uint8_t prev, nuevo, btn, rnd = 0xA5, puntos;

/* ---------- SPI y SSD1306 ---------- */
void spi(uint8_t b) { SSPBUF = b; while (!SSPIF); SSPIF = 0; (void)SSPBUF; }
void comando(uint8_t c) { OLED_DC = 0; spi(c); }
void ventana(uint8_t x, uint8_t pag) {         /* escribe desde la columna x, página pag */
    comando(0x21); comando(x); comando(127);
    comando(0x22); comando(pag); comando(7);
    OLED_DC = 1;
}
void rellenar(uint8_t x, uint8_t pag, uint8_t n, uint8_t v) { ventana(x, pag); while (n--) spi(v); }
void sprite(uint8_t x, uint8_t pag, const uint8_t *s, uint8_t n, uint8_t vacio) {
    ventana(x, pag);
    while (n--) spi(*s++);
    while (vacio--) spi(0);                    /* borra lo que quedó detrás */
}
void borrar(void) { ventana(0, 0); for (uint16_t i = 0; i < 1024; i++) spi(0); }
void caracter(char c) { for (uint8_t i = 0; i < 5; i++) spi(FUENTE[c - 32][i]); spi(0); }
void texto(uint8_t x, uint8_t pag, const char *s) { ventana(x, pag); while (*s) caracter(*s++); }
void numero(uint8_t n) { caracter('0' + n / 100); caracter('0' + (n / 10) % 10); caracter('0' + n % 10); }
void oled_init(void) {
    OLED_CS = 0; OLED_RES = 0; __delay_ms(10); OLED_RES = 1; __delay_ms(10);
    for (uint8_t i = 0; i < sizeof(INIT); i++) comando(INIT[i]);
    borrar();
}

/* ---------- utilidades ---------- */
void leer_botones(void) { btn = (uint8_t)(~PORTB & 0x07); nuevo = btn & (uint8_t)~prev; prev = btn; }
uint8_t aleatorio(void) { uint8_t c = rnd & 1; rnd >>= 1; if (c) rnd ^= 0xB8; return rnd; }

void fin_del_juego(void) {
    __delay_ms(250);
    borrar();
    texto(22, 2, "FIN DEL JUEGO");
    texto(28, 4, "PUNTOS "); numero(puntos);
    texto(34, 6, "OK: MENU");
    do { __delay_ms(30); leer_botones(); } while (!(nuevo & 4));
}

/* ---------- juego 1: PONG ---------- */
void pong(void) {
    uint8_t bx, by, dx, dy, pad = 3, vidas = 3, pag;
    borrar(); puntos = 0;
    for (uint8_t p = 0; p < 8; p++) rellenar(127, p, 1, 0xFF);          /* pared derecha */
    while (vidas) {
        bx = 64; by = 32; dx = 1; dy = aleatorio() & 1;                  /* dx = 1: hacia la izquierda */
        while (1) {
            leer_botones();
            if ((btn & 1) && pad > 0) pad--;
            if ((btn & 2) && pad < 6) pad++;
            rellenar(bx, by >> 3, 4, 0);                                  /* borra la bola */
            if (dy) { if (by == 0) dy = 0; else by -= 4; } else { if (by == 60) dy = 1; else by += 4; }
            if (!dx) { if (bx == 120) dx = 1; else bx += 4; }
            else if (bx == 8 && (uint8_t)((by >> 3) - pad) < 2) { dx = 0; puntos++; }   /* golpe */
            else if (bx == 0) { vidas--; break; }                        /* falla */
            else bx -= 4;
            if (pad) rellenar(2, pad - 1, 3, 0);                         /* raqueta */
            rellenar(2, pad, 3, 0xFF); rellenar(2, pad + 1, 3, 0xFF);
            if (pad < 6) rellenar(2, pad + 2, 3, 0);
            pag = by >> 3;
            rellenar(bx, pag, 4, (by & 4) ? 0xF0 : 0x0F);                /* bola de 4x4 */
            __delay_ms(25);
        }
    }
}

/* ---------- juego 2: CARRERA ---------- */
void carrera(void) {
    uint8_t carro = 4, ox[3], op[3] = {0, 0, 0}, spawn = 1;
    borrar(); puntos = 0; texto(0, 0, "PUNTOS ");
    while (1) {
        leer_botones();
        uint8_t nueva = carro;
        if ((nuevo & 1) && carro > 1) nueva--;
        if ((nuevo & 2) && carro < 7) nueva++;
        if (nueva != carro) { rellenar(4, carro, 8, 0); carro = nueva; }
        for (uint8_t i = 0; i < 3; i++) {
            if (!op[i]) continue;                                         /* inactivo */
            if (ox[i] == 0) { rellenar(0, op[i], 12, 0); op[i] = 0; puntos++; continue; }
            ox[i] -= 4;
            sprite(ox[i], op[i], ROCA, 8, 4);
            if (op[i] == carro && ox[i] < 12) { fin_del_juego(); return; }
        }
        if (--spawn == 0) {
            spawn = (uint8_t)((aleatorio() & 7) + 5);
            for (uint8_t i = 0; i < 3; i++)
                if (!op[i]) { uint8_t c = aleatorio() & 7; op[i] = c ? c : 7; ox[i] = 116; break; }
        }
        sprite(4, carro, CARRO, 8, 0);
        ventana(48, 0); numero(puntos);
        __delay_ms(30);
    }
}

/* ---------- juego 3: SALTO ---------- */
void salto(void) {
    uint8_t dino = 5, salto = 0, cx = 116, espera = 10, nueva;
    borrar(); puntos = 0; texto(0, 0, "PUNTOS ");
    rellenar(0, 6, 128, 0x01);                                            /* suelo */
    while (1) {
        leer_botones();
        if ((nuevo & 5) && !salto) salto = 1;
        if (salto) {
            nueva = SALTO_P[salto - 1];
            if (++salto == 9) salto = 0;
            if (nueva != dino) { rellenar(8, dino, 8, 0); dino = nueva; }
        }
        if (espera) espera--;
        else if (cx == 0) { rellenar(0, 5, 10, 0); puntos++; cx = 116; espera = aleatorio() & 15; }
        else {
            cx -= 4;
            sprite(cx, 5, CACTUS, 6, 4);
            if (dino == 5 && cx >= 4 && cx < 16) { fin_del_juego(); return; }
        }
        sprite(8, dino, DINO, 8, 0);
        ventana(48, 0); numero(puntos);
        __delay_ms(35);
    }
}

void main(void) {
    uint8_t sel = 0;
    OPTION_REGbits.nRBPU = 0;
    TRISC = 0b11010000;
    SSPSTAT = 0x40;                          /* CKE = 1 */
    SSPCON = 0x20;                           /* SPI maestro, Fosc/4 */
    oled_init();
    while (1) {
        borrar();
        texto(28, 0, "PIC JUEGOS");
        texto(22, 2, "1 PONG"); texto(22, 3, "2 CARRERA"); texto(22, 4, "3 SALTO");
        texto(0, 7, "ARRIBA/ABAJO  OK");
        while (1) {
            ventana(12, sel + 2); caracter('>');
            __delay_ms(30); aleatorio(); leer_botones();
            if (nuevo & 3) {
                ventana(12, sel + 2); caracter(' ');
                if (nuevo & 1) sel = sel ? sel - 1 : 2; else sel = (sel == 2) ? 0 : sel + 1;
            }
            if (nuevo & 4) break;
        }
        if (sel == 0) { pong(); fin_del_juego(); }
        else if (sel == 1) carrera();
        else salto();
    }
}

In [ ]:
%%writefile ej03_sensores.c
/* Ejemplo 3 · LM35 en AN0 y LDR en AN1 con LCD 16x2 (MPLAB X + XC8)
   RB0 = ventilador (T > 30 °C), RB1 = lámpara (luz < 30 %). */
#include <xc.h>
#include <stdint.h>
#pragma config FOSC = XT, WDTE = OFF, PWRTE = ON, BOREN = OFF, LVP = OFF, CPD = OFF, WRT = OFF, CP = OFF
#define _XTAL_FREQ 4000000UL
/* ---------- LCD 16x2 (HD44780) en modo de 4 bits: RS = RD0, E = RD1, D4..D7 = RD4..RD7, RW a tierra ---------- */
#define LCD_RS PORTDbits.RD0
#define LCD_E  PORTDbits.RD1

void lcd_nibble(uint8_t n) {            /* pone n<7:4> en D7..D4 y da un pulso en E */
    PORTD = (uint8_t)((PORTD & 0x0F) | (n & 0xF0));
    LCD_E = 1; __delay_us(1); LCD_E = 0;
}
void lcd_cmd(uint8_t c) {
    LCD_RS = 0; lcd_nibble(c); lcd_nibble((uint8_t)(c << 4));
    if (c < 3) __delay_ms(2); else __delay_us(50);
}
void lcd_char(char d) {
    LCD_RS = 1; lcd_nibble((uint8_t)d); lcd_nibble((uint8_t)(d << 4)); __delay_us(50);
}
void lcd_init(void) {
    TRISD = 0x00; PORTD = 0x00;
    __delay_ms(20);
    lcd_nibble(0x30); __delay_ms(5);
    lcd_nibble(0x30); __delay_us(150);
    lcd_nibble(0x30); __delay_us(50);
    lcd_nibble(0x20); __delay_us(50);     /* modo de 4 bits */
    lcd_cmd(0x28);                        /* 2 líneas, 5x8 */
    lcd_cmd(0x0C);                        /* display encendido, sin cursor */
    lcd_cmd(0x06);                        /* incremento automático */
    lcd_cmd(0x01);                        /* borrar */
}
void lcd_goto(uint8_t fila, uint8_t col) { lcd_cmd((uint8_t)((fila ? 0xC0 : 0x80) + col)); }
void lcd_print(const char *s) { while (*s) lcd_char(*s++); }

uint16_t leer_adc(uint8_t canal) {
    ADCON0 = (uint8_t)(0x41 | (canal << 3));   /* Fosc/8, canal, ADC encendido */
    __delay_us(50);                            /* adquisición */
    GO_nDONE = 1;
    while (GO_nDONE);
    return (uint16_t)((ADRESH << 8) | ADRESL);
}

void main(void) {
    uint16_t adc, decimas;
    uint8_t luz;
    TRISB = 0x00; PORTB = 0x00;
    ADCON1 = 0x84;                             /* justificado a la derecha; AN0, AN1, AN3 analógicos */
    lcd_init();
    lcd_goto(0, 0); lcd_print("Temp:");
    lcd_goto(1, 0); lcd_print("Luz:");
    while (1) {
        adc = leer_adc(0);
        decimas = (uint16_t)(((uint32_t)adc * 625) >> 7);   /* 4,883 mV por paso / 10 mV/°C */
        lcd_goto(0, 6);
        lcd_char(decimas >= 1000 ? '0' + decimas / 1000 : ' ');
        lcd_char('0' + (decimas / 100) % 10);
        lcd_char('0' + (decimas / 10) % 10);
        lcd_char('.');
        lcd_char('0' + decimas % 10);
        lcd_char((char)0xDF); lcd_char('C');
        RB0 = (decimas > 300);

        adc = leer_adc(1);
        luz = (uint8_t)(((uint32_t)adc * 25) >> 8);         /* 0..99 % */
        lcd_goto(1, 7);
        lcd_char(luz >= 100 ? '1' : ' ');
        lcd_char(luz >= 10 ? '0' + (luz / 10) % 10 : ' ');
        lcd_char('0' + luz % 10);
        lcd_char('%');
        RB1 = (luz < 30);
        __delay_ms(250);
    }
}

In [ ]:
%%writefile ej04_yolo_serial.c
/* Ejemplo 4 · YOLO + PIC por USART 9600 baudios (MPLAB X + XC8)
   'C' carro -> LED verde RB0, 'M' moto -> LED rojo RB1, 'P' persona -> LED amarillo RB2, 'N' -> apagar. */
#include <xc.h>
#include <stdint.h>
#pragma config FOSC = XT, WDTE = OFF, PWRTE = ON, BOREN = OFF, LVP = OFF, CPD = OFF, WRT = OFF, CP = OFF
#define _XTAL_FREQ 4000000UL
/* ---------- LCD 16x2 (HD44780) en modo de 4 bits: RS = RD0, E = RD1, D4..D7 = RD4..RD7, RW a tierra ---------- */
#define LCD_RS PORTDbits.RD0
#define LCD_E  PORTDbits.RD1

void lcd_nibble(uint8_t n) {            /* pone n<7:4> en D7..D4 y da un pulso en E */
    PORTD = (uint8_t)((PORTD & 0x0F) | (n & 0xF0));
    LCD_E = 1; __delay_us(1); LCD_E = 0;
}
void lcd_cmd(uint8_t c) {
    LCD_RS = 0; lcd_nibble(c); lcd_nibble((uint8_t)(c << 4));
    if (c < 3) __delay_ms(2); else __delay_us(50);
}
void lcd_char(char d) {
    LCD_RS = 1; lcd_nibble((uint8_t)d); lcd_nibble((uint8_t)(d << 4)); __delay_us(50);
}
void lcd_init(void) {
    TRISD = 0x00; PORTD = 0x00;
    __delay_ms(20);
    lcd_nibble(0x30); __delay_ms(5);
    lcd_nibble(0x30); __delay_us(150);
    lcd_nibble(0x30); __delay_us(50);
    lcd_nibble(0x20); __delay_us(50);     /* modo de 4 bits */
    lcd_cmd(0x28);                        /* 2 líneas, 5x8 */
    lcd_cmd(0x0C);                        /* display encendido, sin cursor */
    lcd_cmd(0x06);                        /* incremento automático */
    lcd_cmd(0x01);                        /* borrar */
}
void lcd_goto(uint8_t fila, uint8_t col) { lcd_cmd((uint8_t)((fila ? 0xC0 : 0x80) + col)); }
void lcd_print(const char *s) { while (*s) lcd_char(*s++); }

void uart_init(void) {
    SPBRG = 25; TXSTA = 0x24; RCSTA = 0x90;    /* 9615 baudios, BRGH = 1 */
}
void uart_tx(char c) { while (!TXIF); TXREG = c; }
void uart_print(const char *s) { while (*s) uart_tx(*s++); }

void main(void) {
    char c;
    TRISB = 0x00; PORTB = 0x00;
    uart_init();
    lcd_init();
    lcd_goto(0, 0); lcd_print("YOLO + PIC16F877");
    lcd_goto(1, 0); lcd_print("Esperando...    ");
    uart_print("PIC listo: envie C, M, P o N\r\n");
    while (1) {
        if (OERR) { CREN = 0; CREN = 1; }       /* reinicia el receptor si se desbordó */
        if (!RCIF) continue;
        c = (char)(RCREG & 0xDF);               /* a mayúscula */
        lcd_goto(1, 0);
        switch (c) {
            case 'C': PORTB = 0x01; lcd_print("Objeto: CARRO   "); break;
            case 'M': PORTB = 0x02; lcd_print("Objeto: MOTO    "); break;
            case 'P': PORTB = 0x04; lcd_print("Objeto: PERSONA "); break;
            case 'N': PORTB = 0x00; lcd_print("Objeto: ninguno "); break;
            default: continue;                  /* \r, \n u otros: se ignoran */
        }
        uart_print("OK:"); uart_tx(c); uart_print("\r\n");
    }
}

In [ ]:
%%writefile ej05_teclado_clave.c
/* Ejemplo 5 · Cerradura con teclado 4x4 (filas RB0-RB3, columnas RB4-RB7) y LCD (MPLAB X + XC8)
   Clave 1234, 'C' borra, '=' confirma. RC0 = verde, RC1 = rojo, RC2 = relé. */
#include <xc.h>
#include <stdint.h>
#pragma config FOSC = XT, WDTE = OFF, PWRTE = ON, BOREN = OFF, LVP = OFF, CPD = OFF, WRT = OFF, CP = OFF
#define _XTAL_FREQ 4000000UL
/* ---------- LCD 16x2 (HD44780) en modo de 4 bits: RS = RD0, E = RD1, D4..D7 = RD4..RD7, RW a tierra ---------- */
#define LCD_RS PORTDbits.RD0
#define LCD_E  PORTDbits.RD1

void lcd_nibble(uint8_t n) {            /* pone n<7:4> en D7..D4 y da un pulso en E */
    PORTD = (uint8_t)((PORTD & 0x0F) | (n & 0xF0));
    LCD_E = 1; __delay_us(1); LCD_E = 0;
}
void lcd_cmd(uint8_t c) {
    LCD_RS = 0; lcd_nibble(c); lcd_nibble((uint8_t)(c << 4));
    if (c < 3) __delay_ms(2); else __delay_us(50);
}
void lcd_char(char d) {
    LCD_RS = 1; lcd_nibble((uint8_t)d); lcd_nibble((uint8_t)(d << 4)); __delay_us(50);
}
void lcd_init(void) {
    TRISD = 0x00; PORTD = 0x00;
    __delay_ms(20);
    lcd_nibble(0x30); __delay_ms(5);
    lcd_nibble(0x30); __delay_us(150);
    lcd_nibble(0x30); __delay_us(50);
    lcd_nibble(0x20); __delay_us(50);     /* modo de 4 bits */
    lcd_cmd(0x28);                        /* 2 líneas, 5x8 */
    lcd_cmd(0x0C);                        /* display encendido, sin cursor */
    lcd_cmd(0x06);                        /* incremento automático */
    lcd_cmd(0x01);                        /* borrar */
}
void lcd_goto(uint8_t fila, uint8_t col) { lcd_cmd((uint8_t)((fila ? 0xC0 : 0x80) + col)); }
void lcd_print(const char *s) { while (*s) lcd_char(*s++); }

const char TECLAS[16] = "789/456*123-C0=+";

char escanear(void) {                        /* devuelve la tecla o 0 */
    for (uint8_t f = 0; f < 4; f++) {
        PORTB = (uint8_t)~(1 << f);          /* solo la fila f en 0 */
        __delay_us(5);
        uint8_t col = (uint8_t)((~PORTB >> 4) & 0x0F);
        for (uint8_t c = 0; c < 4; c++)
            if (col & (1 << c)) return TECLAS[f * 4 + c];
    }
    return 0;
}

void esperar_250ms(uint8_t n) { while (n--) __delay_ms(250); }

void main(void) {
    char clave[4], t;
    uint8_t n = 0, intentos = 0;
    OPTION_REGbits.nRBPU = 0;
    TRISB = 0xF0; TRISC = 0xF8; PORTC = 0;
    lcd_init();
nueva:
    PORTC = 0; n = 0;
    lcd_cmd(0x01); lcd_print("Ingrese la clave");
    lcd_goto(1, 0); lcd_print("> ");
    while (1) {
        t = escanear();
        if (!t) continue;
        __delay_ms(20);
        while (escanear());                  /* espera a soltar */
        __delay_ms(20);
        if (t == 'C') goto nueva;
        if (t == '=') break;
        if (t >= '0' && t <= '9' && n < 4) { clave[n++] = t; lcd_char('*'); }
    }
    lcd_goto(1, 0);
    if (n == 4 && clave[0] == '1' && clave[1] == '2' && clave[2] == '3' && clave[3] == '4') {
        intentos = 0; PORTC = 0x05;          /* verde + relé */
        lcd_print("ACCESO CONCEDIDO"); esperar_250ms(12);
    } else if (++intentos < 3) {
        PORTC = 0x02; lcd_print("CLAVE INCORRECTA"); esperar_250ms(8);
    } else {
        intentos = 0; lcd_print("BLOQUEADO  10 s ");
        for (uint8_t i = 0; i < 20; i++) { PORTC ^= 0x02; esperar_250ms(2); }
    }
    goto nueva;
}

In [ ]:
%%writefile ej06_servo.c
/* Ejemplo 6 · Servomotor en RC2 controlado con potenciómetro en AN0 + LCD (MPLAB X + XC8)
   Pulso de 1000 a 2020 µs cada ~20 ms. */
#include <xc.h>
#include <stdint.h>
#pragma config FOSC = XT, WDTE = OFF, PWRTE = ON, BOREN = OFF, LVP = OFF, CPD = OFF, WRT = OFF, CP = OFF
#define _XTAL_FREQ 4000000UL
/* ---------- LCD 16x2 (HD44780) en modo de 4 bits: RS = RD0, E = RD1, D4..D7 = RD4..RD7, RW a tierra ---------- */
#define LCD_RS PORTDbits.RD0
#define LCD_E  PORTDbits.RD1

void lcd_nibble(uint8_t n) {            /* pone n<7:4> en D7..D4 y da un pulso en E */
    PORTD = (uint8_t)((PORTD & 0x0F) | (n & 0xF0));
    LCD_E = 1; __delay_us(1); LCD_E = 0;
}
void lcd_cmd(uint8_t c) {
    LCD_RS = 0; lcd_nibble(c); lcd_nibble((uint8_t)(c << 4));
    if (c < 3) __delay_ms(2); else __delay_us(50);
}
void lcd_char(char d) {
    LCD_RS = 1; lcd_nibble((uint8_t)d); lcd_nibble((uint8_t)(d << 4)); __delay_us(50);
}
void lcd_init(void) {
    TRISD = 0x00; PORTD = 0x00;
    __delay_ms(20);
    lcd_nibble(0x30); __delay_ms(5);
    lcd_nibble(0x30); __delay_us(150);
    lcd_nibble(0x30); __delay_us(50);
    lcd_nibble(0x20); __delay_us(50);     /* modo de 4 bits */
    lcd_cmd(0x28);                        /* 2 líneas, 5x8 */
    lcd_cmd(0x0C);                        /* display encendido, sin cursor */
    lcd_cmd(0x06);                        /* incremento automático */
    lcd_cmd(0x01);                        /* borrar */
}
void lcd_goto(uint8_t fila, uint8_t col) { lcd_cmd((uint8_t)((fila ? 0xC0 : 0x80) + col)); }
void lcd_print(const char *s) { while (*s) lcd_char(*s++); }

void main(void) {
    uint8_t pos = 128, angulo;
    uint16_t ancho;
    TRISCbits.TRISC2 = 0; RC2 = 0;
    ADCON1 = 0x0E;                           /* AN0, justificado a la izquierda */
    ADCON0 = 0x41;
    lcd_init();
    lcd_goto(0, 0); lcd_print("Angulo:");
    lcd_goto(1, 0); lcd_print("Pulso:      us");
    while (1) {
        RC2 = 1;                             /* pulso de 1000 + 4·pos µs */
        __delay_us(1000);
        for (uint8_t i = pos; i; i--) __delay_us(3);   /* ~4 µs por vuelta */
        RC2 = 0;
        GO_nDONE = 1; while (GO_nDONE);
        pos = ADRESH;
        angulo = (uint8_t)(((uint16_t)pos * 180) >> 8);
        ancho = 1000 + 4u * pos;
        lcd_goto(0, 8);
        lcd_char('0' + angulo / 100); lcd_char('0' + (angulo / 10) % 10); lcd_char('0' + angulo % 10); lcd_char((char)0xDF);
        lcd_goto(1, 7);
        lcd_char('0' + ancho / 1000); lcd_char('0' + (ancho / 100) % 10); lcd_char('0' + (ancho / 10) % 10); lcd_char('0' + ancho % 10);
        __delay_ms(17);                      /* completa el periodo */
    }
}

In [ ]:
%%writefile ej07_motor_dc.c
/* Ejemplo 7 · Motor DC con L293D: PWM en RC2/CCP1 (EN1), sentido con RC0 (IN1) y RC1 (IN2) (MPLAB X + XC8)
   Pot en AN0 = velocidad. RB0 adelante, RB1 atrás, RB2 parar. */
#include <xc.h>
#include <stdint.h>
#pragma config FOSC = XT, WDTE = OFF, PWRTE = ON, BOREN = OFF, LVP = OFF, CPD = OFF, WRT = OFF, CP = OFF
#define _XTAL_FREQ 4000000UL
/* ---------- LCD 16x2 (HD44780) en modo de 4 bits: RS = RD0, E = RD1, D4..D7 = RD4..RD7, RW a tierra ---------- */
#define LCD_RS PORTDbits.RD0
#define LCD_E  PORTDbits.RD1

void lcd_nibble(uint8_t n) {            /* pone n<7:4> en D7..D4 y da un pulso en E */
    PORTD = (uint8_t)((PORTD & 0x0F) | (n & 0xF0));
    LCD_E = 1; __delay_us(1); LCD_E = 0;
}
void lcd_cmd(uint8_t c) {
    LCD_RS = 0; lcd_nibble(c); lcd_nibble((uint8_t)(c << 4));
    if (c < 3) __delay_ms(2); else __delay_us(50);
}
void lcd_char(char d) {
    LCD_RS = 1; lcd_nibble((uint8_t)d); lcd_nibble((uint8_t)(d << 4)); __delay_us(50);
}
void lcd_init(void) {
    TRISD = 0x00; PORTD = 0x00;
    __delay_ms(20);
    lcd_nibble(0x30); __delay_ms(5);
    lcd_nibble(0x30); __delay_us(150);
    lcd_nibble(0x30); __delay_us(50);
    lcd_nibble(0x20); __delay_us(50);     /* modo de 4 bits */
    lcd_cmd(0x28);                        /* 2 líneas, 5x8 */
    lcd_cmd(0x0C);                        /* display encendido, sin cursor */
    lcd_cmd(0x06);                        /* incremento automático */
    lcd_cmd(0x01);                        /* borrar */
}
void lcd_goto(uint8_t fila, uint8_t col) { lcd_cmd((uint8_t)((fila ? 0xC0 : 0x80) + col)); }
void lcd_print(const char *s) { while (*s) lcd_char(*s++); }

void sentido(uint8_t in1, uint8_t in2, const char *txt) {
    RC0 = in1; RC1 = in2;
    lcd_goto(1, 0); lcd_print(txt);
}

void main(void) {
    uint8_t vel, pct, teclas;
    OPTION_REGbits.nRBPU = 0;
    TRISC = 0xF8; PORTC = 0;
    ADCON1 = 0x0E; ADCON0 = 0x41;
    PR2 = 255;                               /* 3,9 kHz con preescalador 1 */
    CCPR1L = 0; CCP1CON = 0x0C;              /* PWM */
    T2CON = 0x04;                            /* TMR2 encendido */
    lcd_init();
    lcd_goto(0, 0); lcd_print("Motor DC  L293D");
    sentido(0, 0, "PARADO  ");
    while (1) {
        __delay_us(50);
        GO_nDONE = 1; while (GO_nDONE);
        vel = ADRESH;
        CCPR1L = vel;                        /* ciclo útil = vel / 256 */
        teclas = (uint8_t)(~PORTB & 0x07);
        if (teclas & 1) sentido(1, 0, "ADELANTE");
        if (teclas & 2) sentido(0, 1, "ATRAS   ");
        if (teclas & 4) sentido(0, 0, "PARADO  ");
        pct = (uint8_t)(((uint16_t)vel * 101) >> 8);
        lcd_goto(1, 11);
        lcd_char(pct >= 100 ? '1' : ' '); lcd_char('0' + (pct / 10) % 10); lcd_char('0' + pct % 10); lcd_char('%');
        __delay_ms(100);
    }
}

In [ ]:
%%writefile ej08_cronometro_7seg.c
/* Ejemplo 8 · Cronómetro 00.00-99.99 s en 4 displays multiplexados (MPLAB X + XC8)
   TMR0 interrumpe cada 1 ms; RB0/INT inicia/detiene; RB1 pone en cero.
   Segmentos en PORTD (a = RD0 ... dp = RD7), comunes en RA0-RA3 (activos en 0). RC0 = LED corriendo. */
#include <xc.h>
#include <stdint.h>
#pragma config FOSC = XT, WDTE = OFF, PWRTE = ON, BOREN = OFF, LVP = OFF, CPD = OFF, WRT = OFF, CP = OFF
#define _XTAL_FREQ 4000000UL

const uint8_t SEG7[10] = {0x3F, 0x06, 0x5B, 0x4F, 0x66, 0x6D, 0x7D, 0x07, 0x7F, 0x6F};
volatile uint8_t vals[4];                    /* decenas, unidades, décimas, centésimas */
volatile uint8_t digito, ms, corriendo, rebote;

void __interrupt() isr(void) {
    if (INTF) {                              /* botón iniciar / detener */
        INTF = 0;
        if (!rebote) { rebote = 200; corriendo ^= 1; }
    }
    if (TMR0IF) {
        TMR0IF = 0;
        TMR0 += 6;                           /* 250 cuentas x 4 µs = 1 ms */
        PORTA |= 0x0F;                       /* apaga los dígitos */
        digito = (digito + 1) & 3;
        PORTD = SEG7[vals[digito]] | (digito == 1 ? 0x80 : 0);
        PORTA &= (uint8_t)~(1 << digito);    /* enciende el actual */
        if (rebote) rebote--;
        if (corriendo && ++ms >= 10) {       /* una centésima */
            ms = 0;
            for (int8_t i = 3; i >= 0; i--) { if (++vals[i] < 10) break; vals[i] = 0; }
        }
    }
}

void main(void) {
    ADCON1 = 0x06;                           /* PORTA digital */
    TRISA = 0xF0; TRISD = 0x00; TRISCbits.TRISC0 = 0;
    PORTA = 0x0F; PORTD = 0;
    OPTION_REG = 0x01;                       /* pull-ups, INT flanco de bajada, TMR0 1:4 */
    INTCON = 0xB0;                           /* GIE, T0IE, INTE */
    while (1) {
        if (!RB1) { corriendo = 0; ms = 0; vals[0] = vals[1] = vals[2] = vals[3] = 0; }
        RC0 = corriendo;
    }
}

In [ ]:
%%writefile ej09_paso_a_paso.c
/* Ejemplo 9 · Motor paso a paso unipolar con ULN2003A en RD0-RD3, medio paso (MPLAB X + XC8)
   RB0 = horario, RB1 = antihorario. Pot en AN0 = 2..33 ms entre pasos. RC0/RC1 = LED de sentido. */
#include <xc.h>
#include <stdint.h>
#pragma config FOSC = XT, WDTE = OFF, PWRTE = ON, BOREN = OFF, LVP = OFF, CPD = OFF, WRT = OFF, CP = OFF
#define _XTAL_FREQ 4000000UL

const uint8_t SEC[8] = {0x01, 0x03, 0x02, 0x06, 0x04, 0x0C, 0x08, 0x09};

void main(void) {
    uint8_t idx = 0, demora;
    OPTION_REGbits.nRBPU = 0;
    TRISD = 0xF0; TRISC = 0xFC; PORTD = 0; PORTC = 0;
    ADCON1 = 0x0E; ADCON0 = 0x41;
    while (1) {
        __delay_us(50);
        GO_nDONE = 1; while (GO_nDONE);
        demora = (uint8_t)(2 + (ADRESH >> 3));
        if (!RB0)      { idx = (idx + 1) & 7; PORTC = 0x01; }
        else if (!RB1) { idx = (idx - 1) & 7; PORTC = 0x02; }
        else           { PORTC = 0; PORTD = 0; continue; }
        PORTD = SEC[idx];
        while (demora--) __delay_ms(1);
    }
}

In [ ]:
%%writefile ej10_semaforo.c
/* Ejemplo 10 · Semáforo con botón peatonal (RB0/INT) y base de tiempo con TMR1 cada 100 ms (MPLAB X + XC8)
   PORTD: RD0 rojo, RD1 amarillo, RD2 verde (vehicular); RD3 rojo, RD4 verde (peatonal); RD5 espere. */
#include <xc.h>
#include <stdint.h>
#pragma config FOSC = XT, WDTE = OFF, PWRTE = ON, BOREN = OFF, LVP = OFF, CPD = OFF, WRT = OFF, CP = OFF
#define _XTAL_FREQ 4000000UL

volatile uint8_t decimas, solicitud;

void __interrupt() isr(void) {
    if (TMR1IF) {
        TMR1IF = 0;
        TMR1ON = 0; TMR1H = 0xCF; TMR1L = 0x2C; TMR1ON = 1;   /* 65536 - 12500 */
        decimas++;
    }
    if (INTF) { INTF = 0; solicitud = 1; }
}

void esperar(uint8_t d) { decimas = 0; while (decimas < d); }

void main(void) {
    TRISD = 0x00; PORTD = 0;
    OPTION_REG = 0x00;                       /* pull-ups, INT por flanco de bajada */
    TMR1H = 0xCF; TMR1L = 0x2C;
    T1CON = 0x31;                            /* preescalador 1:8, encendido */
    TMR1IE = 1;
    INTCON = 0xD0;                           /* GIE, PEIE, INTE */
    while (1) {
        PORTD = 0b00001100;                  /* verde vehicular + rojo peatonal */
        decimas = 0;
        while (decimas < 100 && !(solicitud && decimas >= 30)) if (solicitud) RD5 = 1;
        PORTD = 0b00001010 | (solicitud ? 0x20 : 0);   /* amarillo */
        esperar(20);
        solicitud = 0;
        PORTD = 0b00010001;                  /* rojo vehicular + verde peatonal */
        esperar(50);
        for (uint8_t i = 0; i < 6; i++) { PORTD ^= 0x10; esperar(5); }
        PORTD = 0b00001001;                  /* despeje */
        esperar(10);
        solicitud = 0;
    }
}

## 3. Ensamblar y generar los `.hex`
En Proteus: doble clic sobre el PIC → *Program File* = el `.hex`, *Processor Clock Frequency* = 4 MHz (20 MHz en el ejemplo 2).

In [ ]:
import importlib, picasm, pic16sim, dispositivos
for m in (picasm, pic16sim, dispositivos): importlib.reload(m)
from picasm import Ensamblador, leer_hex, desensamblar
from pic16sim import PIC16F877
from dispositivos import LCD16x2, OLED, Teclado4x4, Pulso, Paso, Multiplex7

EJ = [('ej01_contador_lcd', 4000000.0), ('ej02_oled_juegos', 20000000.0), ('ej03_sensores', 4000000.0), ('ej04_yolo_serial', 4000000.0), ('ej05_teclado_clave', 4000000.0), ('ej06_servo', 4000000.0), ('ej07_motor_dc', 4000000.0), ('ej08_cronometro_7seg', 4000000.0), ('ej09_paso_a_paso', 4000000.0), ('ej10_semaforo', 4000000.0)]
PROG, FOSC = {}, {}
for n, f in EJ:
    a = Ensamblador(open(n + '.asm', encoding='utf-8').read()).ensamblar()
    open(n + '.hex', 'w').write(a.hex())
    with open(n + '.lst', 'w', encoding='utf-8') as L:
        for pc, w, t in a.lista: L.write(f'{pc:04X}  {w:04X}  {desensamblar(w):<22s} ; {t}\n')
    PROG[n], conf = leer_hex(a.hex()); FOSC[n] = f
    print(f'{n + ".hex":28s} {len(PROG[n]):5d} palabras · config 0x{conf:04X} · {f/1e6:.0f} MHz')

def nuevo(n):
    return PIC16F877(PROG[n], FOSC[n])

In [ ]:
# Descargar todo en un .zip (fuentes, .hex y listados)
import zipfile, glob
with zipfile.ZipFile('pic16f877a_10_ejemplos.zip', 'w') as z:
    for f in sorted(glob.glob('ej[01]*_*.*')):
        if not f.endswith('.zip'): z.write(f)
print('Creado pic16f877a_10_ejemplos.zip')
try:
    from google.colab import files; files.download('pic16f877a_10_ejemplos.zip')
except ImportError:
    print('(fuera de Colab el archivo queda en la carpeta actual)')

In [ ]:
# Así se ve el código de máquina: dirección, palabra de 14 bits e instrucción (inicio del ejemplo 1)
print(''.join(open('ej01_contador_lcd.lst', encoding='utf-8').readlines()[:25]))

## 4. Funciones de dibujo

In [ ]:
import numpy as np, matplotlib.pyplot as plt
plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False, 'axes.spines.right': False})

def ver_lcd(lcd, titulo=''):
    l1, l2 = lcd.texto()
    print(('  ' + titulo) if titulo else '')
    print('  ┌' + '─' * 16 + '┐'); print('  │' + l1.ljust(16) + '│'); print('  │' + l2.ljust(16) + '│'); print('  └' + '─' * 16 + '┘')

def ver_oled(o, ax=None, titulo=''):
    img = np.array([[o.pixel(x, y) for x in range(128)] for y in range(64)])
    ax = ax or plt.figure(figsize=(5, 2.6)).gca()
    from matplotlib.colors import ListedColormap
    ax.imshow(img, cmap=ListedColormap(['#05080A', '#7FD8FF']), vmin=0, vmax=1, interpolation='nearest'); ax.set_xticks([]); ax.set_yticks([]); ax.set_title(titulo)

def pulsar(p, puerto, bit, t=0.06, despues=0.06):
    p.poner_entrada(puerto, 0xFF & ~(1 << bit)); p.correr(segundos=t)
    p.poner_entrada(puerto, 0xFF); p.correr(segundos=despues)

## 5. Ejemplo 1 · Contador con LCD 16x2
RB0 sube, RB1 baja, RB2 reinicia. El modelo de la LCD también revisa los tiempos: si el programa le escribe mientras está ocupada, lo reporta en `lcd.errores`.

In [ ]:
p = nuevo('ej01_contador_lcd'); lcd = LCD16x2(p)
p.correr(segundos=0.1); ver_lcd(lcd, 'al encender')
for _ in range(7): pulsar(p, 'B', 0)
ver_lcd(lcd, 'después de 7 pulsaciones de SUBIR')
for _ in range(10): pulsar(p, 'B', 1)
ver_lcd(lcd, 'después de 10 pulsaciones de BAJAR (pasa por 9999)')
print('Errores de tiempo de la LCD:', lcd.errores or 'ninguno')

## 6. Ejemplo 2 · Tres juegos en la OLED (SPI, 20 MHz)
Se juega con un *piloto automático* que lee la RAM del PIC (posición de la bola, de las rocas y de los cactus) y presiona los botones en el momento justo.

In [ ]:
S = Ensamblador(open('ej02_oled_juegos.asm', encoding='utf-8').read()).ensamblar().simbolos   # direcciones de las variables
p = nuevo('ej02_oled_juegos'); o = OLED(p); r = lambda v: p.ram[S[v]]
def cuadro(boton=None, t=0.03):
    p.poner_entrada('B', 0xFF if boton is None else 0xFF & ~(1 << boton)); p.correr(segundos=t)
fig, ax = plt.subplots(2, 2, figsize=(10, 5.4))
p.correr(segundos=0.2); ver_oled(o, ax[0, 0], 'Menú')
cuadro(2); cuadro()                                         # PONG
for k in range(200):
    pag, pad = r('BY') >> 3, r('PAD'); cuadro(0 if pag < pad else 1 if pag > pad + 1 else None)
ver_oled(o, ax[0, 1], f'PONG · {r("SCORE")} puntos, {r("VIDAS")} vidas')
p = nuevo('ej02_oled_juegos'); o = OLED(p); p.correr(segundos=0.2)
cuadro(1); cuadro(); cuadro(2); cuadro()                    # CARRERA
for k in range(220):
    car = r('CAR'); malos = {p.ram[S['OBS'] + 2*i + 1] for i in range(3) if p.ram[S['OBS'] + 2*i + 1] and p.ram[S['OBS'] + 2*i] < 40}
    b = None
    if car in malos:
        for d, bt in ((-1, 0), (1, 1)):
            if 1 <= car + d <= 7 and car + d not in malos: b = bt; break
    cuadro(b)
    if b is not None: cuadro(None, 0.001)
ver_oled(o, ax[1, 0], f'CARRERA · {r("SCORE")} puntos')
p = nuevo('ej02_oled_juegos'); o = OLED(p); p.correr(segundos=0.2)
cuadro(1); cuadro(); cuadro(1); cuadro(); cuadro(2); cuadro()   # SALTO
for k in range(350):
    cuadro(2 if r('ESPERA') == 0 and 14 <= r('CX') <= 22 and r('JUMP') == 0 else None, 0.036)
ver_oled(o, ax[1, 1], f'SALTO · {r("SCORE")} puntos')
plt.tight_layout(); plt.show()

## 7. Ejemplo 3 · Sensores LM35 y LDR
Se barre la temperatura del LM35 y se compara lo que muestra la LCD con el valor real.

In [ ]:
p = nuevo('ej03_sensores'); lcd = LCD16x2(p); p.analog[1] = 3.0
reales, medidas, vent = [], [], []
for tc in np.arange(0, 101, 5):
    p.analog[0] = tc * 0.01; p.correr(segundos=0.3)
    reales.append(tc); medidas.append(float(lcd.texto()[0][5:11].replace('°', ''))); vent.append(p.puerto('B') & 1)
ver_lcd(lcd, 'a 100 °C')
fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))
ax[0].plot(reales, medidas, 'o-', color='#1F8A70'); ax[0].plot(reales, reales, '--', color='#999'); ax[0].set_xlabel('T real (°C)'); ax[0].set_ylabel('T en la LCD (°C)'); ax[0].set_title('LM35 → ADC → LCD (resolución 0,49 °C)')
ax[1].step(reales, vent, where='mid', color='#2F6DB5'); ax[1].set_yticks([0, 1]); ax[1].set_yticklabels(['apagado', 'encendido']); ax[1].axvline(30, ls=':', color='#E2453A'); ax[1].set_xlabel('T real (°C)'); ax[1].set_title('Ventilador en RB0 (umbral 30 °C)')
plt.tight_layout(); plt.show()
print('Error máximo:', round(max(abs(a - b) for a, b in zip(reales, medidas)), 2), '°C')

## 8. Ejemplo 4 · YOLO + PIC por la USART
Primero el protocolo con letras escritas a mano; después **YOLO real** (Ultralytics) decide la letra a partir de una imagen.

En el laboratorio, `yolo_pic.py` hace lo mismo con la cámara web y un puerto serie real (o el COMPIM de Proteus con un par de puertos virtuales).

In [ ]:
p = nuevo('ej04_yolo_serial'); lcd = LCD16x2(p)
p.correr(segundos=0.2); print('PIC →', repr(bytes(p.tx).decode())); p.tx.clear()
for letra in 'CMPN':
    p.recibir(ord(letra)); p.correr(segundos=0.02)
    leds = p.puerto('B') & 7
    print(f"PC envía '{letra}'  →  LCD: {lcd.texto()[1]}  LEDs verde/rojo/amarillo = {leds & 1}/{leds >> 1 & 1}/{leds >> 2 & 1}  · PIC responde {bytes(p.tx).decode().strip()!r}")
    p.tx.clear()

In [ ]:
%%writefile yolo_pic.py
"""
YOLO + PIC16F877A (ejemplo 4): detecta carros, motos y personas con la cámara (o un video / imagen)
y envía una letra al PIC por el puerto serie a 9600 baudios.

    C = carro (LED verde)   M = moto (LED rojo)   P = persona (LED amarillo)   N = nada

Instalación (en el PC):   pip install ultralytics pyserial opencv-python
Uso:
    python yolo_pic.py --puerto COM3                 # cámara web
    python yolo_pic.py --puerto COM3 --fuente video.mp4
    python yolo_pic.py --listar                      # muestra los puertos serie disponibles
    python yolo_pic.py --sin-pic                     # solo detección, sin puerto serie

Con Proteus: instale un par de puertos virtuales (com0com en Windows, p. ej. COM3 <-> COM4),
ponga un COMPIM en el esquema con Physical Port = COM4, 9600 baudios, y ejecute este programa con --puerto COM3.
Con el PIC real: use un conversor USB-serie TTL (CH340, CP2102, FT232) conectado a RC6/TX y RC7/RX.
"""
import argparse, time

CLASES = {'car': 'C', 'truck': 'C', 'bus': 'C', 'motorcycle': 'M', 'bicycle': 'M', 'person': 'P'}
PRIORIDAD = ['M', 'C', 'P']          # si hay varios objetos, la moto tiene prioridad, luego el carro


def decidir(nombres_detectados):
    letras = {CLASES[n] for n in nombres_detectados if n in CLASES}
    for l in PRIORIDAD:
        if l in letras: return l
    return 'N'


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument('--puerto', default='COM3')
    ap.add_argument('--baudios', type=int, default=9600)
    ap.add_argument('--fuente', default='0', help='0 = cámara web, o la ruta de un video o imagen')
    ap.add_argument('--modelo', default='yolov8n.pt')
    ap.add_argument('--confianza', type=float, default=0.45)
    ap.add_argument('--listar', action='store_true')
    ap.add_argument('--sin-pic', action='store_true')
    a = ap.parse_args()

    if a.listar:
        from serial.tools import list_ports
        for p in list_ports.comports(): print(p.device, '-', p.description)
        return

    import cv2
    from ultralytics import YOLO
    pic = None
    if not a.sin_pic:
        import serial
        pic = serial.Serial(a.puerto, a.baudios, timeout=0.05)
        time.sleep(2)                                   # el PIC termina de arrancar
        print('PIC:', pic.read(200).decode(errors='ignore').strip())

    modelo = YOLO(a.modelo)
    fuente = int(a.fuente) if a.fuente.isdigit() else a.fuente
    ultima = None
    for r in modelo.predict(source=fuente, stream=True, conf=a.confianza, verbose=False):
        nombres = [modelo.names[int(c)] for c in r.boxes.cls]
        letra = decidir(nombres)
        if letra != ultima:                             # solo envía cuando cambia la decisión
            print(f'{time.strftime("%H:%M:%S")}  detectado: {sorted(set(nombres)) or "-"}  ->  envía {letra}')
            if pic:
                pic.write(letra.encode())
                time.sleep(0.05)
                resp = pic.read(20).decode(errors='ignore').strip()
                if resp: print('   PIC responde:', resp)
            ultima = letra
        cv2.imshow('YOLO -> PIC16F877A (q para salir)', r.plot())
        if cv2.waitKey(1) & 0xFF == ord('q'): break
    cv2.destroyAllWindows()
    if pic: pic.close()


if __name__ == '__main__':
    main()

In [ ]:
# YOLO real en Colab (descarga el modelo yolov8n la primera vez)
!pip -q install ultralytics

In [ ]:
from ultralytics import YOLO
from yolo_pic import decidir
modelo = YOLO('yolov8n.pt')
imagenes = ['https://ultralytics.com/images/bus.jpg', 'https://ultralytics.com/images/zidane.jpg']
# Para probar con sus fotos: from google.colab import files; imagenes = list(files.upload())
p = nuevo('ej04_yolo_serial'); lcd = LCD16x2(p); p.correr(segundos=0.2); p.tx.clear()
fig, axs = plt.subplots(1, len(imagenes), figsize=(6 * len(imagenes), 5))
for ax, img in zip(np.atleast_1d(axs), imagenes):
    r = modelo(img, verbose=False)[0]
    nombres = [modelo.names[int(c)] for c in r.boxes.cls]
    letra = decidir(nombres)
    p.recibir(ord(letra)); p.correr(segundos=0.02)
    ax.imshow(r.plot()[:, :, ::-1]); ax.axis('off')
    ax.set_title(f"YOLO: {sorted(set(nombres))}\n→ envía '{letra}' · LCD: {lcd.texto()[1].strip()} · LEDs {p.puerto('B') & 7:03b}")
plt.tight_layout(); plt.show()

## 9. Ejemplo 5 · Cerradura con teclado 4x4
El modelo del teclado conecta la fila que el PIC pone en 0 con la columna de la tecla presionada.

In [ ]:
p = nuevo('ej05_teclado_clave'); lcd = LCD16x2(p); kp = Teclado4x4(p); p.correr(segundos=0.1)
def tecla(k):
    kp.tecla = k; p.correr(segundos=0.06); kp.tecla = None; p.correr(segundos=0.06)
for k in '1299': tecla(k)
ver_lcd(lcd, 'escribiendo 1299')
tecla('='); ver_lcd(lcd, 'al confirmar'); print('LED rojo (RC1):', p.puerto('C') >> 1 & 1)
p.correr(segundos=2.2)
for k in '1234=': tecla(k)
ver_lcd(lcd, 'con la clave 1234'); print('LED verde (RC0):', p.puerto('C') & 1, '· relé (RC2):', p.puerto('C') >> 2 & 1)

## 10. Ejemplo 6 · Servomotor
Se mide el ancho del pulso en RC2 con el emulador para cada posición del potenciómetro.

In [ ]:
p = nuevo('ej06_servo'); lcd = LCD16x2(p); s = Pulso(p, 'C', 2)
vs, anchos = np.linspace(0, 5, 11), []
for v in vs:
    p.analog[0] = v; p.correr(segundos=0.1); anchos.append(s.ancho * 1e3)
ver_lcd(lcd, 'con 5 V'); print(f'Periodo medido: {s.periodo*1e3:.1f} ms')
fig, ax = plt.subplots(1, 2, figsize=(11, 3.3))
ax[0].plot(vs, anchos, 'o-', color='#1F8A70'); ax[0].set_xlabel('V en AN0'); ax[0].set_ylabel('pulso (ms)'); ax[0].set_title('Ancho del pulso en RC2')
ax[1].plot(vs, (np.array(anchos) - 1.5) / 0.5 * 90, 'o-', color='#F2A33A'); ax[1].set_xlabel('V en AN0'); ax[1].set_ylabel('ángulo (°)'); ax[1].set_title('Ángulo del servo (1 ms = −90°, 2 ms = +90°)')
plt.tight_layout(); plt.show()

## 11. Ejemplo 7 · Motor DC con PWM y L293D

In [ ]:
p = nuevo('ej07_motor_dc'); lcd = LCD16x2(p)
fig, ax = plt.subplots(figsize=(10, 3))
for k, (v, col) in enumerate(((1.0, '#2F6DB5'), (2.5, '#1F8A70'), (4.5, '#E2453A'))):
    p.analog[0] = v; p.correr(segundos=0.25); pw = p.pwm()
    t = np.linspace(0, 3 / pw['f'], 900); ax.plot(t * 1e6, ((t * pw['f']) % 1 < pw['D']) * 5 + k * 6, color=col, label=f'{v} V → {pw["D"]*100:.0f} %')
ax.set_xlabel('µs'); ax.set_yticks([]); ax.legend(frameon=False, loc='upper left', bbox_to_anchor=(1.0, 1.0)); ax.set_title(f'PWM en RC2/CCP1 a {pw["f"]:.0f} Hz'); plt.tight_layout(); plt.show()
pulsar(p, 'B', 1, 0.15); ver_lcd(lcd, 'después de ATRÁS'); print('IN1 =', p.puerto('C') & 1, ' IN2 =', p.puerto('C') >> 1 & 1)

## 12. Ejemplo 8 · Cronómetro con interrupciones
¿Qué tan exacto es? Se compara el tiempo que muestra el display con el tiempo simulado.

In [ ]:
p = nuevo('ej08_cronometro_7seg'); d = Multiplex7(p); p.correr(segundos=0.05)
t0 = p.tiempo(); pulsar(p, 'B', 0, 0.03, 0)
reales, mostrados = [], []
for k in range(10):
    p.correr(segundos=0.5); reales.append(p.tiempo() - t0); mostrados.append(float(d.texto()))
print('Display:', d.texto(), '· tiempo real:', round(reales[-1], 3), 's')
plt.figure(figsize=(6, 3)); plt.plot(reales, (np.array(mostrados) - np.array(reales)) * 1e3, 'o-', color='#7B5EA7'); plt.ylim(-30, 30)
plt.xlabel('tiempo real (s)'); plt.ylabel('error (ms)'); plt.title('Error del cronómetro (TMR0 recargado cada 1 ms)'); plt.show()

## 13. Ejemplo 9 · Motor paso a paso

In [ ]:
p = nuevo('ej09_paso_a_paso'); m = Paso(p)
ts, angs = [], []
for v, b, dur in ((0.0, 0, 0.5), (5.0, 0, 0.5), (1.0, 1, 0.5)):
    p.analog[0] = v; p.poner_entrada('B', 0xFF & ~(1 << b))
    for _ in range(25): p.correr(segundos=dur / 25); ts.append(p.tiempo()); angs.append(m.angulo())
p.poner_entrada('B', 0xFF)
plt.figure(figsize=(8, 3)); plt.plot(ts, angs, color='#1F8A70')
plt.axvspan(0, 0.5, alpha=.08); plt.axvspan(1.0, 1.5, alpha=.08, color='orange')
plt.xlabel('s'); plt.ylabel('ángulo (°)'); plt.title('Horario rápido → horario lento → antihorario'); plt.show()
print('Pasos totales:', m.pasos, '· errores de secuencia:', m.errores)

## 14. Ejemplo 10 · Semáforo peatonal
Línea de tiempo de las luces; el peatón presiona el botón a los 23 s.

In [ ]:
p = nuevo('ej10_semaforo'); hist = []
for k in range(320):
    if k == 230: p.poner_entrada('B', 0xFE)
    if k == 231: p.poner_entrada('B', 0xFF)
    p.correr(segundos=0.1); hist.append(p.puerto('D'))
H = np.array([[(v >> b) & 1 for v in hist] for b in range(6)])
cols = ['#E2453A', '#F2B33A', '#2E9B57', '#E2453A', '#2E9B57', '#888888']
nombres = ['rojo V', 'amarillo V', 'verde V', 'rojo P', 'verde P', 'espere']
plt.figure(figsize=(11, 3))
for b in range(6):
    t = np.arange(len(hist)) * 0.1
    plt.fill_between(t, b, b + 0.8 * H[b], step='post', color=cols[b])
plt.yticks(np.arange(6) + 0.4, nombres); plt.xlabel('s'); plt.axvline(23, ls=':', color='k'); plt.title('Semáforo: el botón (línea punteada) acorta el verde'); plt.show()

## 15. Ejercicios propuestos
1. **Ejemplo 1**: agregue un cuarto pulsador (RB3) que sume de 10 en 10.
2. **Ejemplo 2**: haga que la velocidad de CARRERA aumente cada 10 puntos (reduzca el retardo del cuadro).
3. **Ejemplo 3**: agregue histéresis al ventilador (enciende a 30 °C y apaga a 28 °C).
4. **Ejemplo 4**: agregue la letra `B` (bus) que haga parpadear el LED amarillo, y cámbiela en `yolo_pic.py`.
5. **Ejemplo 5**: guarde la clave en la EEPROM interna y permita cambiarla.
6. **Ejemplo 8**: convierta el cronómetro en temporizador regresivo con alarma.

Plantilla: escriba un programa, ensámblelo, simúlelo y descargue su `.hex`.

In [ ]:
MI_PROGRAMA = """
        LIST    P=16F877A, R=DEC
        #include <p16f877a.inc>
        __CONFIG _XT_OSC & _WDT_OFF & _PWRTE_ON & _BOREN_OFF & _LVP_OFF & _CP_OFF
        ORG     0
        BANKSEL TRISB
        clrf    TRISB
        BANKSEL PORTB
        movlw   b'10100101'
        movwf   PORTB
FIN     goto    FIN
        END
"""
a = Ensamblador(MI_PROGRAMA).ensamblar(); open('mi_programa.hex', 'w').write(a.hex())
p = PIC16F877(a.mem); p.correr(1000)
print(f'PORTB = {p.puerto("B"):08b} · mi_programa.hex listo para Proteus')

## 16. Simulador interactivo en el navegador
Suba `laboratorio_pic16f877a.html` (entregado con las diapositivas) y ejecute la celda.

In [ ]:
import os, html as _html
from IPython.display import HTML, display
f = 'laboratorio_pic16f877a.html'
if not os.path.exists(f):
    try:
        from google.colab import files; files.upload()
    except ImportError:
        print('Copie laboratorio_pic16f877a.html a esta carpeta.')
if os.path.exists(f):
    display(HTML('<iframe srcdoc="' + _html.escape(open(f, encoding='utf-8').read(), quote=True) + '" style="width:100%;height:900px;border:1px solid #D5DEE2;border-radius:12px" scrolling="yes"></iframe>'))